# <center>Harness Engineering 驾驭工程 · 第四节：Hermes 记忆系统 + Nudge 机制</center>

&emsp;&emsp;前三节课你已经走完了从认知地图到亲手搓 mini Harness、再到用 `DeepAgents` 框架把 11 个机制装进生产系统的完整路径。今天我们打开的是另一种取舍——`Hermes Agent`。它不是 `Claude Code` 的升级版，也不是 `DeepAgents` 的竞品，而是把 `Garbage Collection`（熵减/动态化）这条支柱推到极致的样本：自己维护 skill 库、跨 19 个平台一致响应、curator 后台定期大扫除。我们会带你直接读它的源码、本地跑一个 MVP demo、然后用一把"尺"(**3 问评价法 + 四维评价框架**,3 问对应 L1 三支柱里的 GC/AC + 一个三支柱外的入口治理维度,3.5 节合 CE 共 4 维)去评价它选了什么、放弃了什么。

&emsp;&emsp;这节课要回答三个具体问题。第一，你在第三节看到的 `LangGraph state messages` 列表，长对话怎么截断、跨 session 怎么续——`Hermes` 用 `SQLite + FTS5 双索引 + LLM 摘要` 冷记忆三件套给出了它的答案（再加上 `MEMORY.md / USER.md` 热记忆双层），我们会逐层拆开看。第二，`README` 上那句"creates skills from experience"是怎么落地的——拨开"AI 自主学习"的浪漫想象，背后是一条 `计数器 → 判定 → fork agent → 写文件` 的机械闭环，我们会在 `run_agent.py` 里逐行核实。第三，`PDF` 上提到的 `GEPA Skill Evolution` 和 "四层记忆是默认架构" 这些说法究竟在 `hermes-agent` 主仓库源码里能不能对得上——这一节我们会站在源码这一侧给出诚实的答案，让你以后再读 `harness` 类产品的宣传材料时，知道怎么看清取舍。

&emsp;&emsp;为了让你带走的不只是"一个 harness 的细节"，今天最后一章我们会抽出一把通用的尺——**3 问评价法**(能力怎么增长 / 风险怎么控制 / 入口怎么组织)直接对应 L1 三支柱里的 `Garbage Collection / Architectural Constraints / 入口治理`(三支柱外补一维),再加上第 1 章已展开的 `Context Engineering`,3.5 节合成完整的**四维评价框架**。这把尺不是又一套要背的新词，而是把第一节学过的三支柱**换个评价坐标系**——补一个三支柱不直接覆盖的入口治理维度,让评价完整。

> 📅 **时效性说明**：本课全部源码引用截止 2026 年 5 月，基于 `Hermes Agent` GitHub 主分支当时的 `run_agent.py` 等核心文件。所有 `file:line` 引用都是真实可核对的——你可以在自己电脑 `git clone` 仓库后用 `vim run_agent.py +3613` 打到对应位置。 github地址：https://github.com/NousResearch/hermes-agent

> 📌 **目标受众与前置要求**：本课面向已经完成前三节课的学员——你能默写第一节八大故障与三支柱、读懂第二节的 11 机制清单（含 `Permission Gate / Hooks / Token Budget` 三个第二节第 9 章扩展机制）、记得第三节用 `deepagents==0.5.3` 的 8 步流水线和 4 大组件（`Middleware / Backend / 子代理 / HITL`）。本课不再重新解释这些前置内容，仅在需要时做一句话引用。技术上你只需要 `Python` 基础 + 改 `YAML/JSON` 配置的能力，**不需要从零敲新框架代码**——主线是听原理、读源码、本地跑 `MVP demo`、改参数观察现象。

> 📌 **学完本节你将带走四件产物**：① `Hermes` 默认记忆系统架构图——**默认链路**：`MEMORY.md / USER.md` 热记忆双层（每次对话注入）+ `SQLite + FTS5 双索引 + LLM 摘要` 冷记忆三件套（按需检索）；并知道**扩展层** 8 种外部 `Memory Provider`（含 `Holographic`）全部 opt-in，不在默认链路内；② `Nudge` 机制源码 walk 笔记（计数器 → 判定 → fork → 落盘 → 防递归）；③ 一份本地可跑、改参数即可观察现象的 `MVP demo` 代码包（用 `langchain AgentMiddleware` + `@tool` + `create_agent` 三件套模拟 `Hermes` 核心机制）；④ **四维评价尺**(`Garbage Collection 动态化 / Architectural Constraints 架构约束 / Context Engineering 上下文工程 / 入口治理`,前 3 个直接复用 L1 三支柱,第 4 个是三支柱外的合规视角补充),下节课末尾我们会用这把尺去横评 `Claude Code / Codex / DeepAgents / OpenClaw` 四家。

---

## <center>第0章：开场角色声明 + 衔接前置课程</center>

&emsp;&emsp;在进入新内容之前，先用 6 分钟把"今天我们学的是什么、跟之前学的是什么关系"这件事讲透。这一章不讲新知识，只建立学习坐标系——你会先看到一个学员普遍的直觉错误被打碎，然后拿到本讲在系列时间轴上的位置、`Hermes` 与 `DeepAgents` 的逻辑分工，以及把后面三章串起来的三个核心问题。这一章的目标是让你在进入第 1 章之前心里先有一张地图，避免学到一半还在问"我现在在哪"。

### 0.1 本讲在系列里的位置：从框架到产品

&emsp;&emsp;先打碎一个学员第一次接触 `Hermes Agent` 时最常出现的直觉——"`Hermes` 是不是 `DeepAgents` 的升级版？我用 `DeepAgents` 不就够了？"答案是：**不是升级，是另一种取舍**。`DeepAgents` 是一个 batteries-included 的开源框架，你拿到它就像拿到一组零件，要自己挑组合搭出 Agent；`Hermes` 是一个端到端的生产级产品，开箱即用的"自己产 skill / 跨平台一致响应 / 后台自动清理"——这些东西在 `DeepAgents` 框架层根本看不到。两者面向不同的工程姿态：一个让你自己组装并完全可控，一个把一组取舍打包给你减少决策成本。我们用一组对比直接看清差异。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 0-1 直觉打碎对照表（学员心里那个"升级版"假设 vs 实际差异）</font></p>
<div class="center">

| 心里的直觉假设 | 实际情况 | 你今天会在哪一节看到证据 |
|---|---|---|
| Hermes 是 DeepAgents 的"高级版"，功能更全 | Hermes 是另一个项目，工程姿态完全不同（产品级 vs 框架级）| 0.2 节 + 1.1/2.1 节对比表 |
| skill 都是人写的，Hermes 多了个"自动写"按钮 | Hermes 自动产 skill 是一条 prompt 驱动的机械闭环（计数器→fork→落盘）| 第 2 章全章 |
| 19 个平台覆盖说明 Hermes 更强大 | 19 个平台是"动态化高配"的代价证据，对应 AC 维度和入口治理维度的让步 | 3.2 节 + 3.5 节四维评价表 |
| 第三节学过 `DeepAgents`，本节 demo 是不是要重学一套新框架 | demo 用你已经熟悉的 `langchain AgentMiddleware`+`@tool` 实现，跟第二节《mini Harness》手搓 hook 链同构，没有新框架要学 | 2.8 节 demo 整段 |

</div>

&emsp;&emsp;打碎这个直觉后，你已经站在了今天课程的入口——不是问"哪个更好"，而是问"`Hermes` 选了什么、放弃了什么"。这正是这把**四维评价尺**要做的事。

&emsp;&emsp;回顾一下，前三节走的路径是**认知 → 实操 → 工程化**。第一节《认知地图》告诉你 harness 是什么、为什么需要它，给了你三支柱（`Context Engineering / Architectural Constraints / Garbage Collection`）和八大故障的诊断清单。第二节《mini Harness》让你亲手把 11 个机制搓出来，看到 `Baseline 1104 → Full 15053 tokens（13.6×）` 这个数字从你自己的代码里跑出来。第三节《DeepAgents 实战》让你用 `langchain-ai/deepagents` 框架 5 分钟搭一个生产级 Agent，把 11 个机制装进去。

&emsp;&emsp;今天本节是**从框架到产品**这一步——`Hermes Agent` 把"动态化"这条支柱推到了你在 `DeepAgents` 框架里见不到的程度：skill 库自己维护、跨 19 个平台一致响应、curator 后台定期清理。再往后，下一节我们会打开 `Curator` 的实现，并把 `Hermes` PDF 上的几个争议表述（`GEPA Skill Evolution` 和 "四层记忆是默认架构"）跟源码真相做完整对比——逐条标注哪些是真东西、哪些只是宣传话术、哪些被夸大了，最后用今天抽出的这把**四维尺**把四家 harness（`Claude Code` / `Codex` / `DeepAgents` / `OpenClaw`）放上同一张表做系列收束。

> 📌 **为什么是 Hermes 不是 Cursor/Aider/Cline**：`Cursor` / `Aider` / `Cline` 是 IDE/coding assistant 类产品（场景=代码编辑器内辅助），它们的 harness 设计目标是"协助工程师写代码"，主战场在 IDE 上下文，自学习闭环不是它们的核心卖点。本课需要一个 **"自学习闭环 + 记忆/Nudge/Curator"足够外显**的产品样本——agent 越跑越聪明、跨 19 平台一致响应、后台自动清理过期 skill，这些机制 `Cursor` 等产品要么不做、要么做但不开源。`Hermes` 是 **开源 + 自学习 + 多渠道** 三件齐备的稀有样本，正适合用来抽出这把**四维评价尺**，下一节复用到四家横评。换句话说：**`DeepAgents` 是你组装 agent 用的框架，`Hermes` 是你拆开源码看取舍用的产品实例**——这也是本节在五讲中的独特角色。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260508155348971.png" width=50%></div>

&emsp;&emsp;这里有一件事要先明确——本节是**读源码 + 跑 MVP demo**，不是从零敲新代码。学习模式跟第二节"亲手搓 mini Harness"、第三节"用 `DeepAgents` 搭生产 Agent"完全不同：你不需要写新框架，需要的是看懂别人怎么写、改个参数观察现象、然后用一把尺去评价它选的取舍。如果你期望"继续往 `DeepAgents` 框架里加东西"，那本节暂时不会满足这个期望——这件事要等到下一节重新衔接（`Curator` 的工作机制对 `DeepAgents` 用户有直接的借鉴价值）。

### 0.2 今天要回答的三个具体问题

&emsp;&emsp;`Hermes Agent` 这个项目最容易让人误解的一点是它"什么都做"——做了 19 个平台的对接、做了自己的记忆数据库、做了 skill 自动生产、做了后台清理服务。第一次接触它的学员经常问"我用 `DeepAgents` 不也行吗，干嘛要看这个？"。这个问题的答案不在功能清单里，在**取舍**里——`Hermes` 把第一节三支柱里的 `Garbage Collection`（熵减/动态化）这条线推到了能推的极致：让 agent 自己维护记忆、自己产 skill、自己跨平台、后台自己清。代价是另外两条评价维度(`Architectural Constraints` 偏信任端、入口治理偏分治端)都做出了明显让步，这一点我们会在第 3 章用证据展开。

&emsp;&emsp;为了让今天的 90 分钟课程有清晰的钩子，这节课围绕三个具体问题展开。**问题一**（第 1 章回答）：`Hermes` 怎么存记忆、搜历史？跟第三节的 `LangGraph state messages` 列表相比多了什么？**问题二**（第 2 章回答）：`Hermes` 怎么"自动生产 skill"？是真的"AI 自主学习"还是另有机制？**问题三**(第 3 章回答):用**四维评价尺**(3 问评价法 + 上下文工程,对应 L1 三支柱 + 入口治理补充维度)看，`Hermes` 选了什么、放弃了什么?三个问题构成第 1/2/3 章的主线，每章末尾都有一个 `vs DeepAgents` 的对比句帮你锁定差异。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 0-2 第三节 → 本节学习路径对比</font></p>
<div class="center">

| 维度 | 第三节 DeepAgents（已学） | 本节 Hermes（今天） |
|---|---|---|
| 课件性质 | 框架级，batteries-included | 生产级 harness，动态化轴高配 |
| 学员动作 | `create_deep_agent()` 搭 Agent | 读源码 + 跑 MVP demo + 改参数 |
| 记忆机制 | `LangGraph state messages` 列表 | `SQLite + FTS5 双索引 + LLM 摘要` |
| Skill 来源 | 手写 Markdown / 调 LLM 写 | `nudge` 计数器自动触发 fork agent |
| 平台覆盖 | 框架不约束（用户自接） | 19 platform adapter + 8 backend |
| 评价工具 | 4 大组件命名(`Middleware / Backend / 子代理 / HITL`) | **四维评价尺**(`GC 动态化 / AC 架构约束 / CE 上下文工程 / 入口治理`,前 3 项复用 L1 三支柱) |

</div>

> &emsp;&emsp;注意上表第一行——"另一种取舍，不是升级"。`DeepAgents` 是更接近 `framework` 的角色（你在它的封装上选组件），`Hermes` 是更接近 `product` 的角色（它已经选好了一组取舍并端到端落地）。两种姿态各有适用场景，今天我们要做的是看清 `Hermes` 选的这组取舍长什么样。

### 0.3 业界参照：Hermes vs OpenClaw——开源 harness 赛道的两条路径

> 📌 **【扩展阅读 · 可在学完 §3 四维框架后返回对照】**:本节用 `OpenClaw` 作为**业界坐标对照系**,提前用了一些四维评价语言(`GC` 动态化 / `AC` 信任端 / 入口治理分治)。如果你是首次接触 `Hermes`,可以**先快速浏览本节"两条路径"的核心结论**,等到第 3 章把**四维评价框架**建立完整后,再回头精读这张 8 维对比表会更有体感。第 3 章 §3.5 末尾会带你回到这张表做横评收束。

&emsp;&emsp;开课前还要补一个业界坐标。当你打开 `GitHub` 搜"AI agent harness"，会发现 `Hermes Agent` 不是这个赛道唯一的开源选项——更显眼的另一个项目叫 `OpenClaw`（logo 是只龙虾），截至 2026 年 5 月已经攒到 **368k stars**（同期 `Hermes` 约 40k；数据来源：各项目 GitHub 主页 stars 计数，查询日期 2026-05-05；stars 是动态数字仅供量级参考）。两个项目都是"个人 AI assistant + 多渠道 harness"定位，但走的工程路径完全不同。今天既然要深度学 `Hermes`，就有必要先建立"为什么不直接选 `OpenClaw`"的判断基础——这不是要你站队，是让你看清两条路径各自押注了什么。

&emsp;&emsp;最直观的差异在于 **skill 怎么来**。`OpenClaw` 的 skill 是工程师手写的 `SKILL.md` 文件，放在 `~/.openclaw/workspace/skills/<name>/` 目录里，用 `Git` 版本化，通过 `ClawHub` 社区市场分享——你可以理解成"agent 版的 npm registry"，社区贡献越多 skill 库越丰富。`Hermes Agent` 的 skill 是 nudge 机制自动产生的（你在课程目录里看到 2.x 节会专门讲这个），落盘到 `~/.hermes/skills/<name>/`，没有市场，全靠 agent 跑得越久 skill 库越丰富。一个押注**社区生态**，一个押注**自学习闭环**——这是这两条路径最根本的取舍分歧。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 0-3 Hermes vs OpenClaw 8 维对比（截至 2026 年 5 月）</font></p>
<div class="center">

| 维度 | OpenClaw | Hermes Agent |
|---|---|---|
| **工程姿态** | Personal AI Assistant（消费级、单用户）| 生产级 harness（agent 工程框架）|
| **技术栈** | `TypeScript` + `Node.js 24` + `pnpm` | `Python` + 自研 `AIAgent`（14201 行 `run_agent.py` + `openai`/`anthropic` SDK 直调，**不依赖 LangChain/DeepAgents**） |
| **Skill 生产** | 工程师手写 `SKILL.md` + `Git` 版本化 + `ClawHub` 社区市场 | `nudge` 自动产（计数器→fork→落盘，2.x 节展开）|
| **Skill 修改** | 覆盖整个文件 | `patch` 方法（只传 old/new string，更安全 + 省 token）|
| **学习闭环** | 无（每次任务从零开始，跑 100 次也不会更好）| 有（agent 跑越久 skill 库越丰富）|
| **客户端形态** | macOS / iOS / Android 三端原生 + Voice Wake + `Live Canvas`（A2UI 视觉工作区）| 主要 CLI / 服务端，无原生客户端 |
| **平台覆盖** | 24+ 消息渠道（WhatsApp / Telegram / iMessage / Feishu / WeChat / QQ / ...）| 19 platform + 8 backend |
| **社区生态** | OpenAI / GitHub / NVIDIA / Vercel 大厂 sponsor + ClawHub 市场（含 supply chain 风险）| 自维护无市场（也因此 0 reported agent CVE 截至 2026-04）|

</div>

&emsp;&emsp;读这张表的姿势——别盯着哪个"更好"的判断。两条路径有完全不同的适用场景：如果你要的是"在手机/桌面/Slack 都有官方原生入口、微信等中文渠道也有社区适配方案、开发者愿意贡献 skill"，`OpenClaw` 的工程取舍更对；如果你要的是"agent 跑得越久越聪明的自学习产品，愿意接受复杂部署和无现成 marketplace"，`Hermes` 的工程取舍更对。这两件事不是同一个产品能同时做到的——加 skill 自学习就要承担虚假产出风险（这是 2.5 节会展开的"`ACTIVE` 倾向"代价）；走社区市场就要面对 supply chain 攻击面。

> &emsp;**入口"开箱"程度提示**：`OpenClaw` 的渠道清单中，`Slack` / `Telegram` / `Discord` 是一方原生集成，`macOS` / `iOS` / `Android` 是原生 companion app；但 `WhatsApp`（走 `Baileys` QR pairing）、`iMessage`（走 `AppleScript` 或 `BlueBubbles` bridge）、`WeChat`（社区适配版，未在官方 integrations 页面列出）等渠道需要第三方桥接或中国社区适配方案，**不属于即装即用**——选型时按渠道粒度评估配置成本。

> &emsp;&emsp;**真实用户体验数据**（来自 `Reddit r/LocalLLaMA` 迁移用户反馈，仅作参考）：同一终端任务 `OpenClaw` 用 `50+` 次工具调用、`Hermes` 用 `5` 次完成（工具调用粒度差异——`Hermes` 单 `terminal` 工具覆盖面更广，可审计性更弱但快）；`Hermes` 每次 API 调用约 `73%` 是固定开销，光 `47` 工具的 schema 就占约 `14,000 tokens`——按 token 计费时可通过 `config.yaml` 的 `toolsets.disabled` 关掉不用的 `Toolset` 大幅降低固定开销。

&emsp;&emsp;**今天我们选 `Hermes` 作为研究对象的原因不是"它更好"，是"自学习闭环这条工程路径更值得拆开看"**——这个机制更复杂、更有迁移价值，下一节（第五节）末尾我们会用同一把**四维尺**把这两个项目跟其他两家一起放上同一坐标系横评，你会发现两者在四维上的位置差异比 stars 数差异更说明问题。3.5 节我们用四维表对比 `Hermes` 时也会顺手放一张 `OpenClaw` 的四维位置表，让你提前感受这把尺的横评力度。

### 0.4 环境准备

&emsp;&emsp;在进入第 1 章前，我们一次性把课件后续 demo 要用的环境配好。本课 §1.4（`LLM 摘要 demo`）和 §2.8（`MVP demo`）共用同一组 DeepSeek 凭证 + 同一个 `langchain` 客户端栈，按课件规范**只在第 0 章配置一次**——后续两个 demo 不再重复 `pip install` / `.env` / `load_dotenv` 三件套，直接 `os.getenv(...)` 读取即可。

&emsp;&emsp;**步骤一：创建 `.env` 文件（项目根目录）**

&emsp;&emsp;`.env` 文件内容如下——3 个变量是 §1.4 / §2.8 全集（`DEEPSEEK_API_KEY` 必填鉴权、`DEEPSEEK_MODEL` §1.4/§2.8 共用模型名、`DEEPSEEK_BASE_URL` §2.8 `ChatDeepSeek(api_base=...)` 参数；缺一会让其中一个 demo 跑不起来）：

In [31]:
# DeepSeek API 凭证（注册地址：https://platform.deepseek.com）
# DEEPSEEK_API_KEY=your_deepseek_api_key_here
# DEEPSEEK_MODEL=deepseek-chat
# DEEPSEEK_BASE_URL=https://api.deepseek.com

> ⚠️ **安全提醒**：`.env` 文件包含敏感凭证，切勿提交到 Git。请确保 `.gitignore` 中已包含 `.env`。如果你已经在第三节《`DeepAgents` 实战》里配过 `DEEPSEEK_API_KEY`，本课直接复用同一份 `.env` 即可，不必重新申请 key。

&emsp;&emsp;**步骤二：用 `requirements.txt` 一键装齐依赖**

&emsp;&emsp;课件目录下提供了 `requirements.txt`（跟 `lesson.ipynb` 同级），列出了本节所有第三方依赖（精确版本对齐第三节课堂环境）。一行命令装齐：

In [ ]:
!pip install -r requirements.txt -q

&emsp;&emsp;`requirements.txt` 核心内容预览（精确 `==` pin，避免跨学员环境漂移）：

```text
# Jupyter 环境
jupyter==1.1.1
notebook==7.2.2
ipykernel==6.29.5
nbformat==5.10.4

# Agent 框架核心（跟第三节同版本）
langchain==1.2.15
langchain-core==1.3.2

# LLM Provider（课堂默认 DeepSeek）
langchain-deepseek==1.0.1

# 通用工具
python-dotenv==1.0.1
```

> 📌 **国内镜像加速**：访问 PyPI 慢时改用清华源 `!pip install -r requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple -q`。

&emsp;&emsp;**步骤三：加载 `.env` + 关闭 LangSmith tracing + 验证**

&emsp;&emsp;下面是全课**唯一**的环境变量加载 cell——`find_dotenv(usecwd=True)` 兼容 notebook / 终端 / stdin 三种执行环境，避免裸 `load_dotenv()` 在子目录跑 notebook 时找不到 `.env`：

In [1]:
from dotenv import find_dotenv, load_dotenv
import os

# 从当前工作目录向上查找 .env（兼容 notebook / 终端 / stdin 三种执行环境）
env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)

# 教学 demo 不需要 LangSmith tracing；关闭它避免离线环境额外访问 api.smith.langchain.com
os.environ["LANGCHAIN_TRACING_V2"] = "false"

# 验证 key 已加载
api_key = os.getenv("DEEPSEEK_API_KEY")
if api_key:
    print(f"[OK] DEEPSEEK_API_KEY 加载成功（{api_key[:6]}...{api_key[-4:]}）")
    print(f"[OK] DEEPSEEK_MODEL = {os.getenv('DEEPSEEK_MODEL', 'deepseek-chat')}")
    print(f"[OK] DEEPSEEK_BASE_URL = {os.getenv('DEEPSEEK_BASE_URL', 'https://api.deepseek.com')}")
else:
    print("[FAIL] DEEPSEEK_API_KEY 未找到，请检查 .env 文件配置")

[OK] DEEPSEEK_API_KEY 加载成功（sk-d51...014d）
[OK] DEEPSEEK_MODEL = deepseek-chat
[OK] DEEPSEEK_BASE_URL = https://api.deepseek.com


&emsp;&emsp;执行后看到 3 行 `[OK]` 输出说明环境就绪，§1.4 / §2.8 后续 demo 全程复用这一份加载结果。如果看到 `[FAIL]`，回到步骤一检查 `.env` 内容、确保 `DEEPSEEK_API_KEY=` 后面填了真实 key（不是 placeholder `your_deepseek_api_key_here`）。

> 📌 **【贯穿全课声明】**：本节配置一次完成后，§1.4 `LLM 摘要 demo` 和 §2.8 `MVP demo` **不再重复** `pip install` / `.env` 配置 / `load_dotenv` 三件套——直接 `import os; os.getenv("DEEPSEEK_API_KEY")` 即可使用。如果你跳到具体章节单独跑，回到 §0.4 跑这 3 步再继续。

---

## <center>第1章：记忆系统——热/冷分离三段式架构</center>

&emsp;&emsp;这一章我们打开 `Hermes` 的第一个动态化机制：记忆系统。第三节课程结束时，你已经会用 `LangGraph state messages` 列表来管理对话历史——简单、轻量、嵌入 `LangGraph state` 拓扑里。但这个方案有它的天花板，`Hermes` 给出的回答是**热/冷分离 + 可插拔扩展**的三段式架构：**热记忆双层**（`MEMORY.md` Agent 自维护事实 + `USER.md` 用户画像，每次对话默认注入系统提示词；注意 `USER.md` 的**自动**填充依赖外部 `Honcho` Provider，默认关闭，所以默认状态下 `USER.md` 是空的或人工维护的）、**冷记忆三件套**（`SQLite + FTS5 + LLM 摘要`，按需检索）、**扩展层**（8 种外部 `Memory Provider`，全部 opt-in）。我们会按"问题引入 → 冷记忆三件套（SQLite / FTS5 / 摘要）→ 热记忆双层（MEMORY.md / USER.md）→ 8 种外部 Provider 扩展"的顺序逐层展开，每加一层就解一类问题，最后在板块末尾做一次 `vs DeepAgents` 的差异锁定。

### 1.1 问题引入：DeepAgents 的记忆缺了什么？

&emsp;&emsp;先把第三节学到的方案重新拿到台面上。在 `DeepAgents`（以及 `LangGraph` 大多数 agent 实现）里，记忆就是 `state` 字段里的 `messages: List[BaseMessage]`——每来一个 user turn 就 `append` 一个 `HumanMessage`，每个 `tool` 调用结果就 `append` 一个 `ToolMessage`。这个方案在短对话、单 session 场景下完全够用，但放到生产环境会立刻暴露三个具体的断裂点。

- **第一个断裂点是无全文搜索**。`messages` 是个 `Python list`，要"找昨天我跟 Agent 聊过 `kafka rebalance` 这事的所有 turn"，你只能 `for msg in messages: if "kafka" in msg.content` 线性扫描，1000 条以上就开始慢，10000 条以上完全不可接受。

- **第二个断裂点不在"能不能跨 session 持久"本身**——`DeepAgents` 通过 `create_deep_agent(checkpointer=..., store=...)` 两个参数已经支持持久化(`Checkpointer` 处理 per-thread 续上 + `Store` 处理 cross-thread 长期记忆,API 走 `runtime.store.put / asearch / aget`)——而是**官方 `BaseStore` 内置实现(`InMemoryStore` / `PostgresStore` / `SqliteStore` 来自 `langgraph-checkpoint-sqlite` 独立包)都是通用 `(namespace, key, value)` 长期记忆,不带 FTS5 keyword 全文搜索 + 召回式 LLM 归纳**。如果部署场景是"单机本地 `SQLite` + 按内容跨 session 全文搜索 + 召回后 LLM 归纳",官方 `SqliteStore` 给的是通用 KV 存储,这条路要自己实现 `BaseStore` 子类(在 SQLite 后端上加 FTS5 索引 + 召回式摘要层)。

- **第三个断裂点也不是"没有 LLM 摘要"**——`DeepAgents` 默认 middleware 链就内置 `create_summarization_middleware`(`graph.py` 自动装入 `deepagent_middleware`),它解决的是**对话超长时压缩当前 session 的 context 防爆**——而是**没有"按内容召回历史 session 后做 LLM 归纳"这一档**。这两件事看起来都叫"摘要",但工作时机和目的完全不同:`DeepAgents` 内置 summarization 是**当前 session 内的 context 压缩**(防 100K tokens 爆),`Hermes` 的 LLM 摘要是**跨 session 检索命中后对召回结果的二次归纳**(在 `session_search_tool.py` 里跟 FTS5 双索引绑定,目的是"召回最相关 N 段 + 摘成结构化的 5 元素结果")。

> **【常见误区】**：很多学员看到 `LangGraph` 的 `messages` 列表加上 `SqliteSaver` 之后，会以为"持久化已经解决了"。但 `SqliteSaver` 解决的是**"重启后能不能续上同一个 thread"**(per-thread 短期记忆),它不解决**"跨 thread 按内容搜"**这件事。`thread_id` 是按 session 分桶的钥匙，不是搜索索引。要做"跨 thread 按内容搜",得走 `Store`(cross-thread 长期记忆)那一层 — 但官方 `Store` 内置实现(`InMemoryStore` / `PostgresStore` / `SqliteStore`)都是通用 KV 存储,**不带 FTS5 全文搜索能力**,所以"按内容跨 session 召回"这条路要么自己加 keyword 索引层,要么自己实现 `BaseStore` 子类。

> **【概念精修：用 `LangChain 1.0+` 双轨术语精确说】**：上述断裂点更精确的表达是 — `LangChain 1.0+` 的 `Checkpointer`(per-thread,`SqliteSaver` 属于此层)+ `Store`(cross-thread,`InMemoryStore` / `PostgresStore` / `SqliteStore` 三种官方实现)**共同覆盖了"跨 session 持久"这个 contract**, `DeepAgents` 通过 `create_deep_agent(checkpointer=..., store=...)` 已经把 contract 暴露出来。但官方 `Store` 内置实现**都是通用 `(namespace, key, value)` KV 存储,不带 FTS5 keyword 全文搜索**;且原生 `summarization middleware` 服务的是 **当前 session 内 context 压缩**(防爆 token),不是 **跨 session 召回后归纳**(配合检索用)。`Hermes` 的 `SQLite + FTS5 双索引 + 召回式 LLM 摘要` 三件套,工程上等价于"在 SQLite 后端基础上自己加 FTS5 keyword 搜索层 + 面向召回结果的二次归纳层"——跟 `LangChain 1.0+` 的 `Store` 抽象是**同一概念分层、整合方案不同 + 摘要时机不同**。这也是 `Hermes` 选自研的原因之一:官方 SqliteStore 是通用 KV、官方 summarization 不绑检索,走"本地 SQLite + 按内容召回 + 召回后归纳"这条整合路线就得自己造轮子。

&emsp;&emsp;`Hermes` 不是把这三个断裂点用一个魔法 API 一次性解决，它的做法是叠了三层结构——`SQLite` 解第二个（持久化），`FTS5` 解第一个（全文搜索），`LLM 摘要` 解第三个（命中后归纳）。我们一层一层拆。

&emsp;&emsp;先用一张运行时流程图把后面几节要拆的组件串起来。注意这张图画的是**一次 user turn 里记忆怎么参与运行**，不是静态模块清单：热记忆先默认进入 prompt，冷记忆只有在 Agent 判断需要历史信息时才走 `session_search`，而 `Nudge` 对 `MEMORY.md` 的更新发生在 turn 结束之后的后台 review。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260508155349012.png" width=50%></div>

### 1.2 第一层：SQLite 持久化存储

&emsp;&emsp;我们先看最底层。`Hermes` 的核心存储不是内存 list，是 `SQLite` 文件——这是它跟 `DeepAgents` 默认 `state messages` 的第一处差别。打开 `hermes_state.py:104-117`，你会看到 `messages` 表的 schema 定义（这里我们只展示与教学相关的关键字段，完整 schema 见源码）。

In [ ]:
# hermes_state.py 节选（教学示意，完整 schema 见源码 L88-L156）
# 以下为 SQLite messages 表 DDL + 复合索引定义
SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS messages (
    id INTEGER PRIMARY KEY AUTOINCREMENT, -- 行级唯一主键，自增；FTS5 表 rowid 也用它
    session_id TEXT NOT NULL,             -- 跨 session 持久化的分桶 key，按它 + timestamp 走索引
    role TEXT NOT NULL,                   -- user / assistant / tool / system
    content TEXT,                         -- 消息正文
    tool_name TEXT,                       -- 若 role=tool，记录工具名（FTS5 trigger 会一并索引）
    tool_calls TEXT,                      -- JSON 序列化的 tool_calls 列表（也会进 FTS5 索引）
    timestamp REAL NOT NULL               -- Unix epoch float，(session_id, timestamp) 复合索引的排序键
);

-- 复合索引：按 session_id 分桶后按 timestamp 排序，支撑"拉某 session 最近 N 条"查询走索引
CREATE INDEX IF NOT EXISTS idx_messages_session ON messages(session_id, timestamp);
"""

&emsp;&emsp;这段代码定义的就是 `Hermes` 记忆的"账本"。每条消息是一行，按 `session_id` 分桶，按 `timestamp` 排序——结构上跟 `LangGraph` 的 `messages` 列表几乎一样，差别只在它落在磁盘上的 `SQLite` 文件里而不是 Python 进程的内存里。注意 `session_id TEXT NOT NULL` 这个字段，它是跨 session 持久化的关键——重启后只要这个 SQLite 文件还在，所有历史 session 都还在。`(session_id, timestamp)` 复合索引让按时间倒序拉某个 session 的最近 N 条消息可以走索引、不需要全表扫描。

> &emsp;&emsp;这里要明示一个学员经常混淆的点——`SQLite` 不是独立软件，它是 `Python` 标准库 `sqlite3` 自带的嵌入式数据库引擎，单文件、零配置。"持久化"在这里就是字面意义的"消息写到磁盘 `.db` 文件里"，进程退出文件还在；这跟 `LangGraph` 内存 list 的根本区别就是这一句话。

&emsp;&emsp;到这里我们已经解决了"跨 session 持久"问题。但 `messages` 表本身只是结构化存储，它不解决"按内容搜索"——一句 `SELECT * FROM messages WHERE content LIKE '%kafka rebalance%'` 在百万级数据下慢得无法接受。所以我们进入第二层。

### 1.3 第二层：FTS5 双索引——全文搜索引擎

&emsp;&emsp;`SQLite` 内置了一个叫 `FTS5`（Full-Text Search v5）的虚拟表扩展——它不是独立软件，不需要额外安装，绝大多数发行版的 Python `sqlite3` 都默认编译启用了它（少数自编译/精简版可能没启用，遇到 `SQL logic error: no such module: fts5` 时检查 `import sqlite3; print(sqlite3.sqlite_version)` 并确认 SQLite 编译时带 `-DSQLITE_ENABLE_FTS5`）。`Hermes` 不只用了 `FTS5`，而且用了**两张** `FTS5` 表：`messages_fts`（默认 `unicode61` tokenizer）和 `messages_fts_trigram`（`trigram` tokenizer）。**这里要先做一个术语区分**——本节的 `tokenizer` 指的是 `FTS5` 在数据库层把字符串切成可索引 token 的算法（unicode61 / trigram 等），跟 LLM 推理时把 prompt 切成 token id 的 `tokenizer`（BPE/SentencePiece 等）不是同一回事，两者只是同名。我们先看 schema 摘录。

In [ ]:
# hermes_state.py:103-156 节选（双 FTS5 表 + 自动同步 trigger）
# 以下为 messages_fts（unicode61）+ messages_fts_trigram（trigram）双索引 DDL
FTS_SQL = """
-- 默认 unicode61 tokenizer 的 FTS5 虚拟表，按词切分，适合英文/拉丁语系全文搜索
CREATE VIRTUAL TABLE IF NOT EXISTS messages_fts USING fts5(                                                 
    content                                                   -- 唯一被索引列；rowid 与 messages.id 对齐  
);
-- 三个 trigger 让应用层只写一张 messages 表，FTS5 索引自动同步（避免双写忘记）

CREATE TRIGGER IF NOT EXISTS messages_fts_insert AFTER INSERT ON messages BEGIN
    -- INSERT trigger：新消息进来时把 content+tool_name+tool_calls 拼接后一并索引
    -- 这样既能搜消息正文，也能用工具名当 query 召回所有调用过该工具的消息

    INSERT INTO messages_fts(rowid, content) VALUES (
        new.id,
        COALESCE(new.content, '') || ' ' || COALESCE(new.tool_name, '') || ' ' || COALESCE(new.tool_calls, '')
    );
END;
-- 省略 messages_fts_delete / messages_fts_update（结构对称：DELETE 触发对应 rowid 删除；UPDATE 触发先 delete 再 insert，保持 FTS5 索引始终镜像 messages 表）
"""

FTS_TRIGRAM_SQL = """
-- trigram tokenizer 的 FTS5 虚拟表，按连续 3 字符滑窗切分，对中文子串匹配友好
CREATE VIRTUAL TABLE IF NOT EXISTS messages_fts_trigram USING fts5(
    content,
    tokenize='trigram'                     -- 关键差异：trigram tokenizer（每 3 个字符为一组，滑窗式拆 token）
);
-- 同样配 INSERT/DELETE/UPDATE 三个 trigger，结构与上面 messages_fts 对称
-- 双索引并存：search_messages 按 query 特征自动选用 unicode61 还是 trigram（CJK 且长度≥3 走 trigram）
"""

&emsp;&emsp;这段代码做了三件事。

- 第一，定义两个 `VIRTUAL TABLE`——`messages_fts` 用默认的 `unicode61` tokenizer，按 Unicode 词法切分（对英文/拉丁语系友好；遇到 CJK 字符会退化成逐字分词，子串匹配能力差）；`messages_fts_trigram` 用 `trigram` tokenizer，按连续 3 字符滑窗切分（对中文这种"无空格分词"的语言友好）。

- 第二，每张 FTS5 表挂了三个 trigger，让 `messages` 表的 INSERT/DELETE/UPDATE 自动同步到对应 FTS5 表——也就是说应用层不需要"双写"，写 `messages` 一张表就够了。

- 第三，注意 `INSERT INTO messages_fts(rowid, content) VALUES (new.id, COALESCE(new.content, ...) || ' ' || COALESCE(new.tool_name, ...))` 这段——它不只索引 `content`，还把 `tool_name` 和 `tool_calls` 拼进来一起索引，这意味着你可以用工具名当 query 召回所有调用过该工具的消息。

&emsp;&emsp;但还有第四件事课件之前没点透——**两张影子索引表怎么和真表对应起来**？答案是 `rowid` 桥。`FTS5` 虚拟表自带一个隐藏的 `rowid` 列（就像普通表的主键，但默认不在 schema 里写出来），上面 trigger 里那句 `INSERT INTO messages_fts(rowid, content) VALUES (new.id, ...)` 的精髓就是把 `messages` 表的主键 `id` 当成 `FTS5` 表的 `rowid` 写进去。这样一来：`FTS5` 表本身只存"这个 token 在 rowid=42 这条记录里"，要拿 `messages` 表的 14 个真实字段（`session_id` / `role` / `tool_name` / ...）必须用 `JOIN messages m ON m.id = messages_fts.rowid` 反向查回去——这正是后面 §1.3 末尾 demo 里 `SELECT` 语句反复出现 `JOIN ... ON m.id = messages_fts.rowid` 的原因。<font color=red>记住这个 mental model：messages 是真表，两张 FTS5 是影子索引；它们之间没有外键，全靠 rowid 桥联通。</font>

> **【常见误区】**:学员经常把 `unicode61` 和 `trigram` 当成"二选一"。实际上 `Hermes` 的设计是**写入端无路由、查询端做分流**——这两件事必须分开理解,常见误区往往把它们混成一件:
> - **写入端**:同一条消息**同时**写两张 FTS5 表,`messages_fts_insert` 和 `messages_fts_trigram_insert` 都是 `AFTER INSERT ON messages`,**都没有 `WHEN` 子句**,内容表达式完全相同——SQLite 引擎层不做任何语言判定。
> - **查询端**:根据 query 字符特征**三档分支**选用哪张表(纯英文 / 含 CJK / 兜底),由 `_sanitize_fts5_query` 之后的分支逻辑决定(下方 cell 详解)。
> 一句话记忆:**写时全冗余,读时分流**。判断是否"二选一"已经在 query 这一刻才发生,而消息进库时两张索引一定都已经有了对应行——这是 SQLite trigger 引擎层保证的。

> **【概念解释·CJK 是什么】**:**CJK = `C`hinese / `J`apanese / `K`orean** 的缩写,是 Unicode 标准里指代**中日韩文字**的术语(三种语言共享大量汉字字符,Unicode 把它们的共同字符放在 "CJK Unified Ideographs" 区段 U+4E00–U+9FFF)。这里判定 CJK 的目的是**检测当前搜索 query 是不是中日韩文字,以决定走哪条索引** — 因为中日韩**没有词间空格**(英文写 "kafka rebalance" 有空格分词,中文写"机器人调试"是连续无空格的),常规 `unicode61` tokenizer 看不到词边界会**降级到字符级**(把每个 CJK 字当独立 token),搜"机器人"就会变成搜任何含"机"或"器"或"人"的消息(精度灾难);而 `trigram` 按 UTF-8 字节滑窗切 token,1 个 CJK 字符占 3 bytes,**需要 ≥9 UTF-8 bytes(=3 个 CJK 字)才能切出第一个完整 token**,所以 1-2 字 CJK query 必须走 `LIKE` 全表扫描兜底。下面三档分支判定就是基于这个语言特性差异——非 CJK 走 `unicode61`、CJK ≥3 字走 `trigram`、CJK 1-2 字走 `LIKE`。

&emsp;&emsp;搜索路径的分支判定可以用一个具体例子讲清楚——假设你搜中文词"机器人"，3 字符长度。如果走 `messages_fts`（默认 `unicode61`），中文在没有空格分隔的情况下子串匹配能力很弱，搜出来的结果可能完全错位（漏掉真正连续出现"机器人"的消息）。所以 `Hermes` 在 `search_messages` 里判定的实际是**三档分支**而不是二档（源码 `hermes_state.py:1748-1830`）：

- ① **非 CJK** → 走 `messages_fts`（unicode61），按 Unicode 词法 token 匹配；

- ② **CJK 且 ≥ 3 字符** → 改写 SQL 走 `messages_fts_trigram`，按连续 3 字符滑窗精准匹配子串（trigram 需要 ≥9 UTF-8 bytes = 3 个 CJK 字才能切出第一个 token）；

- ③ **CJK 但只有 1-2 字符** → 既不走 `unicode61` 也不走 `trigram`，**回退到 `m.content LIKE '%...%'` 全表扫描**作为兜底（`:1804-1834`）。这是为什么需要两张表加一档兜底的根本原因——任何一张单独都解决不了"全 Unicode 全长度"的搜索覆盖，而 `trigram` 的"≥3 CJK 字"硬约束逼着系统留一条 LIKE 后路。

In [ ]:
# hermes_state.py:1586-1842 节选（_sanitize_fts5_query 防注入示意）
def _sanitize_fts5_query(query: str) -> str:
    """
    教学示意：去掉 FTS5 保留字符（双引号、负号、星号、AND/OR/NOT 等），
    防止 query 里的特殊符号被解析成 FTS5 操作符引发 syntax error 或注入。
    完整实现见 hermes_state.py:1586。
    """
    # 1. 转义双引号
    # 2. 移除单独出现的 - 号、* 号
    # 3. 把 AND/OR/NOT 关键字降级为普通词
    return cleaned_query

&emsp;&emsp;`_sanitize_fts5_query` 这个函数是 `Hermes` 防 FTS5 注入的关键防线——FTS5 query 语法本身支持 `"..."` / `-` / `*` / `AND` / `OR` / `NOT` 这些操作符，如果用户输入包含它们，没有清洗会直接让 FTS5 抛 syntax error 或返回完全不符合预期的结果。这一点在 `Hermes` 这种"agent 自动调用 search 工具、query 来自 LLM 生成"的场景尤其重要——LLM 会随便生成各种特殊字符。<font color=red>实战中如果你也要嵌入 FTS5 全文搜索，sanitize 这一层不能省。</font>

&emsp;&emsp;到这里 §1.3 讲解的 4 件事——双 `FTS5` 表、6 个 trigger、`rowid` 桥、三档分支——已经形成完整的 mental model。但纯文字描述容易停留在"看起来懂了"的层面，下面给一段不依赖 `Hermes` 主仓库、用 `Python` 标准库 `sqlite3` 就能跑通的最小化 demo，让你**亲手插数据、亲手观察 FTS5 影子索引自动同步、亲手对比双 tokenizer 的行为差异**。这段 demo 的 schema 是 §1.3 真实源码的教学最简化版（去掉 `sessions` 外键 + 11 个不影响双索引演示的字段），确保 cell 之间顺序运行无依赖问题。

**动手验证：双索引 + rowid 桥 + trigger 自动同步**

**步骤一：建表 + trigger（教学最简化 schema）**

&emsp;&emsp;先建一个内存 `sqlite3` 数据库，配 1 张真表（`messages`）+ 2 张 `FTS5` 影子索引表 + 4 个 `insert/delete` trigger。`update` trigger 与 `insert/delete` 同构，本 demo 省略不影响验证主线。

In [2]:
import sqlite3

# 内存 SQLite，演示完即销毁
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row

# 教学最简化 schema：去掉 sessions 外键，只保留 4 个核心字段
conn.executescript("""
    CREATE TABLE messages (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        content TEXT,
        tool_name TEXT,
        tool_calls TEXT
    );

    -- 影子索引 A：默认 unicode61 tokenizer
    CREATE VIRTUAL TABLE messages_fts USING fts5(content);

    -- 影子索引 B：trigram tokenizer（CJK 子串友好）
    CREATE VIRTUAL TABLE messages_fts_trigram USING fts5(
        content,
        tokenize='trigram'
    );

    -- A 表 insert/delete trigger（new.id → rowid 桥）
    CREATE TRIGGER messages_fts_insert AFTER INSERT ON messages BEGIN
        INSERT INTO messages_fts(rowid, content) VALUES (
            new.id,
            COALESCE(new.content, '') || ' ' || COALESCE(new.tool_name, '') || ' ' || COALESCE(new.tool_calls, '')
        );
    END;
    CREATE TRIGGER messages_fts_delete AFTER DELETE ON messages BEGIN
        DELETE FROM messages_fts WHERE rowid = old.id;
    END;

    -- B 表 insert/delete trigger（同构镜像）
    CREATE TRIGGER messages_fts_trigram_insert AFTER INSERT ON messages BEGIN
        INSERT INTO messages_fts_trigram(rowid, content) VALUES (
            new.id,
            COALESCE(new.content, '') || ' ' || COALESCE(new.tool_name, '') || ' ' || COALESCE(new.tool_calls, '')
        );
    END;
    CREATE TRIGGER messages_fts_trigram_delete AFTER DELETE ON messages BEGIN
        DELETE FROM messages_fts_trigram WHERE rowid = old.id;
    END;
""")

print("[OK] 1 真表 + 2 FTS5 影子索引 + 4 trigger 就绪")

[OK] 1 真表 + 2 FTS5 影子索引 + 4 trigger 就绪


&emsp;&emsp;执行后会看到一行 `[OK]` 输出，说明 5 句 DDL 都跑通了。注意 trigger 里那句 `INSERT INTO messages_fts(rowid, content) VALUES (new.id, ...)`——这就是 §1.3 反复强调的 `rowid` 桥：把 `messages.id` 当 `FTS5` 表的隐藏 rowid 写进去，未来 SELECT 时就靠这条暗线 JOIN 回真表。

**步骤二：插入数据 → 验证 trigger 自动同步 + content 三段拼接**

&emsp;&emsp;接下来插 3 条 `messages`，**只 INSERT 真表，不写任何 FTS5**，然后分别 SELECT 三张表，看 trigger 是否真的自动把数据同步进了两张影子索引、且写入的 content 是不是 `content + tool_name + tool_calls` 三段拼接。

In [3]:
# 应用层只写 messages 一张表，FTS5 由 trigger 自动同步
test_data = [
    ("搞定 kafka rebalance 问题", "search_logs", '[{"name":"search_logs"}]'),
    ("docker deployment failed", "exec_shell", '[{"name":"exec_shell"}]'),
    ("机器人控制系统调试", "robot_api", '[{"name":"robot_api"}]'),
]
conn.executemany(
    "INSERT INTO messages (content, tool_name, tool_calls) VALUES (?, ?, ?)",
    test_data,
)
conn.commit()

print("[INFO] messages 真表（只有原始 content 字段）：")
for r in conn.execute("SELECT id, content FROM messages"):
    print(f"   id={r['id']}  content={r['content']}")

print("\n[INFO] messages_fts 影子索引（自动同步，content 是三段拼接结果）：")
for r in conn.execute("SELECT rowid, content FROM messages_fts"):
    print(f"   rowid={r['rowid']}  content={r['content']}")

print("\n[INFO] messages_fts_trigram 影子索引（同步同样的拼接结果）：")
for r in conn.execute("SELECT rowid, content FROM messages_fts_trigram"):
    print(f"   rowid={r['rowid']}  content={r['content']}")

[INFO] messages 真表（只有原始 content 字段）：
   id=1  content=搞定 kafka rebalance 问题
   id=2  content=docker deployment failed
   id=3  content=机器人控制系统调试

[INFO] messages_fts 影子索引（自动同步，content 是三段拼接结果）：
   rowid=1  content=搞定 kafka rebalance 问题 search_logs [{"name":"search_logs"}]
   rowid=2  content=docker deployment failed exec_shell [{"name":"exec_shell"}]
   rowid=3  content=机器人控制系统调试 robot_api [{"name":"robot_api"}]

[INFO] messages_fts_trigram 影子索引（同步同样的拼接结果）：
   rowid=1  content=搞定 kafka rebalance 问题 search_logs [{"name":"search_logs"}]
   rowid=2  content=docker deployment failed exec_shell [{"name":"exec_shell"}]
   rowid=3  content=机器人控制系统调试 robot_api [{"name":"robot_api"}]


&emsp;&emsp;输出会展示三个关键现象：① 应用层一句 `INSERT` 都没碰过 `FTS5`，但两张影子索引表都有了对应行，证明 **trigger 真的在 SQLite 引擎层自动双写**；② FTS5 表的 `rowid` 1/2/3 完全对应 `messages` 表的 `id` 1/2/3，证明 **rowid 桥**生效；③ `messages.content` 字段只有"搞定 kafka rebalance 问题"，但 `FTS5` 表的 content 是"搞定 kafka rebalance 问题 search_logs [{...}]"——证明 trigger 的 `COALESCE` 拼接真的把 `tool_name` 和 `tool_calls` 也索引进去了，这就是为什么搜工具名也能召回消息的源头。

&emsp;&emsp;再注意一个细节:两个 trigger 都是 `AFTER INSERT ON messages`,**都没有 WHEN 条件子句**——也就是说,SQLite 引擎层不会去判断"这条消息是中文还是英文",任何 `INSERT` 都会无条件双写到两张索引表。语言层面的分流被推迟到了**查询时**(三档分支),写入端保持极简、零判断。这也是为什么删除时也是双写——下面 步骤五 会验证 `DELETE` 同样没有 WHEN。

**步骤三：rowid 桥实战 —— FTS5 没有真表字段，必须 JOIN 回去**

&emsp;&emsp;`FTS5` 影子索引表只有一个 `content` 列——它压根不存 `tool_name` / `timestamp` / `session_id` 这些真表字段。所以业务层做 search 时，必须用 `JOIN messages m ON m.id = messages_fts.rowid` 通过 `rowid` 桥反向查回真表，才能拿到完整字段。下面演示用关键词 `kafka` 做一次 unicode61 全文搜索，并用工具名 `search_logs` 直接当 query 验证"用工具名召回消息"。

In [5]:
# 标准搜索 SQL —— 通过 rowid 桥 JOIN 回真表拿完整字段
search_sql = """
    SELECT
        m.id, m.content, m.tool_name,
        snippet(messages_fts, 0, '>>>', '<<<', '...', 40) AS snippet
    FROM messages_fts
    JOIN messages m ON m.id = messages_fts.rowid
    WHERE messages_fts MATCH ?
"""

print("[QUERY] query='kafka'（关键词搜索）：")
for r in conn.execute(search_sql, ("kafka",)):
    print(f"   id={r['id']}  tool_name={r['tool_name']}  snippet={r['snippet']}")

print("\n[QUERY] query='search_logs'（用工具名当 query 召回消息）：")
for r in conn.execute(search_sql, ("search_logs",)):
    print(f"   id={r['id']}  tool_name={r['tool_name']}  snippet={r['snippet']}")

[QUERY] query='kafka'（关键词搜索）：
   id=1  tool_name=search_logs  snippet=搞定 >>>kafka<<< rebalance 问题 search_logs [{"name":"search_logs"}]

[QUERY] query='search_logs'（用工具名当 query 召回消息）：
   id=1  tool_name=search_logs  snippet=搞定 kafka rebalance 问题 >>>search_logs<<< [{"name":">>>search_logs<<<"}]


&emsp;&emsp;两次搜索都会命中 `id=1` 那条记录——第一次匹配的是消息正文里的 `kafka`，第二次匹配的是 trigger 拼进 `FTS5` 索引里的 `tool_name="search_logs"`。这就是双索引 + 三段拼接的真实力量：**search 工具不止能找消息内容，还能反查"哪些消息调用过某个工具"**。同理，如果 `messages` 里塞一条 content 含 `session.timeout.ms` 的消息，搜 `session.timeout` 也能命中——`unicode61` 把非字母字符 `.` 当分词符切成 `["session", "timeout"]`，phrase 查询按连续 token 匹配。

**步骤四：双 tokenizer 行为差异 —— CJK 三档分支真现场**

&emsp;&emsp;现在演示三档分支：① 搜中文 `机器人`（3 字符）走 trigram 精准命中；② 搜中文 `机`（1 字符）trigram 切不出 token，要么报错要么返回空；③ 此时回退 `LIKE` 全表扫描兜底。

In [6]:
sql_tri = """
    SELECT m.id, m.content
    FROM messages_fts_trigram
    JOIN messages m ON m.id = messages_fts_trigram.rowid
    WHERE messages_fts_trigram MATCH ?
"""

# 档 ②：CJK ≥ 3 字符 → trigram 精准匹配
print("[QUERY] 档 ②：query='机器人'（3 个 CJK 字符）走 trigram：")
for r in conn.execute(sql_tri, ('"机器人"',)):
    print(f"   [HIT] trigram 命中 id={r['id']}  content={r['content']}")

# 档 ③：CJK 1-2 字符 → trigram 不够长
print("\n[QUERY] 档 ③：query='机'（1 个 CJK 字符）走 trigram：")
try:
    rows = list(conn.execute(sql_tri, ('"机"',)))
    if rows:
        print(f"   trigram 返回 {len(rows)} 条")
    else:
        print(f"   [WARN] trigram 返回 0 条（trigram 需要 ≥3 字符 = 9 UTF-8 bytes 才能切出 token）")
except sqlite3.OperationalError as e:
    print(f"   [ERROR] trigram 报错：{e}")

# 档 ③ 兜底：回退 LIKE 全表扫描
print("\n[QUERY] 档 ③ 兜底：m.content LIKE '%机%' 全表扫描：")
for r in conn.execute("SELECT id, content FROM messages WHERE content LIKE ?", ("%机%",)):
    print(f"   [HIT] LIKE 命中 id={r['id']}  content={r['content']}")

[QUERY] 档 ②：query='机器人'（3 个 CJK 字符）走 trigram：
   [HIT] trigram 命中 id=3  content=机器人控制系统调试

[QUERY] 档 ③：query='机'（1 个 CJK 字符）走 trigram：
   [WARN] trigram 返回 0 条（trigram 需要 ≥3 字符 = 9 UTF-8 bytes 才能切出 token）

[QUERY] 档 ③ 兜底：m.content LIKE '%机%' 全表扫描：
   [HIT] LIKE 命中 id=3  content=机器人控制系统调试


&emsp;&emsp;这一步是整个 demo 最关键的"现象现场"——你会看到 trigram 在 1 个 CJK 字符 query 时**真的会返回空集或抛 `sqlite3.OperationalError`**，证明课件里说的"≥9 UTF-8 bytes 硬约束"不是理论而是会真实触发的边界。这就是为什么 `Hermes` 在 `hermes_state.py:1804-1834` 必须留一条 `LIKE` 兜底路径——**没有这一档，1-2 字符 CJK query 会让 search 工具直接报错或返回空，agent 拿到的就不是"找不到"而是"出错了"，体验完全不一样。**

**步骤五：DELETE trigger 一致性 —— 删 messages 自动清 FTS5**

&emsp;&emsp;最后验证 trigger 的另一面：删除 `messages` 一行，两张 `FTS5` 影子索引表对应 rowid 的行是否会被 `delete trigger` 自动清掉。

In [7]:
print("[BEFORE] 删除前两张 FTS5 表都有 id=3：")
print(f"   messages_fts:        rowid={[r['rowid'] for r in conn.execute('SELECT rowid FROM messages_fts')]}")
print(f"   messages_fts_trigram: rowid={[r['rowid'] for r in conn.execute('SELECT rowid FROM messages_fts_trigram')]}")

# 应用层只删 messages 真表
conn.execute("DELETE FROM messages WHERE id = 3")
conn.commit()

print("\n[AFTER] DELETE FROM messages WHERE id=3 后（应用层未碰 FTS5 表）：")
print(f"   messages_fts:        rowid={[r['rowid'] for r in conn.execute('SELECT rowid FROM messages_fts')]}")
print(f"   messages_fts_trigram: rowid={[r['rowid'] for r in conn.execute('SELECT rowid FROM messages_fts_trigram')]}")
print("\n[OK] id=3 在两张 FTS5 表都自动消失，证明 delete trigger 同步生效")

[BEFORE] 删除前两张 FTS5 表都有 id=3：
   messages_fts:        rowid=[1, 2, 3]
   messages_fts_trigram: rowid=[1, 2, 3]

[AFTER] DELETE FROM messages WHERE id=3 后（应用层未碰 FTS5 表）：
   messages_fts:        rowid=[1, 2]
   messages_fts_trigram: rowid=[1, 2]

[OK] id=3 在两张 FTS5 表都自动消失，证明 delete trigger 同步生效


&emsp;&emsp;输出对照会显示删除前 `FTS5` 表 rowid 列表是 `[1, 2, 3]`，删除后变成 `[1, 2]`——应用层只动了 `messages` 一张真表，但两张影子索引表的 `id=3` 行同时消失。这就是"`FTS5` 索引始终镜像 `messages` 表"的工程意义：**应用层永远不需要担心索引漂移，trigger 是最后的一致性防线**。如果未来你把 `FTS5` 替换成 `Elasticsearch` / `Tantivy` 等外部索引，这个"trigger 自动同步"机制就消失了，你必须自己在应用层写双写或定期重建索引——这是工程取舍的核心成本来源。

> &emsp;**【可选练习 · 课后】**：把 demo 改成 `Hermes` 的真实 schema（加上 `sessions` 外键 + 完整 14 字段 + `update` trigger），跑一次"先 INSERT、再 UPDATE 同一行 content、再 SELECT FTS5"——你会观察到 `update trigger` 是 "delete + insert" 两步而不是真的修改，这是 `FTS5` 没有原地 update 能力的工程现实，也是 `Hermes` 选择 trigger 同步而不是手动维护索引的另一个原因。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260508155348950.png" width=50%></div>

### 1.4 第三层：LLM 摘要层——命中后异步摘要

&emsp;&emsp;前两层解决了"存得住、搜得到"，但还有一个问题——当一次 query 命中 50 条历史消息、累计 30K tokens 时，把它们全部塞回 prompt 还是会爆 context。所以 `Hermes` 在 `tools/session_search_tool.py:196-265` 的 `_summarize_session` 函数里加了第三层：**LLM 摘要层**，命中后异步用 LLM 把每个 session 压成一个 5 元素的结构化摘要。我们先看摘要的 prompt。

In [ ]:
# tools/session_search_tool.py:196-260 节选（_summarize_session 摘要 prompt）
async def _summarize_session(conversation_text, query, session_meta):
    """
    异步摘要单个 session，焦点是 search query。
    完整实现见 session_search_tool.py:198-260。
    """
    system_prompt = (
        "You are reviewing a past conversation transcript to help recall what happened. "
        "Summarize the conversation with a focus on the search topic. Include:\n"
        "1. What the user asked about or wanted to accomplish\n"        # 用户做什么
        "2. What actions were taken and what the outcomes were\n"        # 实际执行
        "3. Key decisions, solutions found, or conclusions reached\n"    # 关键决定
        "4. Any specific commands, files, URLs, or technical details\n" # 技术细节
        "5. Anything left unresolved or notable\n\n"                     # 未解决项
        "Be thorough but concise..."
    )
    # 失败重试 3 次，回退 raw preview
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = await async_call_llm(...)   # 异步 LLM 调用
            ...
        except RuntimeError:
            # 辅助模型不可用 → 降级返回 None（外层会 fallback raw preview）
            return None

&emsp;&emsp;这段代码里有几个关键设计值得逐个讲。**第一是 5 元素结构**——`What asked / What done / Key decisions / Technical details / Unresolved`。这不是 LLM 自由发挥，是写死在 prompt 里的固定 schema，让每个 session 摘要的形态可预期、可拼接。**第二是异步并发**——源码里 `default_concurrency=3, max_concurrency=5`，意思是你 search 命中 10 个 session 时，`Hermes` 不会串行调 10 次 LLM（那要 10×5s=50s），它并发跑 3-5 路、整体延迟接近单次。**第三是失败 3 次重试**——`for attempt in range(max_retries)` 配合 `await asyncio.sleep(1 * (attempt + 1))` 线性退避（重试间隔 1 秒、2 秒、3 秒等差递增；不是 1 秒、2 秒、4 秒的指数退避，源码这里选了更温和的退避策略），应对 LLM provider 偶发限流。**第四是降级机制**——`except RuntimeError: return None`，当辅助模型完全不可用时不报错，回退给上层 fallback 到 `raw preview`（原文截取前 N 字），这是源码核查的一个关键发现——**Hermes 的 LLM 摘要不是硬依赖，可降级**。

**动手验证：用 LangChain + DeepSeek 复现 LLM 摘要层**

&emsp;&emsp;下面这段 demo 不依赖 `Hermes` 主仓库，只复现 `_summarize_session` 的核心逻辑：把命中的历史 transcript + 当前 search query 交给一个辅助 LLM，让它按固定 5 元素 schema 输出摘要。这里用 `langchain-deepseek` 调 `DeepSeek`，因为第三节你已经熟悉 `LangChain` 的消息对象；真实 `Hermes` 生产版不依赖 `LangChain`，这里是教学等价实现。

> 📌 **环境复用说明**：本节假设你已完成 §0.4 环境准备（`.env` 配好、`pip install -r requirements.txt` 装好、`load_dotenv` 跑过、3 行 `[OK]` 看到）。如未跑过请先回到 §0.4 跑那 3 步，本节直接进入业务代码——**不再重复 `pip install` / `load_dotenv` / `LANGCHAIN_TRACING_V2` 三件套**。

**步骤一：构造与 Hermes 同构的 5 元素摘要 prompt**

In [8]:
import os
from textwrap import dedent
from typing import Optional  # 兼容 Python 3.9，避免使用 str | None

from langchain_core.messages import HumanMessage, SystemMessage

# 可选依赖：未安装时让 demo 跳过，而不是导入阶段崩溃。
try:
    from langchain_deepseek import ChatDeepSeek
except ModuleNotFoundError:
    ChatDeepSeek = None


def summarize_session_with_deepseek(conversation_text: str, query: str) -> Optional[str]:
    """教学版 session 摘要函数。

    conversation_text: FTS5 命中的历史会话原文。
    query: 用户本轮 search topic，用来约束摘要焦点。
    """
    if ChatDeepSeek is None:
        print("[SKIP] 缺少 langchain-deepseek，请先回到 §0.4 跑 pip install -r requirements.txt")
        return None

    if not os.getenv("DEEPSEEK_API_KEY"):
        print("[SKIP] 未检测到 DEEPSEEK_API_KEY，请先回到 §0.4 配置 .env 并跑 load_dotenv")
        return None

    llm = ChatDeepSeek(
        model=os.getenv("DEEPSEEK_MODEL", "deepseek-chat"),  # 模型名，默认 deepseek-chat
        temperature=0,                                      # 摘要任务要求稳定输出
        max_retries=2,                                      # 演示版少量重试即可
    )

    # 固定 5 元素 schema，保证每个 session 摘要形态一致。
    system_prompt = dedent("""
        You are reviewing a past conversation transcript to help recall what happened.
        Summarize the conversation with a focus on the search topic. Include:
        1. What the user asked about or wanted to accomplish
        2. What actions were taken and what the outcomes were
        3. Key decisions, solutions found, or conclusions reached
        4. Any specific commands, files, URLs, or technical details
        5. Anything left unresolved or notable

        Be thorough but concise.
    """).strip()

    # query 控制摘要焦点；conversation_text 提供可被压缩的原文证据。
    user_prompt = f"""Search query: {query}

    Past conversation transcript:
    {conversation_text}
    """

    try:
        # LangChain 标准消息格式：系统规则 + 用户输入。
        response = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt),
        ])
        return response.content
    except Exception as exc:
        # 教学版显式展示 Hermes 的降级语义：摘要失败，不让 search 整体失败。
        print(f"[FALLBACK] DeepSeek 摘要调用失败，生产版 Hermes 会回退 raw preview：{type(exc).__name__}: {exc}")
        return None

**步骤二：喂一段历史会话，观察 5 元素摘要形态**

In [9]:
conversation_text = """
User: Kafka 消费者组频繁 rebalance，帮我查一下原因。
Assistant: 先检查 consumer group 状态和 broker 日志。
Tool: search_logs(query="rebalance session timeout max.poll.interval.ms")
Tool result: 多次出现 max.poll.interval.ms exceeded。
Assistant: 建议提高 max.poll.interval.ms，并检查单条消息处理耗时。
User: 我们单条消息会调用外部风控 API，最慢 8 分钟。
Assistant: 结论是处理耗时超过 max.poll.interval.ms，引发 rebalance。建议拆分批处理或延长配置。
"""

summary = summarize_session_with_deepseek(conversation_text, "kafka rebalance")
if summary:
    print(summary)  # 有摘要才打印；fallback 分支会返回 None

根据对话记录，用户的核心问题是 **Kafka 消费者组频繁 rebalance**。以下是总结：

1. **用户目标**：排查并解决 Kafka 消费者组频繁 rebalance 的原因。
2. **采取的行动与结果**：
   - 检查了 consumer group 状态和 broker 日志。
   - 通过日志搜索发现多次出现 `max.poll.interval.ms exceeded` 错误。
3. **关键决策与结论**：
   - 根本原因是单条消息处理耗时（调用外部风控 API，最慢 8 分钟）超过了 `max.poll.interval.ms` 默认值（通常为 5 分钟），导致消费者被认为“死亡”而触发 rebalance。
   - 解决方案：提高 `max.poll.interval.ms` 配置，或拆分批处理以降低单条消息处理时间。
4. **具体技术细节**：
   - 日志查询命令：`search_logs(query="rebalance session timeout max.poll.interval.ms")`
   - 关键配置参数：`max.poll.interval.ms`
   - 用户业务场景：单条消息调用外部风控 API，最慢耗时 8 分钟。
5. **未解决/注意事项**：
   - 用户未明确是否已实施建议（提高配置或拆分批处理），也未讨论其他潜在原因（如 session.timeout.ms 过小、网络抖动等）。建议后续验证调整后的效果。


&emsp;&emsp;真实输出会因模型版本略有差异，但形态应稳定落在这 5 类信息上：

```text
1. 用户想排查 Kafka consumer group 频繁 rebalance 的原因。
2. Agent 检查了 consumer group 状态和 broker 日志，发现 max.poll.interval.ms exceeded。
3. 结论是单条消息处理耗时过长，超过 max.poll.interval.ms，触发 rebalance。
4. 关键细节包括 search_logs 查询、max.poll.interval.ms、外部风控 API 最慢 8 分钟。
5. 未解决项是最终采用拆分批处理还是延长 max.poll.interval.ms，需要结合业务吞吐决定。
```

&emsp;&emsp;这段 demo 要你观察的不是"DeepSeek 写得多漂亮"，而是**摘要层的工程契约**：输入是"命中的历史 transcript + 当前 query"，输出是稳定的 5 元素结构；如果没有配置 `DEEPSEEK_API_KEY` 或辅助模型不可用，生产版 `Hermes` 会走 fallback raw preview，而不是让整个 search 工具失败。

> **【常见误区】**：学员看到 LLM 摘要常会想"摘要肯定比原文好啊"，然后倾向于 query 命中后就只看摘要。这里有两个坑。第一，摘要必然有信息损耗——5 元素 prompt 决定了摘要会丢掉它认为不重要的细节；第二，降级回 raw preview 时返回的是原文截取，不是摘要，**展示形式不同**——你看到一段没有 5 元素结构、像被截断的文字时不要以为是 bug，这是 LLM 不可用时的预设兜底。

&emsp;&emsp;到这里我们已经把 `Hermes` 默认 backend 的**冷记忆三件套**讲完了——`SQLite` 解持久化、`FTS5` 双索引解全文搜索（含 CJK 子串）、`LLM 摘要` 解命中后归纳（含异步并发 + 降级）。但事情没结束——`Hermes` 记忆系统还有另外两块拼图：**热记忆双层**（`MEMORY.md` + `USER.md`，每次对话开始时直接注入系统提示词，跟"按需检索"的冷层形成互补）和**扩展层**（8 种可插拔外部 `Memory Provider`，全部 opt-in）。下两节分别拆开。

### 1.5 热记忆双层：MEMORY.md 与 USER.md

&emsp;&emsp;前三节我们讲的 `SQLite + FTS5 + LLM 摘要` 都属于**冷记忆**——按需检索、不进入每次对话。但 `Hermes` 还有另一组**热记忆**：`MEMORY.md`（Agent 自维护的长期事实记忆）+ `USER.md`（用户画像），它们在每次对话开始时由 `Prompt Builder` **默认注入**系统提示词，Agent 天然就知道这些信息。

In [40]:
# 节选示意：依赖 self（AIAgent.__init__ 内代码），不可独立运行
# run_agent.py 节选（Prompt Builder 注入热记忆的简化逻辑，教学示意）
def _build_system_prompt(self) -> str:
    parts = []
    parts.append(self._load_soul_md())          # SOUL.md   人格定义
    parts.append(self._load_memory_md())        # MEMORY.md 长期事实记忆（热）
    parts.append(self._load_user_md())          # USER.md   用户画像（热）
    parts.append(self._load_skills_meta())      # skills/   技能描述
    parts.append(self._load_project_context())  # .hermes.md / AGENTS.md
    parts.append(self._load_tool_guides())      # 自动生成的工具指引
    return "\n\n".join(parts)

&emsp;&emsp;这段伪代码体现的是 `Prompt Builder` 的"热记忆注入"机制——每次对话开始前，`MEMORY.md` 和 `USER.md` 的内容被整段拼进系统提示词，Agent 不需要主动检索就能"知道"。这跟冷记忆 `SQLite + FTS5` 的差别就在这里：**热记忆是"默认就知道"，冷记忆是"按需才查"**。

**动手验证：拼出一次最终 System Prompt**

&emsp;&emsp;下面这段 demo 完全离线可跑，不调 LLM，只模拟 `Prompt Builder` 的拼接动作。你要看的不是文字内容本身，而是**拼接位置**：`MEMORY.md` 和 `USER.md` 跟 `SOUL.md`、`skills meta`、项目上下文、工具指引处在同一个系统提示词里，所以它们属于"每次默认携带的热层"，不是 search 命中后才出现的冷层。

In [10]:
# 以下字符串模拟 Hermes 会从不同文件/目录读取的 prompt 片段。
SOUL_MD = """
# SOUL.md
你是 Hermes Agent，一个长期运行的个人 AI assistant。
你需要保持一致人格，并优先遵守用户长期偏好。
"""

MEMORY_MD = """
# MEMORY.md
- 用户的后端项目主要使用 Python + pytest。
- 用户经常排查 Kafka、Docker、部署流水线相关问题。
- 用户偏好先定位根因，再给最小可验证修复。
"""

USER_MD = """
# USER.md
- 用户具备 Python 基础，能读懂 SQL 和简单异步代码。
- 用户希望课程解释要能落到源码和可运行 demo。
"""

SKILLS_META = """
# Available Skills
- kafka-debugging: 排查 consumer group rebalance、offset、lag。
- docker-deploy: 排查容器启动失败、环境变量、端口映射。
"""

PROJECT_CONTEXT = """
# Project Context
当前项目是 Harness Engineering 课程，目标是讲清 Agent harness 的工程取舍。
"""

TOOL_GUIDES = """
# Tool Guides
可用工具：session_search、read_file、write_memory。
长历史问题优先调用 session_search，不要把全部历史塞进上下文。
"""


def build_system_prompt() -> str:
    parts = [
        SOUL_MD.strip(),          # 人格与最高层行为约束
        MEMORY_MD.strip(),        # 热记忆：长期事实
        USER_MD.strip(),          # 热记忆：用户画像
        SKILLS_META.strip(),      # skill 摘要，帮助模型选择能力
        PROJECT_CONTEXT.strip(),  # 当前项目上下文
        TOOL_GUIDES.strip(),      # 工具调用规则
    ]
    # 用分隔线保留片段边界，方便观察最终 prompt 结构。
    return "\n\n---\n\n".join(parts)


system_prompt = build_system_prompt()
print(system_prompt)                  # 查看最终注入给模型的系统提示词
print("\n字符数:", len(system_prompt))  # 粗略观察热记忆带来的上下文成本

# SOUL.md
你是 Hermes Agent，一个长期运行的个人 AI assistant。
你需要保持一致人格，并优先遵守用户长期偏好。

---

# MEMORY.md
- 用户的后端项目主要使用 Python + pytest。
- 用户经常排查 Kafka、Docker、部署流水线相关问题。
- 用户偏好先定位根因，再给最小可验证修复。

---

# USER.md
- 用户具备 Python 基础，能读懂 SQL 和简单异步代码。
- 用户希望课程解释要能落到源码和可运行 demo。

---

# Available Skills
- kafka-debugging: 排查 consumer group rebalance、offset、lag。
- docker-deploy: 排查容器启动失败、环境变量、端口映射。

---

# Project Context
当前项目是 Harness Engineering 课程，目标是讲清 Agent harness 的工程取舍。

---

# Tool Guides
可用工具：session_search、read_file、write_memory。
长历史问题优先调用 session_search，不要把全部历史塞进上下文。

字符数: 558


&emsp;&emsp;这就是"热记忆"的真实含义：它不是数据库里等着被检索的一行记录，而是每次对话开始前就进入系统提示词的默认上下文。代价也在输出里直接看得见——`MEMORY.md` / `USER.md` 越长，`字符数` 和 token 成本就越高；如果 `USER.md` 默认还是空文件或 placeholder，它虽然被拼接进 prompt，但不会提供有效用户画像。

&emsp;&emsp;`MEMORY.md` 是 Agent 自己写的——这跟第 2 章的 `Nudge` 机制直接挂钩：每隔 N 个 user-turn，Agent 会在 background review 里决定要不要往 `MEMORY.md` 加一行新事实（"用户偏好用 `pytest` 而不是 `unittest`"、"项目部署到阿里云华东 2 区"等）。`USER.md` 设计上是用户画像——沟通风格、技能水平、工作习惯，但它的填充机制有一个关键前提，下面这一段必须看清楚。

> **【踩坑预警 · 默认状态必读】**：`USER.md` 的**自动**推理填充依赖外部 `Honcho` Provider，但 `Honcho` **默认是关闭的**——也就是说你 `pip install hermes-agent` 后跑起来，`USER.md` 是**空文件或只有 placeholder**，热记忆注入机制存在但内容为空。要让 `USER.md` 自动被写满，必须在 `~/.hermes/config.yaml` 里加 `memory.provider: honcho` 启用——这件事 §1.6 会在 8 种 Provider 表格里专门标注。判断自己是否踩坑的方法：跑过几个 session 后 `cat ~/.hermes/memories/USER.md`，如果只有几行 placeholder 就是 `Honcho` 没启用。当然你也可以**手动**写 `USER.md`——它就是普通 Markdown 文件，纯人工维护完全合法。

&emsp;&emsp;一个关键数字：`MEMORY.md` 和 `USER.md` 是**纯 Markdown 文件**，存在 `~/.hermes/memories/` 目录下。设计取舍是"热记忆要短、冷记忆才长"——再长就该走冷记忆 `session_search` 检索了。两个文件的内容会在每次对话**整段拼进系统提示词**，所以越长 token 成本越高，需要 Agent 自己（或用户手动）控制精炼度。

### 1.6 扩展层：8 种外部 Memory Provider（非默认 ）

&emsp;&emsp;在介绍这一节之前，先做一个诚实声明——你可能在某些 `Hermes` 营销 PDF 或社区博客里看到过"四层记忆"或"`Holographic` 全息记忆"这种说法。**这两个说法都隐瞒了同一个关键事实：所有外部 `Memory Provider` 都是 opt-in 插件，不在默认架构内**。`Hermes v0.7.0+` 总共支持 8 种外部 `Memory Provider`：`Honcho`（辩证用户建模）/ `Mem0`（向量+图+键值三引擎）/ `OpenViking`（向量后端）/ `Hindsight`（语义+图混合）/ `Holographic`（HRR 向量编码+trust_score 奖惩）/ `RetainDB`（持久化记忆）/ `ByteRover`（字节级回放）/ `Supermemory`（语义图+context fencing，v0.8.0 新增）。**8 个全部默认关闭**，需要在 `~/.hermes/config.yaml` 里手动指定 `memory.provider` 字段才会激活——`run_agent.py:1729-1737` 的 `_load_mem(_mem_provider_name)` 会按这个名字加载对应 plugin，没有声明时整段 `_memory_manager` 流程根本不会启动，外部 Provider 完全不参与运行。

> **【踩坑预警 · 默认体验弱于宣传】**：如果你相信"`Hermes` 默认就有 HRR 向量记忆"或"自带 8 种 memory"，下次在生产环境复现某些"记忆神奇召回"的 demo 效果时会**完全跑不出来**——你启动的默认 backend 只有热记忆 `MEMORY.md / USER.md` + 冷记忆 `SQLite + FTS5 + LLM 摘要`，外部 Provider 全部静默。判断方法：`cat ~/.hermes/config.yaml | grep memory.provider`，没有这个字段就是 0 种 Provider 在跑。**8 种 Provider 各自的工程取舍、社区推荐启用顺序、`Honcho` 为何是社区最推荐启用的"`USER.md` 自动填充器"、`Holographic` HRR 向量编码的研究级取舍、`Hermes` 营销 PDF 上 GEPA 等卖点跟这 8 个 Provider 的真实归属——这些会在**第五节课（L5）**§3.2b「两条进化路径：内置 vs 外挂」展开**，那里把 8 Provider 当作"自进化外挂路径"来教，跟 L4 已讲的"内置路径（Nudge + Curator + SQLite/FTS5）"做并排对比。

&emsp;&emsp;**vs `DeepAgents` 对比锚点**：`DeepAgents` 默认 agent 以 `LangGraph state messages` 为对话轨道,通过 `create_deep_agent(checkpointer=..., store=...)` 暴露持久化 contract(`Checkpointer` per-thread + `Store` cross-thread),默认 middleware 链含 `create_summarization_middleware`(防当前 session context 爆 token);**但官方 `Store` 内置实现(`SqliteStore` 等)都是通用 KV,不带 FTS5 keyword 召回 + 召回式归纳**。`Hermes` 默认记忆 = **热层** `MEMORY.md`(自维护事实)+ `USER.md`(用户画像,需启用 `Honcho` 才会自动写满)+ **冷层** `SQLite 持久化 + FTS5 双索引(unicode61 + trigram)+ LLM 摘要层(异步并发 3、最大 5、失败 3 次重试、可降级 raw preview)`。多了五件事:**本地 SQLite + FTS5 keyword/CJK 召回的开箱整合**(官方 SqliteStore 是 KV 不带 FTS5)、**召回后 LLM 二次归纳**(官方 summarization 是 context 压缩不绑检索)、Agent 自维护长期事实(`MEMORY.md`)、用户画像跨会话累积(`USER.md`+`Honcho`)、`Nudge` 自动写入新事实。本章用一张运行时流程图把热层/冷层/Nudge 事后更新串成完整生命周期，再用两个可运行 demo 分别验证"命中后摘要"和"热记忆默认拼进系统提示词"。**8 种外部 `Memory Provider` 是 opt-in 插件、不是默认架构**——这一层在本讲只建立"它存在 + 默认关闭 + 详见下节"的边界，完整选型决策树留给第五节课（L5）。

---

## <center>第2章：Nudge 机制——Skill 的自动生产</center>

&emsp;&emsp;第 1 章我们解决了"记忆怎么存、怎么搜"。这一章我们打开 `Hermes` 最核心的、也最容易被营销话术浪漫化的机制——**Nudge**。这一章是本讲篇幅最大的章节，它要回答一个具体问题：`Hermes README` 上那句 "creates skills from experience" 是怎么实现的？是真的"AI 自主学习"吗？整章按"7 节源码 walk + 第 8 节实战 demo + 第 9-10 节工程验证（可选深挖）"三档展开：§2.1-§2.7 把整条链路拆开（计数器 → 判定 → fork agent → prompt → 落盘）；§2.8 用一个本地可跑的 `MVP demo` 让你亲眼看到 nudge 触发、skill 自动产出，并通过改参数对比触发频率变化；§2.9-§2.10 是工程验证加餐——用 7 个第一性原理测试把 §2.8 写过的契约逐条证伪，**第一遍学习时间紧的话可以跳过 §2.9-§2.10 直接进 §3，主线不会断**。

### 2.1 问题引入：Skill 从哪来？

&emsp;&emsp;第三节课程里你接触到的 `skill` 都是手写的——`SKILL.md` 在仓库里、frontmatter + body 结构、人类工程师写好放进去。这种方式简单可控，但有一个明显的成本：**写 skill 是人力活**。每次发现"agent 这个错误反复出现"或"我每次都得手动告诉 agent 这个项目要这么调试"，工程师就得停下来写一份新 SKILL.md。`Hermes` 的回答是把这件事自动化——它的 skill 库有四种来源，我们对照看一下。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 2-1 Hermes Skill 四种来源</font></p>
<div class="center">

| 来源 | 写入路径 | 谁写的 | 触发时机 |
|---|---|---|---|
| `bundled` | 随 Hermes 版本发布，仓库内 `skills/` | Nous Research 团队 | 安装时拷贝 |
| `hub` | 用户从 `agentskills.io` 下载 | 第三方贡献者 | 用户主动拉取 |
| `user-created` | 用户手动调 `skill_manage create` | 当前会话 user | 用户显式动作 |
| `agent-created` | `~/.hermes/skills/<name>/SKILL.md`（**本章重点**）| Nudge fork agent 自动产出 | 计数器达阈值后台触发 |

</div>

&emsp;&emsp;前三种来源跟你在第三节接触的没本质区别——都是"先有需求、后有 skill"。第四种 `agent-created` 是 `Hermes` 真正独特的部分，它对应 README 上那句 "creates skills from experience"。我们做一个破题动作——本地装好 `Hermes` 跑过几个 session 后，执行 `ls -la ~/.hermes/skills/`，你会看到一些**你从来没写过**的 skill 目录冒出来，目录名通常是某个 task 类别（比如 `kafka-debugging` / `pytest-flaky-test-isolation`）。这就是 nudge 自动产出的产物。

> **【常见误区】**：很多学员第一次看到 `agent-created` skill 会直接想到"AI 真的在自主学习"。这个直觉是错的。我们打开 `run_agent.py:3559-3700` 后会发现：所谓"自动生产"是一条**纯 prompt 驱动的机械闭环**——计数器到阈值 → fork 一个子 agent → 喂给它一段写死的 prompt（"你看了这次会话，请决定要不要 update/create 一个 skill"）→ 子 agent 写 Markdown → 落盘。**没有"模型自主想到要学"这件事**，每一步都是工程代码控制的。本节源码核查的关键发现就是这个——浪漫化的"AI 自主学习"在源码层是个 prompt 触发的回写流程。

### 2.2 两个计数器：memory nudge vs skill nudge

&emsp;&emsp;Nudge 机制的最底层是两个独立计数器——一个管"memory 该回顾了"，一个管"skill 该总结了"。它们在 `run_agent.py:1701-1815` 初始化，在不同位置累加，触发的工作内容也不同。我们直接看源码。

In [ ]:
# 节选示意：依赖 self（AIAgent.__init__ 内代码），不可独立运行
# run_agent.py:1703-1705 节选（计数器初始化）
self._memory_nudge_interval = 10              # memory nudge 默认 10
self._turns_since_memory = 0                  # memory 计数器：按 user-turn 累加
self._iters_since_skill = 0                   # skill 计数器：按 tool-iteration 累加

# run_agent.py:1810-1813 节选（skill nudge interval 配置加载）
self._skill_nudge_interval = 10               # skill nudge 默认 10
try:
    skills_config = _agent_cfg.get("skills", {})
    self._skill_nudge_interval = int(skills_config.get("creation_nudge_interval", 10))
except Exception:
    pass

&emsp;&emsp;这段代码是 `Hermes` 计数器机制的全部初始化逻辑。你需要注意三个细节。

- 第一，**两个计数器的累加维度不同**——`_turns_since_memory` 按"user 发了一句话就 +1"累加（user-turn），`_iters_since_skill` 按"agent 跑了一轮 tool 调用就 +1"累加（tool-iteration）。

- 第二，**默认值都是 10**——没有手动配置时，每 10 个 user turn 触发一次 memory review，每 10 个 tool iteration 触发一次 skill review。

- 第三，**配置项名字精确**——`creation_nudge_interval` 这个 key 是后面"如何禁用 nudge"的钥匙，记住它。

&emsp;&emsp;你现在看到的这两个计数器，就是 `Hermes` 全部自动行为的开关——改这两个数字就能控制 nudge 频率，2.7 节的合规配置和 2.8 节的 MVP demo 里你会直接调它们观察现象。

In [ ]:
# 节选示意：依赖 self（AIAgent 实例方法体），不可独立运行
# run_agent.py:10583-10593 节选（memory 计数器累加位置：每 user turn）
_should_review_memory = False
if (self._memory_nudge_interval > 0
        and "memory" in self.valid_tool_names
        and self._memory_store):
    self._turns_since_memory += 1                          # 每 user turn 累加 1
    if self._turns_since_memory >= self._memory_nudge_interval:
        _should_review_memory = True
        self._turns_since_memory = 0                       # 触发后归零

# run_agent.py:10866-10870 节选（skill 计数器累加位置：每 tool iteration）
if (self._skill_nudge_interval > 0
        and "skill_manage" in self.valid_tool_names):
    self._iters_since_skill += 1                           # 每 tool iter 累加 1
    # 注：这里只累加，触发判定在 turn 末（见 2.3 节）

&emsp;&emsp;两段代码在不同位置——`memory` 累加在 turn 进入函数时（第 10590 行），`skill` 累加在 tool 调用循环内每一次 iteration（第 10870 行）。这就是为什么源码里两段计数器的累加位置完全不同——它们共同强调"两个计数器维度不同"。它有一个直接后果：**同样的 10 阈值，触发频率完全不同**——一个用户对话 turn 可能涉及 5 次 tool 调用（agent 先 search 再 read 再 grep 再调 LLM 评估再写报告），所以 skill 计数器涨得比 memory 计数器快 5 倍。

> **【常见误区】**：学员经常会问"那为什么不统一用 turn 或统一用 iteration？"。答案在于两件事的工作性质不同：**memory 总结的是"用户偏好/会话状态"**，turn 是天然边界（user 发一句、agent 回一句构成一个完整事件）；**skill 总结的是"task 执行经验"**，更细粒度的 tool iteration 更能捕获"这次 task 用了什么 fix"。源码核查时特别澄清了这个常被弄错的假设——"`memory nudge` 等于 `skill nudge`"是错的。

### 2.3 计数判定与触发时机

&emsp;&emsp;计数器累加完了，接下来是判定与触发。`Hermes` 把判定放在每个 turn 的**末尾**而不是 tool 调用中间——这是一个重要的工程选择，原因我们看完源码后再说。

In [ ]:
# 节选示意：依赖 self（AIAgent 实例方法体），不可独立运行
# run_agent.py:13918-13941 节选（turn 末判定 + 触发 background review）
# Check skill trigger NOW — based on how many tool iterations THIS turn used.
_should_review_skills = False
if (self._skill_nudge_interval > 0
        and self._iters_since_skill >= self._skill_nudge_interval     # 阈值检查
        and "skill_manage" in self.valid_tool_names):
    _should_review_skills = True
    self._iters_since_skill = 0                                       # 触发后归零

# 略：external memory provider sync ...

# Background memory/skill review — runs AFTER the response is delivered
# so it never competes with the user's task for model attention.
if final_response and not interrupted and (_should_review_memory or _should_review_skills):
    try:
        self._spawn_background_review(
            messages_snapshot=list(messages),
            review_memory=_should_review_memory,
            review_skills=_should_review_skills,
        )
    except Exception:
        pass  # Background review is best-effort

&emsp;&emsp;这段代码做了三件事。

- **第一是判定**——`_iters_since_skill >= self._skill_nudge_interval` 比较计数器和阈值，如果达标就把 `_should_review_skills` 标记位置 True 并归零。

- **第二是触发条件**——必须 `final_response` 已生成、用户没有 interrupt、且 memory 或 skill 触发其中之一被点亮，才会进入 `_spawn_background_review`。

- **第三是命名讲究**——函数叫 `background review`、且代码注释明确说"runs AFTER the response is delivered so it never competes with the user's task for model attention"——这是工程上的严肃决策：**nudge 跑在用户回应之后，不抢 user-facing 模型的注意力（model attention）**。

> &emsp;&emsp;那为什么判定要放在 turn 末而不是 tool 调用中间？两个原因。第一，turn 末是天然的"会话事件完成点"——nudge 的 prompt 需要看完整个 turn 的所有 tool 调用才能给出有价值的总结，中间触发会拿到不完整的对话。第二，避免抢 attention——如果 tool 调用中间触发 fork agent，那当前 turn 的 LLM 流量会跟 fork agent 的 LLM 流量并发（两个 agent 同时在调 model API），可能引发 rate limit 或上下文混淆。turn 末触发把这两件事天然错开。

&emsp;&emsp;你现在知道判定放在 turn 末而不是 tool 调用中间——这件事如果将来你自己实现类似机制，放错位置会直接导致 prompt 拿到不完整的对话或者抢 user-facing 模型的 attention，是结构性 bug 不是配置错误。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260508155354280.png" width=50%></div>

### 2.4 Fork Agent 的 4 重约束设计

&emsp;&emsp;判定通过后，`_spawn_background_review` 会创建一个子 agent。这个子 agent 不是普通的 agent，它有4 重约束设计，每一条都是为了防止"自动生产 skill"这件事变成失控的资源黑洞。我们直接读源码。

In [ ]:
# 节选示意：依赖 AIAgent 类（来自 hermes_agent 包），不可独立运行
# run_agent.py:3559-3640 节选（fork 子 agent 的 4 重约束）
review_agent = AIAgent(
    model=self.model,
    max_iterations=16,                              # ① 步数上限：最多 16 步
    quiet_mode=True,                                # ② 安静模式：不输出到用户界面
    platform=self.platform,
    provider=self.provider,
    api_mode=_parent_runtime.get("api_mode") or None,
    base_url=_parent_runtime.get("base_url") or None,
    api_key=_parent_runtime.get("api_key") or None,
    credential_pool=getattr(self, "_credential_pool", None),
    parent_session_id=self.session_id,
    enabled_toolsets=["memory", "skills"],          # ③ 工具集隔离：只给 memory + skills
)
review_agent._memory_write_origin = "background_review"
review_agent._memory_write_context = "background_review"
review_agent._memory_store = self._memory_store
review_agent._memory_enabled = self._memory_enabled
review_agent._user_profile_enabled = self._user_profile_enabled
review_agent._memory_nudge_interval = 0             # ④ 防递归：子 agent 自己不触发 nudge
review_agent._skill_nudge_interval = 0              # ④ 防递归：同上，skill 计数器也禁用
review_agent.suppress_status_output = True

&emsp;&emsp;这段代码里的 4 重约束你必须逐条理解。

- **第一重 `max_iterations=16`**——子 agent 最多跑 16 步，到了直接停。这意味着哪怕子 agent 模型 hallucination 决定要 read 100 个文件，它也读不超过 16 步。

- **第二重 `quiet_mode=True`**——子 agent 的所有 stdout/状态输出不会冒到用户终端，避免污染用户体验。

- **第三重 `enabled_toolsets=["memory", "skills"]`**——子 agent 只能调用 memory 和 skills 这两个工具集，**不能 grep、不能 bash、不能调用任意 tool**——这是关键的工具集隔离，让 fork agent 干不了它本职之外的事。**这里"工具集隔离"的力度需要具象化**：`Hermes` 全局有 47 个工具分属 20 个 `Toolset`（`terminal / file / web / browser / vision / image_gen / tts / code_execution / skills / memory / session_search / cronjob / send_message / delegation / todo / clarify / moa / homeassistant / rl / mcp_*` 等），fork 只放行 2 个等于把工具面切到 1/10。父 agent 可以让 fork 子 agent 写 skill，但子 agent 反过来想 `bash rm -rf` 是物理上不可能——这就是"约束 = 生产力"的具体体现。

- **第四重 `_memory_nudge_interval = 0` + `_skill_nudge_interval = 0`**——这是防递归的核心。如果不写这两行，子 agent 跑 nudge 时它自己又会触发 nudge 计数器，进而 fork 子子 agent，一路下去就是"无限 fork 黑洞"。

In [ ]:
# 节选示意：依赖 review_agent（Block 9 创建的实例），不可独立运行
# run_agent.py:3642-3645 节选（启动子 agent）
review_agent.run_conversation(
    user_message=prompt,                            # 关键：喂给子 agent 的 prompt
    conversation_history=messages_snapshot,         # 历史快照，不污染父 agent 状态
    #...
)

&emsp;&emsp;这一段是 fork 启动语句。注意 `conversation_history=messages_snapshot`——传进去的是父 agent 历史的**快照**（在判定时用 `list(messages)` 浅拷贝过），子 agent 在它基础上做 nudge 反思，但反思过程产生的新 messages 不会污染父 agent 的对话历史。这是状态隔离的体现。

> **【踩坑预警 · 极端重要】** 第 4 重约束 `_memory_nudge_interval = 0` + `_skill_nudge_interval = 0` 是**防递归的唯一防线**。如果你在自己的项目里实现类似 nudge 机制，**必须**在子 agent 实例化后立刻把这两个计数器置零。后果是什么？子 agent 跑 16 步过程中每个 tool iteration 都会让 `_iters_since_skill` 累加，到达阈值再 fork——你的进程内 agent 数量会指数爆炸，几分钟内 LLM API 配额烧光、OOM、整个机器卡死。判断自己是否踩坑的方法：在子 agent 创建后立刻 `print(review_agent._memory_nudge_interval, review_agent._skill_nudge_interval)`，必须都是 0。

&emsp;&emsp;到这里你已经看清 Hermes 真实的 4 重约束是怎么写的——`AIAgent(...)` 风格、自研 `AIAgent` 类的构造调用风格（不是 `deepagents` 框架）。但这种写法对学员有一个问题：**它把"防御机制"和"框架细节"混在一起讲**，你既要懂 nudge 想防什么，又要懂 Hermes 自研 `AIAgent` 类的构造细节。本节后面 2.8 的 MVP demo 用 `langchain` 把这两件事拆开讲——nudge 防御逻辑用 `AgentMiddleware` 的钩子表达，工具集隔离用 `tools=[...]` 参数表达，防递归用 `middleware=[]` 参数表达。下面这张映射表帮你提前锁定两种实现的对应关系，方便等会儿读 demo 代码时一眼找到位置。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 2-2 Hermes 4 重约束的两种实现对照（真实源码 vs 我们 demo）</font></p>
<div class="center">

| 约束编号 | Hermes 真实源码（自研 `AIAgent` 风格） | 我们 demo（langchain 风格） | 对应到本节哪一段 |
|---|---|---|---|
| ① 步数上限 | `max_iterations=16`（AIAgent 构造参数）| `.with_config({"recursion_limit": 8})` 显式收紧到 8（教学版收紧到比 Hermes 16 更激进的档位，让阈值边界一目了然）| 2.4 节本段 / 2.8 节 Part 3 |
| ② 安静模式 | `quiet_mode=True`（不污染用户终端）| `verbose=False` 全程静默（教学默认 `verbose=True` 让你看到 sub-agent 决策）| 2.8 节 Part 3 `_spawn_review` |
| ③ 工具集隔离 | `enabled_toolsets=["memory", "skills"]`（AIAgent 参数限制）| `create_agent(tools=[write_skill, read_memory])` 双半边——`write_skill` = skills 半边、`read_memory` = memory 半边 | 2.8 节 Part 3 `_spawn_review` |
| ④ 防递归 | `_memory/_skill_nudge_interval = 0`（手动置零计数器）| `create_agent(middleware=[])`（不挂 NudgeMiddleware，根本不存在计数器）| 2.8 节 Part 3 `_spawn_review` |
| 五步校验 | `_create_skill` 的 name/category/frontmatter/size/collision | demo 已对齐前 3 步:① name 由 sub-agent 走 prompt 引导(class-level slug)② **frontmatter 验证 + 缺则补全**(YAML name + description ≤1024 chars,渐进式披露 L1)③ description size 上限校验;省略 category 校验(单目录)+ collision 检测(单实例 demo) | 2.6 节 + 2.8 节 Part 2 |

</div>

&emsp;&emsp;读这张表的姿势是——左两列对照看，你会发现 **Hermes 用"自研类的构造参数"表达约束（你看 `AIAgent(max_iterations=16, quiet_mode=True, enabled_toolsets=["memory","skills"], ...)` 是 `run_agent.py` 自己定义的类的构造调用），langchain 用"组合 + 显式声明"表达约束（你看 `create_agent(tools=[...], middleware=[]).with_config({"recursion_limit": 8})`）**。**4 重约束 ①②③④ 在 langchain 里全部显式表达**——`recursion_limit` 显式收紧、`verbose` 显式控制、`tools=[...]` 显式白名单、`middleware=[]` 显式不挂——没有任何一项依赖框架默认值兜底。两种风格各有适用场景：自研一个完整的 agent 主循环（Hermes 14201 行的 `run_agent.py` 涵盖 tool dispatch / credential pool / memory store 全套）适合"我要把 harness 当产品长期维护"的生产级项目；langchain `AgentMiddleware` 适合"我只想给现有 agent 加一组 hook + 一个受限子 agent"的轻量场景。Hermes 选了前者（自己写所有），我们 demo 选后者（用 langchain 标准化）——讲清原理用后者更直观，扩到生产再切前者也没问题。

&emsp;&emsp;**【可选阅读 · §2.4 延伸】`AIAgent` 类的 5 大内部组件**——这是一段补充信息，**不影响主线 Nudge 机制理解**，跳过不会让你听不懂后面的内容；感兴趣的同学可以课后回查。`AIAgent` 是 `run_agent.py:874` 起始的核心类（约 9,200 行），承担整个 `Hermes` 系统的"心脏"角色，内部有 5 大组件：① **`Prompt Builder`**——从 6 个来源动态拼装系统提示词（`SOUL.md` 人格 + `MEMORY.md` + `USER.md` + `skills/` + `.hermes.md`/`AGENTS.md` + 工具指引），每次对话前注入；② **`Provider Resolution`**——4 层凭证发现链（显式 > 配置 > `OAuth Token` > 环境变量），支持 18+ 模型提供商；③ **`Tool Dispatch`**——47 工具 × 20 `Toolset` 的统一调度中心，工具通过注册表模式自注册；④ **`Context Compressor`**——上下文超阈值时自动摘要化中间轮次，含血统追踪；⑤ **三种 `API` 模式**——`chat_completions` / `codex_responses` / `anthropic`，根据所选模型自动切换。理解了这 5 个组件你就明白：`Hermes` 的"自研"不是"重写一遍 LangChain"，而是**这 5 件事都按自己的工程取舍写死**——这是 `langchain create_agent` 通用框架做不到的。**主线回归**：以上 5 大组件跟本节 4 重约束教学目标无直接关联，下面 §2.5 我们继续讲 `prompt` 怎么驱动 fork agent 反思。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260508155354315.png" width=50%></div>

### 2.5 Prompt 驱动的反思：不是自主意识

&emsp;&emsp;子 agent fork 完了，它怎么"决定"要写哪个 skill？答案是 prompt——`run_agent.py:3351-3438` 里写死的 `_SKILL_REVIEW_PROMPT`。我们看核心段落。

In [ ]:
# 节选示意：字符串字面量节选，包含省略号（...），不可独立运行
# run_agent.py:3364-3438 节选（_SKILL_REVIEW_PROMPT 核心段，教学示意）
_SKILL_REVIEW_PROMPT = (
    # ... 前面省略：触发信号识别 ...
    "Preference order — prefer the earliest action that fits, but do "
    "pick one when a signal above fired:\n"
    "  1. UPDATE A CURRENTLY-LOADED SKILL. ..."         # 优先 PATCH 当前 session 用过的 skill
    "  2. UPDATE AN EXISTING UMBRELLA ..."              # 其次 PATCH 类别 umbrella skill
    "  3. ADD A SUPPORT FILE under an existing umbrella. ..."  # 再次添加支持文件
    "  4. CREATE A NEW CLASS-LEVEL UMBRELLA SKILL ..."  # 最后才创建新 skill
    "\n\n"
    # ... 关键的 ACTIVE 倾向 ...
    "'Nothing to save.' is a real option but should NOT be the default. "
    "If the session ran smoothly with no corrections and produced no "
    "new technique, just say 'Nothing to save.' and stop. Otherwise, act."
)

# run_agent.py:3446-3449 节选（_COMBINED_REVIEW_PROMPT 进一步强化 ACTIVE）
_COMBINED_REVIEW_PROMPT = (
    "...\n\n"
    "**Skills**: how to do this class of task. Be ACTIVE — most "
    "sessions produce at least one skill update. A pass that does "
    "nothing is a missed learning opportunity, not a neutral outcome.\n\n"
    ...
)

&emsp;&emsp;读这段 prompt 你应该看出三件事。**第一是优先级**——`PATCH > 创建`。prompt 明确鼓励"先看现有 skill 能不能 patch，实在不行才创建新的"，这是为了防止 skill 库无限膨胀（如果每个 nudge 都创建新 skill，几天就堆出一座垃圾山）。**第二是 ACTIVE 倾向**——`'Nothing to save.' is a real option but should NOT be the default` 和 `Be ACTIVE — most sessions produce at least one skill update`。这两句话是"为什么 fork agent 倾向产出而非沉默"的核心证据：**prompt 主动鼓励 fork agent 倾向"产出"而非"沉默"**——这会显著增加虚假 skill 的产生概率。**第三是结构化输出指令**——`name MUST be at the class level`，禁止用 PR 编号 / 错误字符串 / "fix-X" 这类 session 临时性词作 skill 名，强制让产出的 skill 具备可复用性。

> &emsp;**关于 sub-agent 可用工具集**：上面节选只展示了"决策骨架"——真实 prompt 还会在前缀注入 `enabled_toolsets=["memory","skills"]` 的工具引导块，告诉 sub-agent "你可以先调 `read_memory` 拿最近对话/事实素材，再决定是 `PATCH` 现有 skill 还是 `CREATE` 新 skill"。这跟 §2.8 demo sub-agent `tools=[write_skill, read_memory]` 一一对应：`read_memory` 是反思素材的"读侧"、`write_skill` 是落盘的"写侧"——4 重约束 ③ 工具集隔离的 `memory + skills` 双半边在 prompt 和 tools 两侧都得对得上，否则要么有工具但 prompt 不引导（工具白挂）、要么 prompt 引导但工具没给（sub-agent 抓瞎）。

> **【风险预警】**：`Be ACTIVE` + `should NOT be default 'Nothing to save'` 这两条 prompt 是把双刃剑——好处是"sessions 不会浪费学习机会"，代价是 fork agent 倾向编造一些其实没什么价值的 skill 来"完成任务"。这种倾向属于"中度风险"——需要 curator 在后台做清理——这正好埋下了 下一节的钩子（curator 定期合并去重）。当前本节的层面，你只需要知道这个倾向**存在**、它对 skill 库长期质量有影响。

&emsp;&emsp;到这里我们已经看清"自动生产 skill"的机制本质——它是一条**完全由 prompt 驱动的机械闭环**：计数器达阈值 → fork agent → 喂 prompt → LLM 输出 Markdown → 写文件。中间没有任何"模型自主萌生学习意愿"的环节，每一步都是 Python 代码确定性控制的。这跟 README 上 "creates skills from experience" 的浪漫表述差距很大，但这正是诚实的工程真相——**`Hermes` 的"自学"是 prompt 工程的胜利，不是 AGI 的胜利**。这一点会在 下一节进一步展开。

### 2.6 Skill 落盘与产物形态

&emsp;&emsp;子 agent 在 prompt 驱动下输出 Markdown 后，最后一步是落盘。落盘入口是 `tools/skill_manager_tool.py:371` 的 `_create_skill` 函数。

In [ ]:
# 节选示意：依赖 typing.Dict 等外部 import（实际文件已 import），不可独立运行
# tools/skill_manager_tool.py:367-395 节选（_create_skill 落盘入口）
def _create_skill(name: str, content: str, category: str = None) -> Dict[str, Any]:
    """创建一个新的 user skill，写入 SKILL.md 内容。"""
    # 1. 校验 name（class-level naming，禁用 PR 编号/会话临时词）
    err = _validate_name(name)
    if err:
        return {"success": False, "error": err}

    # 2. 校验 category
    err = _validate_category(category)
    if err:
        return {"success": False, "error": err}

    # 3. 校验 frontmatter（YAML 结构合法性）
    err = _validate_frontmatter(content)
    if err:
        return {"success": False, "error": err}

    # 4. 校验内容大小（防超大 SKILL.md 撑爆 prompt）
    err = _validate_content_size(content)
    if err:
        return {"success": False, "error": err}

    # 5. 检查命名碰撞（跨所有 skill 目录全局唯一）
    existing = _find_skill(name)
    if existing:
        return {...}  # 返回冲突错误，引导改用 PATCH
    # ... 后续：创建目录、写 SKILL.md 文件

&emsp;&emsp;这段代码展示的是 `_create_skill` 的五步校验流水线——name 命名规范、category 校验、**frontmatter YAML 合法性(渐进式披露 L1:必含 name + description,description ≤1024 chars)**、内容大小、命名碰撞。每一步失败都返回结构化错误，`fork agent` 拿到错误后会重试或改走 PATCH 路径。**第 3 步 frontmatter 校验是 Anthropic Claude Skills 渐进式披露原则的硬约束**——agent 在 skill 列表里只看 name + description 两行决定要不要加载 body,**没有 frontmatter 的 SKILL.md 在 skill 列表里就是隐形的**,这就是为什么 Hermes 必须在落盘前强制校验。第 5 步**命名碰撞** —— `_find_skill` 会扫描所有 skill 目录(bundled / hub / user-created / agent-created),如果发现重名直接拒绝创建并返回 conflict 错误。这是 prompt 优先级排序里"PATCH > CREATE"的强制兜底。

In [ ]:
# ~/.hermes/skills/<skill-name>/ 目录结构（教学示意）
# ~/.hermes/skills/kafka-rebalance-debugging/
# ├── SKILL.md            # 主 skill 文件（YAML frontmatter + body）
# ├── references/         # session 特定细节、错误抄录、API 摘录
# │   └── 2026-05-error-stack.md
# ├── templates/          # 拷贝即用的模板文件
# │   └── consumer_config.yaml
# ├── scripts/            # 静态可重跑的脚本（验证、fixture 生成）
# │   └── verify_offset.py
# └── assets/             # 其他静态资产
#     └── architecture.png

&emsp;&emsp;这个目录结构体现了 `Hermes` skill 的设计哲学——**SKILL.md 是入口，配套 4 类目录是"按需启用"的支持文件**。`references/` 用来存调试时积累的具体错误抄录或外部文档摘录；`templates/` 存"拷贝改一下就能用"的模板；`scripts/` 存"agent 可以直接运行"的验证脚本（比 LLM 反复手敲命令更可靠）；`assets/` 存图片等静态资源。要注意的是这 4 类目录是**可选支持目录，不是每个 skill 都必须创建**——简单 skill 可能只有 `SKILL.md` 一个文件，复杂 skill 才会按需补 `references/` 或 `scripts/`。这种结构意味着 `Hermes` 的 skill 上限比 `DeepAgents` 接触到的"单文件 SKILL.md" 容量大得多。

&emsp;&emsp;当你打开自己机器上的 `~/.hermes/skills/` 目录时，应该能看到这种结构在部分 skill 下出现（具体哪些目录冒出来取决于 fork agent 当时的判断）——这就是接下来 2.8 节 MVP demo 里你会亲眼验证的现象。

> &emsp;&emsp;这里要补充一点 5 月新增的细节——`tools/skill_provenance.py`（2026-05 新增）引入了一个 `ContextVar` 用来区分"当前正在写入 skill 的是 fork agent 还是 foreground agent"，这样 curator 在做后台合并/清理时**只动 agent-created 的 skill，不动用户手写的**。这个机制叫做"血统隔离"，是为下一节 curator 章节做的工程铺垫——本课不深讲，知道有这件事即可。

### 2.7 无 Explicit Consent 的设计取舍

&emsp;&emsp;现在我们把 nudge 机制的全链路看完了，是时候问一个关键问题：**用户在这整个流程里做了什么？** 答案是——**什么都没做**。`Hermes` 的 fork agent 直接写 `~/.hermes/skills/`，没有任何 UI 弹窗确认、没有 `Y/n` 提示、用户事后才会看到一行通知 `[Self-improvement review]: ...`。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260508143139652.png" width=70%></div>

In [ ]:
# 用户唯一感知的事件：turn 结束后偶尔出现的状态行
# [Self-improvement review]: PATCHED skill 'kafka-debugging' (added section: rebalance edge case)
# [Self-improvement review]: CREATED skill 'pytest-flaky-test-isolation'
# [Self-improvement review]: Nothing to save.

&emsp;&emsp;这就是 nudge 机制的"用户感知层"——一行事后通知。这种设计可以概括为"用户在改进流程外"（user out of the loop for improvements）——用户对自动化产出的过程没有任何介入机会。它是一个明确的工程取舍——**以 0% 用户主动介入换取自动化产出机会**：每次触发后 fork agent 自主决定是否产出（PATCH/CREATE/Nothing to save.），无需用户操作即可累积 skill 库；代价是产出质量不可保证（fork agent 也可能产出低质 skill，参见 2.5 节风险预警）。

> **【踩坑预警 · 高敏感场景必读】** 如果你在合规要求严格的环境（医疗 / 金融 / 法律 / 政府）部署 `Hermes`，这种"无 explicit consent 直接写文件"的设计可能违反"用户对 AI 行为有显式知情权"的合规要求。判断的方法很简单——`Hermes` 在你的 session 跑完后是不是直接出现了你没写过的 SKILL.md 文件？是。这意味着部署到合规环境前必须**禁用 nudge**。

&emsp;&emsp;**禁用方式**——这是你课后必须记住的具体配置。在 **`~/.hermes/config.yaml`**（Hermes 用户级配置文件，路径由 `hermes_cli/config.py:248` 的 `get_config_path() = get_hermes_home() / "config.yaml"` 定义）里添加以下两行。

```yaml
# ~/.hermes/config.yaml
skills:
  creation_nudge_interval: 0    # 0 = 完全禁用 skill nudge

memory:
  nudge_interval: 0             # 0 = 完全禁用 memory nudge
```

&emsp;&emsp;`creation_nudge_interval: 0` 就是 `run_agent.py:1813` 那行 `int(skills_config.get("creation_nudge_interval", 10))` 加载的配置——把它设成 0，`run_agent.py:13920` 的判定条件 `self._skill_nudge_interval > 0` 就直接 False，nudge 永远不会触发。`memory.nudge_interval: 0` 同理禁用 memory review。这两行配合在一起，等价于把整个 nudge 机制开关关掉。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260508155354038.png" width=50%></div>

&emsp;&emsp;这一节的诚实声明也为后面的**四维评价**埋下了伏笔——`Hermes` 在 `Architectural Constraints` 维度的"信任"端点是显著的：它信任 fork agent 不会乱写、信任 prompt 优先级会自动避免重复创建、信任 curator 后台能合并掉低质 skill。这是一组联动信任，而不是孤立的"无 consent 写文件"——只是这组联动信任在生产环境中表现如何，需要长期观察才能定论。

&emsp;&emsp;**vs DeepAgents 对比锚点**：`DeepAgents` skill 来源 = 手动写 Markdown 或工程师代码里调 LLM 写；`Hermes` skill 来源 = `nudge 计数器`（memory: user-turn / skill: tool-iteration） → `turn 末判定` → `fork 子 agent`（4 重约束：步数 16 / 安静模式 / 工具集 memory+skills / 防递归 interval=0） → `_SKILL_REVIEW_PROMPT 驱动反思`（PATCH > CREATE，ACTIVE 倾向） → `_create_skill 五步校验` → `~/.hermes/skills/<name>/SKILL.md` 落盘。多了自动生产链路，代价是无 explicit consent + 虚假产出风险，需要下一节 curator 兜底。

&emsp;&emsp;到这里你已经能完整复述 Nudge 链路的每一步——从计数器累加到落盘的完整闭环都在脑子里——并且知道在 `~/.hermes/config.yaml` 里加哪两行配置就能在合规场景下完全禁用它。

> 📌 **【认知停顿点 · 进入实战前的休息位】**：前 7 节都是**读源码 + 看机制**——你已经看过 4 重约束怎么写、`prompt` 怎么驱动反思、`skill` 怎么落盘、用户为什么 0% 介入。**接下来 §2.8 是这 7 节内容的"实地验证"**——所有抽象机制都会在 langchain 上跑一遍，让你不只是"读懂"还"看到"。

&emsp;&emsp;接下来 §2.8 是 90 分钟课件中**唯一的动手实战环节**——前 7 节读源码积累的所有概念都会在这一节落地成可运行代码。最后一章我们再用这把**四维评价尺**来评价这些机制背后的取舍。

### 2.8 MVP Demo：langchain 中间件 + tool 模拟 Hermes 核心机制（本地可跑）

&emsp;&emsp;前面 7 节都是源码 walk，这一节是本讲唯一的实战代码。我们要做的事是**回到第二节《mini Harness》第九章你亲手搓过的 hook 链思路，用 `langchain 1.0` 的标准 `AgentMiddleware` 把它工业化实现**——把 Hermes 的 nudge 机制（计数器 + turn 末判定 + fork sub-agent + 落盘）拆成你熟悉的三个 langchain 原语：**`AgentMiddleware`**（hook 钩子，对应 Hermes 的 `_iters_since_skill` 累加和 `turn_end` 判定）+ **`@tool`**（落盘工具，对应 `skill_manager_tool.py`）+ **`create_agent`**（受限 sub-agent，配 `tools=[write_skill, read_memory]` 双半边即工具集隔离 + `middleware=[]` 防递归 + `.with_config({"recursion_limit": 8})` 显式收紧步数上限）。这种实现的好处是每一行 langchain 代码都 1:1 对应 Hermes 一段源码——你能直接对着源码 file:line 读懂 demo 在做什么。

&emsp;&emsp;为什么不用第三节的 `DeepAgents` 框架？答案是 `DeepAgents` 是 batteries-included 全套（virtual file system / planning / state schema / 子代理拓扑），它的 90% 功能跟 nudge 机制无关，反而会让"计数器累加 → 触发 → fork → 落盘"这条主线被淹没。`langchain` 的 `AgentMiddleware` 是更轻的 hook 链原语，跟 Hermes 真实的"hook 拦截 → 计数 → spawn"架构同构度更高——这是教学上更合适的脚手架。**注意**：真实 Hermes 既不用 `DeepAgents` 也不用 `langchain`——它是一个完全自研的 `AIAgent` 类（`run_agent.py` 14201 行自己实现了 agent 主循环、tool dispatch、credential pool、memory store 全套），依赖只是 `openai` / `anthropic` 等裸 SDK。Block 9-10 的源码引用展示的就是这个自研 `AIAgent` 类的构造调用——我们 demo 用 langchain 是为了把核心机制讲透，2.4 节的对照表（表 2-2）已经给出两种实现的对应关系（往上回看一下）。

&emsp;&emsp;**这个 demo 不依赖 `Hermes` 完整源码**（`Hermes` 仓库本身的部署门槛较高、依赖外部 SQLite 文件路径等），它只在本地实现 nudge 机制的核心逻辑，让你亲眼看到：跑几轮 user turn 之后 `skills/` 目录里冒出 `agent-created` 的 SKILL.md 文件；改 `nudge_interval=5`（比 demo 默认值 3 触发更慢，但比 Hermes 生产默认 10 仍更频繁）观察触发频率变化；改 `nudge_interval=0` 完全禁用。

> 📌 **环境复用说明**：本节假设你已完成 §0.4 环境准备（`.env` 配好、`pip install -r requirements.txt` 装好、`load_dotenv` 跑过、3 行 `[OK]` 看到）。**本节直接进入业务代码**，不再重复 `pip install` / `load_dotenv` 三件套。下面唯一一行业务变量是 `MODEL` —— LangChain `provider:model` 字符串标识符，后续 `create_agent(model=MODEL)` 用它。

In [11]:
import os

# LangChain provider:model 格式标识符（继承 §0.4 已 load 的 DEEPSEEK_MODEL）
MODEL = f"deepseek:{os.getenv('DEEPSEEK_MODEL', 'deepseek-chat')}"
print(f"[OK] MODEL = {MODEL}")

[OK] MODEL = deepseek:deepseek-chat


&emsp;&emsp;**步骤一：实现两个业务工具（agent 在调试 kafka 时用的"手"）**

&emsp;&emsp;主 agent 干活时需要工具——它要 search 日志、读配置文件。我们用 `langchain` 的 `@tool` 装饰器把两个简单函数包成 langchain 标准工具，对应 Hermes 真实场景里 agent 调的 `grep` / `read_file`。

In [12]:
# hermes_mvp_demo.py — Part 1：业务工具
import os
from pathlib import Path
from langchain_core.tools import tool

# 落盘根目录（演示用，对应 ~/.hermes/skills/）
SKILLS_DIR = Path("./hermes_mvp_skills")
SKILLS_DIR.mkdir(parents=True, exist_ok=True)


@tool
def search_kafka_logs(keyword: str) -> str:
    """搜索 kafka 相关日志（教学占位，真实场景接 Elasticsearch / Loki / journalctl）。"""
    fake_db = {
        "rebalance": ("2026-05-06 14:23:01 [WARN] consumer-group=order-svc rebalance "
                      "triggered, session.timeout.ms=10000ms, members=3"),
        "session.timeout": ("application.yaml: kafka.consumer.session.timeout.ms=10000  "
                            "(默认 10s, rebalance 触发频繁)"),
        "max.poll": ("application.yaml: kafka.consumer.max.poll.interval.ms=300000  "
                     "(5 分钟, 业内合理范围)"),
        "heartbeat": ("application.yaml: kafka.consumer.heartbeat.interval.ms=3000  "
                      "(3s, 默认值, 是 session.timeout.ms 的 1/3, 合理)"),
    }
    for k, v in fake_db.items():
        if k in keyword.lower():
            return v
    return f"[search_kafka_logs] 未找到与 '{keyword}' 匹配的日志记录"


@tool
def read_yaml_config(path: str) -> str:
    """读取 application.yaml 配置（教学占位，真实场景接 fs.read_text）。"""
    if "application.yaml" in path or "application.yml" in path:
        return ("kafka:\n  consumer:\n    session.timeout.ms: 10000\n"
                "    max.poll.interval.ms: 300000\n"
                "    heartbeat.interval.ms: 3000\n"
                "    group.id: order-svc")
    return f"[read_yaml_config] 路径未知：{path}"

&emsp;&emsp;这两个工具刻意写得短——它们是 demo 里 agent 干"业务活"用的两只手，不是教学重点。重点在下一步定义的 `write_skill` 和 `NudgeMiddleware`。

&emsp;&emsp;**步骤二：实现落盘工具 + Hermes prompt（agent 自动学习的"动作 + 念头"）**

&emsp;&emsp;`write_skill` 这个工具对应 Hermes 真实源码里的 `tools/skill_manager_tool.py` —— 它是 fork sub-agent 决定"要保存这次经验"时调用的落盘动作。我们把它单独定义成 langchain 的 `@tool`，因为下一步的 sub-agent 需要被显式限制"只能用这一类工具"——具体是 `write_skill`（落盘的"写侧"，skills 半边）+ `read_memory`（反思素材的"读侧"，memory 半边）两件，对应 Hermes `enabled_toolsets=["memory","skills"]` 双半边——这就是 4 重约束 ③ 工具集隔离的 langchain 实现方式。本节先定义 `write_skill`，`read_memory` 在下一段 prompt 准备完后一并给出。

In [13]:
# hermes_mvp_demo.py — Part 2：落盘工具 + Skill Review Prompt

import re
import yaml

MAX_DESCRIPTION_LENGTH = 1024  # 对齐 Hermes skill_manager_tool.py 同名常量

@tool
def write_skill(name: str, action: str, content: str, description: str = "") -> str:
    """写入或 patch 一个 SKILL.md 文件(对应 Hermes tools/skill_manager_tool.py 的 _create_skill)。

    **渐进式披露(Progressive Disclosure)原则**: SKILL.md 必须以 YAML frontmatter 开头
    (含 name + description),agent 在 skill 列表里**只看这两行**决定要不要进 body。
    渐进式披露 3 层: ① name+description (always 加载) → ② SKILL.md body (用时加载) →
    ③ references/scripts (子任务时加载)。本函数对齐 Hermes _validate_frontmatter 的核心校验。

    Args:
        name: skill 名称(必须 class-level slug,禁止 issue 编号 / 错误字符串)
        action: 'create' 表示新建;'patch' 表示更新现有 skill
        content: SKILL.md 的完整 markdown 内容(若已含 frontmatter 则验证,否则自动补全)
        description: skill 用途简述(≤1024 字符,说明 agent 何时该用这个 skill;
                     content 已含 frontmatter 时此参数可省略)
    """
    has_frontmatter = content.startswith("---")  # 严格对齐 Hermes _validate_frontmatter

    if has_frontmatter:
        # 路径 A: content 已含 frontmatter → 1:1 复刻 Hermes _validate_frontmatter
        end_match = re.search(r'\n---\s*\n', content[3:])
        if not end_match:
            return f"[ERROR] frontmatter 未闭合(缺结尾 ---),请检查 content 格式"
        try:
            parsed = yaml.safe_load(content[3:end_match.start() + 3])
        except yaml.YAMLError as e:
            return f"[ERROR] YAML frontmatter 解析失败: {e}"
        if not isinstance(parsed, dict):
            return "[ERROR] frontmatter 必须是 YAML 字典(key: value 对)"
        if "name" not in parsed:
            return "[ERROR] frontmatter 缺 'name' 字段"
        if "description" not in parsed:
            return "[ERROR] frontmatter 缺 'description' 字段"
        if len(str(parsed["description"])) > MAX_DESCRIPTION_LENGTH:
            return f"[ERROR] description 超 {MAX_DESCRIPTION_LENGTH} 字符上限"
        body = content[end_match.end() + 3:].strip()
        if not body:
            return "[ERROR] frontmatter 后必须有 body(指令/步骤等)"
        final_content = content
    else:
        # 路径 B: content 没有 frontmatter → 用 name + description 自动补全
        # (教学场景常见;生产里 sub-agent 应自己输出含 frontmatter 的 content)
        body_check = content.strip()
        if not body_check:
            return "[ERROR] content 为空,无法生成 SKILL.md(必须有 body)"
        if not description:
            description = f"Procedure for {name.replace('-', ' ')}"
        if len(description) > MAX_DESCRIPTION_LENGTH:
            return f"[ERROR] description 超 {MAX_DESCRIPTION_LENGTH} 字符上限"
        frontmatter = f"---\nname: {name}\ndescription: {description}\n---\n\n"
        final_content = frontmatter + content.lstrip()
        action = f"{action} (auto-frontmatter)"  # 标记被自动补的现象,便于学员观察

    skill_dir = SKILLS_DIR / name
    skill_dir.mkdir(parents=True, exist_ok=True)
    skill_md = skill_dir / "SKILL.md"
    skill_md.write_text(final_content, encoding="utf-8")
    return f"[落盘] {action.upper()} skill: {skill_md}"


# 对应 Hermes 真实源码 _SKILL_REVIEW_PROMPT(run_agent.py:3364-3438)的精简移植版
_SKILL_REVIEW_PROMPT = """你正在审视一个最近的 agent 会话片段,判断里面是否有可复用的"经验/教训"值得保存为 skill。

【对话内容】
{conversation_summary}

【现有 skill 列表】
{existing_skills}

【你的决策规则(与 Hermes 源码一致)】
1. PATCH 优先:先看现有 skill 能不能补丁,没有匹配 skill 时才 CREATE 新的
2. ACTIVE 倾向:'Nothing to save' 是合法选项但不应该是默认——多数会话至少有 1 条可保存
3. skill 名必须 class-level(如 `kafka-debugging`、`pytest-flaky-test-isolation`),禁止 issue 编号 / 错误字符串
4. **渐进式披露**:content 必须以 YAML frontmatter 开头(含 name + description),
   description ≤1024 字符且说明"何时用这个 skill"——agent 在 skill 列表只看这两行决定是否加载 body

【可用动作】
- 决定保存:调用 write_skill(
    name='<class-level slug>',
    action='create' 或 'patch',
    description='<≤1024 字符,说明触发条件>',
    content='---\\nname: <name>\\ndescription: <description>\\n---\\n\\n<完整 markdown body>'
  )
- 决定不保存:直接回复 "Nothing to save.",不调用任何工具
"""


&emsp;&emsp;这段 prompt 是 `run_agent.py:3364-3438` 真实 `_SKILL_REVIEW_PROMPT` 的精简移植版——三个核心约束（`PATCH > CREATE` 优先级、`ACTIVE` 倾向、`class-level` 命名）等价保留，去掉了部分非核心 schema 校验和 token 预算提示。这是 fork sub-agent 收到的唯一指令，所有"自动学习"行为都来自这段写死的字符串——再次验证 2.5 节强调的"prompt 驱动的机械闭环，不是 AI 自主意识"。

&emsp;&emsp;**步骤三：实现 NudgeMiddleware（核心：Hermes hook 链的 langchain 标准化）**

> 📅 **版本声明**：以下代码已按 `langchain==1.2.15` 实测通过。`langchain` 1.x minor 升级时（如未来的 1.3.x），`AgentMiddleware` 的钩子签名（`after_model(state, runtime=None)` 这种参数列表）有可能微调，遇到方法签名相关错误时（langchain 在 `create_agent(middleware=[...])` 注入路径上对 `runtime` 参数有特殊处理，本 demo 用 `runtime=None` 默认值是安全覆写；如果未来你脱离 `create_agent` 直接调用 hook 方法，签名要求可能不同——以当前 langchain 版本的 `help()` 输出为准），先 `python -c "from langchain.agents.middleware import AgentMiddleware; help(AgentMiddleware.after_model)"` 看当前签名再对应调整。

&emsp;&emsp;现在到了 demo 的核心——把 Hermes 的 nudge 机制（计数器累加 → turn 末判定 → fork sub-agent → prompt 驱动 → 落盘）实现成一个 langchain `AgentMiddleware`。这个类的每个方法都对应 Hermes 源码的一段——`after_model` ↔ `_iters_since_skill += 1`；`after_agent` ↔ turn 末判定 + 触发 background review；`_spawn_review` 内部 `create_agent(tools=[write_skill, read_memory], middleware=[]).with_config({"recursion_limit": 8})` ↔ fork agent 4 重约束 ①②③④ 全部显式表达。

In [20]:
# hermes_mvp_demo.py — Part 3：NudgeMiddleware（核心 hook 链）+ fork agent 工具集
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek

# ===== fork agent 的"读记忆"工具（对应 Hermes enabled_toolsets=["memory","skills"] 的 memory 半边）=====
# 真 Hermes 的 fork agent 能读 ~/.hermes/memories/MEMORY.md 长期记忆，用来做"反思素材"。
# 演示版用一个模块级 list 模拟："每次 turn 结束把对话片段 append 进去 → review agent 调 read_memory 取出"。
MEMORY_STORE: list[str] = []

@tool
def read_memory(n: int = 5) -> str:
    """读取 agent 最近 N 条长期记忆 ↔ Hermes ~/.hermes/memories/MEMORY.md 读取动作。

    fork agent（review_agent）调这个 tool 拿"反思素材"——比 prompt 注入更显式、更受 langchain 工具
    调用追踪。生产环境对接 SQLite + FTS5 检索（§1.2-1.4 已讲），这里教学用最简 list。
    """
    if not MEMORY_STORE:
        return "(empty memory store)"
    return "\n".join(MEMORY_STORE[-n:])


class NudgeMiddleware(AgentMiddleware):
    """Hermes nudge 机制的 langchain 实现：计数器 + turn 末判定 + fork sub-agent。

    每个钩子方法 1:1 对应 Hermes run_agent.py 一段源码，参见行内 ↔ 注释。
    """

    def __init__(self, review_model, nudge_interval: int = 3, verbose: bool = True):
        """构造函数。

        Args:
            review_model: fork sub-agent 用的模型（演示中跟主 agent 共享）
            nudge_interval: 触发阈值，0 = 完全禁用（合规配置）
            verbose: True 时打印 [counter]/[trigger]/[review] 等观察日志（教学默认）；
                     False 时全程静默——对应 Hermes 4 重约束 ② quiet_mode=True 的真静默姿态。
                     生产部署应设为 False。
        """
        super().__init__()
        # ↔ run_agent.py:1705 self._iters_since_skill = 0
        self.counter = 0
        # ↔ run_agent.py:1810 self._skill_nudge_interval = int(skills_config.get("creation_nudge_interval", 10))
        self.nudge_interval = nudge_interval
        # 用于 fork sub-agent 的模型（生产中跟父 agent 共享，演示也保持一致）
        self.review_model = review_model
        # quiet_mode 开关（4 重约束 ②）
        self.verbose = verbose

    def after_model(self, state, runtime=None):
        """每次 LLM 调用后触发 ↔ run_agent.py:10870 self._iters_since_skill += 1"""
        self.counter += 1
        if self.verbose:
            print(f"[counter] nudge_counter = {self.counter} / {self.nudge_interval}")
        return None  # langchain middleware 约定：不修改 state 时返回 None

    def after_agent(self, state, runtime=None):
        """每个 user turn 结束时触发 ↔ run_agent.py:13918-13941 turn 末判定。"""
        if self.nudge_interval == 0:
            # ↔ 源码 self._skill_nudge_interval > 0 短路判定
            if self.verbose:
                print("[disabled] nudge_interval=0，完全禁用 skill nudge")
            return None
        if self.counter >= self.nudge_interval:
            # ↔ 触发 background review
            if self.verbose:
                print(f"[trigger] 计数器达阈值（{self.counter} ≥ {self.nudge_interval}），启动 background review")
            # 把当前 turn 对话片段写入 MEMORY_STORE（模拟"turn 结束写入长期记忆"），
            # review agent 调 read_memory 时能读到——补全 enabled_toolsets=["memory","skills"] 的 memory 半边
            messages = state.get("messages", [])
            for m in messages[-4:]:  # 取最近 4 条作为本 turn 的反思素材
                MEMORY_STORE.append(f"[{getattr(m, 'type', 'msg')}] {str(getattr(m, 'content', m))[:120]}")
            self._spawn_review(state)
            # ↔ 触发后归零
            self.counter = 0
        return None

    def _spawn_review(self, state):
        """fork 一个受限 sub-agent ↔ run_agent.py:3559-3645 _spawn_background_review。

        Hermes 4 重约束 → langchain 的对应实现（一一对应，全部显式收紧）：
          ① max_iterations=8         → .with_config({"recursion_limit": 8}) 显式收紧
                                        （不再依赖 langgraph 默认 25，对齐 Hermes 课纲设计稿）
          ② quiet_mode=True          → self.verbose=False 时全程静默（生产姿态）；
                                        verbose=True 是教学默认（让你看到现象）
          ③ enabled_toolsets=memory+skills  → tools=[write_skill, read_memory]
                                              （write_skill = skills 半边，read_memory = memory 半边）
          ④ 子 agent 防递归 interval=0     → middleware=[] 不再装 NudgeMiddleware
        """
        # 收集本轮 turn 的对话历史（最近 12 条消息足以覆盖典型 user turn）
        messages = state.get("messages", [])
        conversation = "\n".join(
            f"[{getattr(m, 'type', 'msg')}] {str(getattr(m, 'content', m))[:200]}"
            for m in messages[-12:]
        )
        # 列现有 skill（对应 prompt 中 existing_skills 上下文）
        existing = [p.name for p in SKILLS_DIR.iterdir() if p.is_dir()]
        prompt = _SKILL_REVIEW_PROMPT.format(
            conversation_summary=conversation[:2000],   # 限长 2000 字防 context 爆
            existing_skills=", ".join(existing) or "(empty)"
        )
        # fork 受限 sub-agent —— 4 重约束 ① ② ③ ④ 全部显式表达
        review_agent = create_agent(
            model=self.review_model,
            tools=[write_skill, read_memory],   # ③ 工具集隔离：skills 半边 + memory 半边
            middleware=[],                       # ④ 防递归：sub-agent 不装 NudgeMiddleware
        ).with_config({"recursion_limit": 8})    # ① 步数上限显式收紧到 Hermes 课纲档（8）
        # 执行
        try:
            result = review_agent.invoke({"messages": [HumanMessage(content=prompt)]})
            # 拿到 sub-agent 的最终回复（如 "Nothing to save." 或工具调用结果）
            final = result.get("messages", [])[-1]
            if self.verbose:
                print(f"[review] sub-agent 决定: {str(getattr(final, 'content', final))[:120]}")
        except Exception as e:
            if self.verbose:
                print(f"[review-error] sub-agent 异常（不影响主流程）: {type(e).__name__}: {str(e)[:100]}")

&emsp;&emsp;这段代码是整个 demo 的灵魂。请重点看三件事：

- 第一，**`after_model` 和 `after_agent` 的语义对应**——`after_model` 在每次 LLM 决策完后触发（对应 Hermes 的 `_iters_since_skill += 1`），`after_agent` 在整个 user turn 结束后触发（对应 Hermes 的 turn 末判定）。

- 第二，**`_spawn_review` 用 `create_agent(tools=[write_skill, read_memory], middleware=[]).with_config({"recursion_limit": 8})` 实现的 4 重约束**——这一行参数链把 Hermes 4 重约束的 ① ② ③ ④ **全部显式表达**：① 步数上限通过 `.with_config({"recursion_limit": 8})` 显式收紧到 Hermes 课纲档；② `quiet_mode` 通过 `verbose=False` 真静默（教学默认 `verbose=True` 让你看现象）；③ `enabled_toolsets=memory+skills` 对应 `tools=[write_skill, read_memory]` 双工具；④ 防递归对应 `middleware=[]` 不再装 `NudgeMiddleware`。

- 第三，**整个 `NudgeMiddleware` 不到 80 行**——但它跟 Hermes 真实源码的 `_iters_since_skill` 累加、`_should_review_skills` 判定、`_spawn_background_review` 执行三段共 200+ 行的核心逻辑同构。

> 📌 **【检查点 B · 核心机制实现完毕】**：到这里 §2.8 真正的核心——**业务工具（步骤一）+ 落盘工具 + read_memory + prompt（步骤二）+ NudgeMiddleware（步骤三）**——都已经写完了。这 3 步把 Hermes **4 重约束的 ①②③④ 全部显式表达**（步数上限 / 安静模式 / 工具集隔离 / 防递归）和"计数器→fork→落盘"的核心闭环全部用 langchain 标准件实现了一遍。**接下来 3 步是组装 + 跑测 + 改参实验**，主要是观察现象、不是新概念。如果觉得前 3 步信息密集，这里是另一个合理的休息点。

&emsp;&emsp;**两个看似相似但本质不同的"禁用机制"—— `nudge_interval=0` vs `middleware=[]`**：刚刚的 `NudgeMiddleware` 实现里出现了两个都跟"关掉 nudge"有关的开关——一个是 `nudge_interval=0`（合规配置），另一个是 `middleware=[]`（防递归）。这两个看起来都是"让 fork 不发生"，但**语义和粒度差异巨大**——容易混淆。下面这张对比表帮你提前锁定差异（避免后面 §2.9 的"测试 4"和 §2.10 的"测试 7"反复疑惑）。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 2-2.5：`nudge_interval=0` vs `middleware=[]` 两种禁用机制对比</font></p>
<div class="center">

| 维度 | `nudge_interval=0` | `middleware=[]` |
|---|---|---|
| 含义 | 计数器仍累加，但 turn 末判定**永不触发 fork** | **根本不装** `NudgeMiddleware`，连计数器都没有 |
| 关闭粒度 | **软关闭**（运行时可改回 `nudge_interval=10` 即恢复）| **硬剔除**（构造时不挂 middleware，事后无法激活）|
| 用在哪里 | **主 agent**（用户合规配置）| **fork 子 agent**（系统强制，4 重约束 ④ 防递归）|
| `after_model` 是否触发 |  仍触发，counter 每次 +1 |  整个 hook 不存在 |
| `after_agent` 行为 | 触发但首行短路 `if nudge_interval == 0: return` | hook 不存在 |
| 类比 | 报警器开关切到 OFF，但传感器还在工作 | 整套报警系统从未安装 |
| §2.9 / §2.10 测试覆盖 | 测试 4 守护（高频调用下 0 次 fork）| 测试 7 守护（review_agent `middleware=[]` 防递归链路）|

</div>

&emsp;&emsp;读这张表的姿势是——**这两个机制不可互换**：合规场景关闭主 agent 的 nudge 必须用 `nudge_interval=0`（保留运行时调节能力 + 不破坏 middleware 注入）；防递归 fork 子 agent 必须用 `middleware=[]`（确保子 agent 没有 hook 入口，从源头排除递归可能）。**如果两者搞反**——主 agent 用 `middleware=[]` 就完全失去 nudge 能力（永远不能再开），fork 子 agent 用 `nudge_interval=0` 就只是"计数器跑但不 fork"，子 agent 仍然可以被 prompt 操纵直接调 `write_skill`（绕过 fork 流程）——两种错误都让 4 重约束 ④ 形同虚设。

&emsp;&emsp;**步骤四：组装主 agent（业务 agent + nudge 中间件）**

&emsp;&emsp;最后一步——把业务工具、`NudgeMiddleware` 和 LLM 拼成一个主 agent。这一步等价于 Hermes 的 user-facing agent 主循环——它接收 user 消息，决定调哪个工具，挂着 nudge middleware 在每次 tool 调用后累加计数器，在 turn 末判定是否要触发自动学习。

&emsp;&emsp;在写工厂函数之前，我们先把**主 agent 的业务工具列表**提取成一个模块级常量 `MAIN_AGENT_TOOLS`——把"工具集隔离"从隐式约束（藏在函数实现里）升级成显式约束（一眼可见的常量声明），既让 4 重约束 ③ 在源码上一目了然，也是 §2.9 的"测试 5：工具集隔离"能直接 `assert "write_skill" not in MAIN_AGENT_TOOLS` 的前提。

In [21]:
# 主 agent 工具白名单——write_skill 不在此列表（工具集隔离 / Hermes 4 重约束 ③）
MAIN_AGENT_TOOLS = [search_kafka_logs, read_yaml_config]

&emsp;&emsp;这个常量之后会被 `make_main_agent` 内部 `tools=` 参数引用，工厂函数本身保持原貌。

In [22]:
# hermes_mvp_demo.py — Part 4：组装主 agent
def make_main_agent(nudge_interval: int = 3):
    """创建挂了 NudgeMiddleware 的主业务 agent，对应 Hermes 的 user-facing agent。"""
    model = ChatDeepSeek(
        model=os.getenv("DEEPSEEK_MODEL", "deepseek-chat"),
        api_key=os.getenv("DEEPSEEK_API_KEY"),
        api_base=os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com"),
        temperature=0.1,
    )
    return create_agent(
        model=model,
        tools=MAIN_AGENT_TOOLS,                               # 引用模块级常量（设计意图源头）
        middleware=[NudgeMiddleware(                          # nudge hook 链
            review_model=model,                               # sub-agent 复用同一模型
            nudge_interval=nudge_interval,
        )],
    )

&emsp;&emsp;到这里所有原语都齐了——主 agent 业务工具白名单（步骤四）+ NudgeMiddleware（步骤三）+ 落盘工具（步骤二）+ sub-agent 受限创建（步骤三内 `_spawn_review`）。**4 重约束 ①②③④ 在 langchain 里全部显式表达**：① `.with_config({"recursion_limit": 8})`、② `verbose=False` 静默、③ `tools=[write_skill, read_memory]`、④ `middleware=[]`——没有一项依赖框架默认值。**demo 边界声明**:§2.6 提到的五步校验(name / category / frontmatter / size / collision)中,**前 3 步已在 write_skill 接入**——name 由 sub-agent 走 prompt 引导(class-level slug)/ frontmatter 严格验证 + 缺则自动补全(渐进式披露 L1)/ description size ≤1024 字符校验;**仅 category(单目录 demo 不需要)+ collision(单实例 demo 不会撞)简化**。这跟 4 重约束 ①②③④ 的工程让步是两件事——4 重约束零让步,五步校验已对齐前 3 步。下面进入步骤五真正跑 demo。

In [23]:
# hermes_mvp_demo.py — Part 5：主程序，跑 5 轮 kafka 调试

# 5 轮 user turn——模拟工程师用 agent 调试 kafka rebalance 问题
fake_conversations = [
    "我的 kafka consumer 频繁 rebalance，先帮我搜一下日志看哪个 consumer group 出问题。",
    "看到 order-svc 的 rebalance 日志了，再读一下 application.yaml 看 session.timeout.ms 配置。",
    "session.timeout.ms 是 10s，怀疑太短了——再帮我搜一下生产数据看实际心跳间隔。",
    "确认一下 max.poll.interval.ms 配置是不是合理。",
    "max.poll.interval.ms=300000（5 分钟）合理。最终修复方案：把 session.timeout.ms 改成 30s。task 完成。",
]


def main():
    """5 轮 user turn → 每轮触发若干次 after_model + 1 次 after_agent → 累计触发 nudge。"""
    main_agent = make_main_agent(nudge_interval=3)
    for i, query in enumerate(fake_conversations, 1):
        print(f"\n=== 第 {i} 轮 user turn ===")
        print(f"用户：{query}")
        try:
            main_agent.invoke({"messages": [HumanMessage(content=query)]})
        except Exception as e:
            print(f"[main-error] 第 {i} 轮异常：{type(e).__name__}: {str(e)[:120]}")

    # 跑完查看 skills 目录
    print("\n=== 跑完 demo，列出 ./hermes_mvp_skills/ 产物 ===")
    for p in sorted(SKILLS_DIR.iterdir()):
        print(f"  - {p}")


if __name__ == "__main__":
    main()


=== 第 1 轮 user turn ===
用户：我的 kafka consumer 频繁 rebalance，先帮我搜一下日志看哪个 consumer group 出问题。
[counter] nudge_counter = 1 / 3
[counter] nudge_counter = 2 / 3
[counter] nudge_counter = 3 / 3
[counter] nudge_counter = 4 / 3
[counter] nudge_counter = 5 / 3
[trigger] 计数器达阈值（5 ≥ 3），启动 background review
[review] sub-agent 决定: 已 **PATCH** 现有 skill `kafka-consumer-rebalance-debugging`。

**补丁内容**：将对话中体现的"三步排查法"（搜日志定位 group → 检查 session.timeout.ms 

=== 第 2 轮 user turn ===
用户：看到 order-svc 的 rebalance 日志了，再读一下 application.yaml 看 session.timeout.ms 配置。
[counter] nudge_counter = 1 / 3
[counter] nudge_counter = 2 / 3

=== 第 3 轮 user turn ===
用户：session.timeout.ms 是 10s，怀疑太短了——再帮我搜一下生产数据看实际心跳间隔。
[counter] nudge_counter = 3 / 3
[counter] nudge_counter = 4 / 3
[trigger] 计数器达阈值（4 ≥ 3），启动 background review
[review] sub-agent 决定: 已保存。

**决策说明：**

对话中展示了一个关键模式：当有人怀疑 `session.timeout.ms` 太短时，首先要检查 `heartbeat.interval.ms` 是否按 **1/3 比例** 同步配置。这个经验正好可以补

=== 第 4 轮 user turn ===
用户：确认一下 max.poll.interval.ms 配置是不是合

&emsp;&emsp;这段代码就是真实的 `Hermes` 工作模式——agent 主循环里每次 LLM 决策完成后过一次 `after_model` 钩子（计数累加），每个 user turn 整体结束后过一次 `after_agent` 钩子（做判定），达阈值就 fork 一个**只能写 skill** 的 sub-agent 让它根据 `_SKILL_REVIEW_PROMPT` 决定保存什么。整个流程没有任何"AI 自主"—— 每一步都是确定性的 middleware 钩子回调 + 工具调用。

&emsp;&emsp;**怎么判断演示跑通了？** 看终端输出有没有出现这条序列就是成功信号：`[counter] nudge_counter = N / 3` →（达阈值时）`[trigger] 计数器达阈值` → `[review] sub-agent 决定: ...`（含 `Nothing to save.` 或工具调用）→（如果 LLM 决定保存）`[落盘] CREATE skill: ...` 或 `[落盘] PATCH skill: ...`。如果你看到 `[review-error] sub-agent 异常`，通常是 sub-agent 的 prompt 解析或工具参数问题，不影响主 agent 流程；如果反复出现 `Nothing to save.`，说明 LLM 判定这次会话片段没有可复用经验——这也是合法结果，可以重跑或下一步把 `nudge_interval` 调小再观察。跑完 5 轮后，执行下一个 cell 验证落盘。

In [24]:
!ls -R ./hermes_mvp_skills/

kafka-consumer-rebalance-debugging kafka-consumer-timeout-tuning

./hermes_mvp_skills/kafka-consumer-rebalance-debugging:
SKILL.md

./hermes_mvp_skills/kafka-consumer-timeout-tuning:
SKILL.md


&emsp;&emsp;**判读输出**：如果看到子目录里有 `SKILL.md` 文件冒出来，说明完整链路（计数器累加 → 阈值触发 → fork sub-agent → prompt 驱动 → 工具落盘）走通——这是成功。如果目录是空的，按以下顺序排查：① 终端是否反复出现 `[review] sub-agent 决定: Nothing to save.`？是 → LLM 合法判定这次会话没有可复用经验，**不是 bug**，重跑一次或调小 `nudge_interval` 增加触发次数即可看到产物；② 终端是否有 `DEEPSEEK_API_KEY` 相关错误或 `Connection refused`？是 → 检查 `.env` 文件配置和网络；③ 既没有 `[trigger]` 也没有 `[review]` 行？是 → 检查 `make_main_agent` 创建是否成功、`NudgeMiddleware` 是否真挂上去了。

&emsp;&emsp;**步骤六：改参数实验对比**

&emsp;&emsp;实战演示的真正价值在于让你改参数观察现象。下面三组实验帮你建立对 nudge 机制的直觉——同一组对话，仅改 `nudge_interval` 这个 middleware 参数，触发频率立即不同。**注意**：三组实验**必须按 A → B → C 顺序顺次跑**，每组单独一个 cell；`SKILLS_DIR` 是模块级全局变量（被 `write_skill` tool 引用），每组实验前重定向到不同子目录方便对比。如果你只想单独重跑某一组，先把那组的 `SKILLS_DIR = Path(...)` 那行先单独执行一次。

&emsp;&emsp;**实验 A**：把 `nudge_interval` 调到 `5`（比 demo 默认值 3 触发更慢；但比 Hermes 生产默认 10 仍更频繁），落盘到 `./hermes_mvp_skills_slow/` 子目录方便对比。

In [ ]:
# 实验 A：nudge_interval=5（相对演示默认值 3 触发更慢；相对 Hermes 生产默认 10 仍更频繁——3 / 5 / 10 三档之间的中间档）
import shutil
shutil.rmtree("./hermes_mvp_skills_slow", ignore_errors=True)
SKILLS_DIR = Path("./hermes_mvp_skills_slow")     # 重定向落盘到 _slow 子目录
SKILLS_DIR.mkdir(parents=True, exist_ok=True)

agent_slow = make_main_agent(nudge_interval=5)
for q in fake_conversations:
    print(f"\n[A] 用户：{q[:30]}...")
    try:
        agent_slow.invoke({"messages": [HumanMessage(content=q)]})
    except Exception as e:
        print(f"[A-error] {type(e).__name__}: {str(e)[:80]}")
# 预期：5 轮内 after_model 累加约 7-10 次，触发 1-2 次 review，落盘 ./hermes_mvp_skills_slow/ 应有 1-2 个 SKILL.md

&emsp;&emsp;**实验 B**：把 `nudge_interval` 调到 `2`，触发更密集，便于看 review 在不同 session 上的反思差异，落盘到 `./hermes_mvp_skills_fast/`。

In [70]:
# 实验 B：nudge_interval=2（更快触发，便于看多次 review）
shutil.rmtree("./hermes_mvp_skills_fast", ignore_errors=True)
SKILLS_DIR = Path("./hermes_mvp_skills_fast")     # 重定向落盘到 _fast 子目录
SKILLS_DIR.mkdir(parents=True, exist_ok=True)

agent_fast = make_main_agent(nudge_interval=2)
for q in fake_conversations:
    print(f"\n[B] 用户：{q[:30]}...")
    try:
        agent_fast.invoke({"messages": [HumanMessage(content=q)]})
    except Exception as e:
        print(f"[B-error] {type(e).__name__}: {str(e)[:80]}")
# 预期：5 轮内累计触发 3-5 次 review，落盘 ./hermes_mvp_skills_fast/ 应有 3+ 个 SKILL.md


[B] 用户：我的 kafka consumer 频繁 rebalance...
[counter] nudge_counter = 1 / 2
[counter] nudge_counter = 2 / 2
[counter] nudge_counter = 3 / 2
[counter] nudge_counter = 4 / 2
[counter] nudge_counter = 5 / 2
[counter] nudge_counter = 6 / 2
[counter] nudge_counter = 7 / 2
[trigger] 计数器达阈值（7 ≥ 2），启动 background review
[review] sub-agent 决定: 已保存 skill **`kafka-consumer-rebalance-debugging`**。

**保存理由：** 该对话片段展示了 Kafka consumer 频繁 rebalance 的典型排查场景——`session.ti

[B] 用户：看到 order-svc 的 rebalance 日志了，再...
[counter] nudge_counter = 1 / 2
[counter] nudge_counter = 2 / 2
[trigger] 计数器达阈值（2 ≥ 2），启动 background review
[review] sub-agent 决定: 我来分析这个对话片段。

**分析：**
1. 对话中，用户让 AI 读取 `application.yaml` 查看 `session.timeout.ms` 配置，AI 成功读取并展示了 Kafka consumer 的配置项。
2. 

[B] 用户：session.timeout.ms 是 10s，怀疑太短了...
[counter] nudge_counter = 1 / 2
[counter] nudge_counter = 2 / 2
[trigger] 计数器达阈值（2 ≥ 2），启动 background review
[review] sub-agent 决定: **已保存。** 我 patch 了现有 skill `kafka-consumer-rebalance-debugging`，新增了 **Timeou

&emsp;&emsp;**实验 C**：把 `nudge_interval` 调到 `0`，对应医疗 / 金融 / 法律 / 政府等合规场景必备的"绝对禁用"，应看到 5 轮全部 `[disabled]`、0 次 fork、`./hermes_mvp_skills_off/` 全程为空。

In [27]:
import shutil

# 实验 C：nudge_interval=0（完全禁用，对应生产环境合规配置）
shutil.rmtree("./hermes_mvp_skills_off", ignore_errors=True)
SKILLS_DIR = Path("./hermes_mvp_skills_off")      # 重定向落盘到 _off 子目录
SKILLS_DIR.mkdir(parents=True, exist_ok=True)

agent_off = make_main_agent(nudge_interval=0)
for q in fake_conversations:
    print(f"\n[C] 用户：{q[:30]}...")
    try:
        agent_off.invoke({"messages": [HumanMessage(content=q)]})
    except Exception as e:
        print(f"[C-error] {type(e).__name__}: {str(e)[:80]}")
# 预期：5 轮全部 [disabled]，0 次 review，./hermes_mvp_skills_off/ 应为空


[C] 用户：我的 kafka consumer 频繁 rebalance...
[counter] nudge_counter = 1 / 0
[counter] nudge_counter = 2 / 0
[counter] nudge_counter = 3 / 0
[counter] nudge_counter = 4 / 0
[counter] nudge_counter = 5 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge

[C] 用户：看到 order-svc 的 rebalance 日志了，再...
[counter] nudge_counter = 6 / 0
[counter] nudge_counter = 7 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge

[C] 用户：session.timeout.ms 是 10s，怀疑太短了...
[counter] nudge_counter = 8 / 0
[counter] nudge_counter = 9 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge

[C] 用户：确认一下 max.poll.interval.ms 配置是不...
[counter] nudge_counter = 10 / 0
[counter] nudge_counter = 11 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge

[C] 用户：max.poll.interval.ms=300000（5 ...
[counter] nudge_counter = 12 / 0
[counter] nudge_counter = 13 / 0
[counter] nudge_counter = 14 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge


&emsp;&emsp;实验 A 模拟温和触发（间隔 5）；实验 B 模拟密集触发（间隔 2，方便观察 prompt 驱动的反思在不同 session 上的表现差异）；实验 C 是合规场景必须的禁用配置（间隔 0）。

&emsp;&emsp;**判读三组对比**：跑完三组后用 `!ls -R ./hermes_mvp_skills_*/` 一次性看三个目录的产物——你应该看到 

① `_off` 目录**完全为空**（0 个 SKILL.md，因为 nudge 全程 disabled）；

② `_slow` 目录有 **1-2 个** SKILL.md（5 轮内只触发 1-2 次 review）；

③ `_fast` 目录有 **3+ 个** SKILL.md（5 轮内每 2 次 model 调用就触发一次 review）。三组目录里 SKILL.md 数量从 0 → 1-2 → 3+ 递增，对应 `nudge_interval` 从禁用 → 温和 → 密集。**如果数量不严格递增**，最常见原因是 LLM 偶发返回 `Nothing to save.`（合法行为，不是 bug）—— `_fast` 组在多次触发中只要有一次 LLM 决定保存，就一定多于 `_slow` 组。

> &emsp;&emsp;关于 demo 的边界——这个最小实现把 **4 重约束 ① ② ③ ④ 全部显式表达**:① `max_iterations=8` → `.with_config({"recursion_limit": 8})`;② `quiet_mode=True` → `verbose=False` 全程静默(教学默认 `verbose=True` 让你看现象);③ `enabled_toolsets=["memory","skills"]` → `tools=[write_skill, read_memory]`;④ 防递归 → `middleware=[]`。**五步校验已对齐前 3 步**:name(sub-agent prompt 引导 class-level slug)+ frontmatter 严格验证(必含 name + description,description ≤1024 chars)+ size 校验;**仅 category 和 collision 简化**(单目录单实例 demo 不需要)。本演示作为教学最小骨架,**4 重约束零让步 + 五步校验对齐前 3 步 + 渐进式披露 L1 落地**,跟 §2.6 真 Hermes `_validate_frontmatter` 行为一致。

&emsp;&emsp;运行完这个 demo，你已经亲眼看到了 nudge 从计数器到文件的完整链路——计数器累加、阈值触发、fork agent、prompt 驱动决策、SKILL.md 落盘，每一步都在你的终端输出里留下了痕迹。**第二章的主线到此结束**——`Hermes` Nudge 机制 + MVP demo 已经把"自动学 skill"的工程真相讲透。

&emsp;&emsp;**接下来怎么走?两条路自选**:① **第一遍学习时间紧** → 跳过 §2.9-§2.10 直接进第 3 章四维取舍分析(主线完整无缺);② **想把这套 langchain 中间件工程能力带回自己项目** → 跟着 §2.9-§2.10 跑 7 个第一性原理测试：§2.9 先用 6 个部件级测试把计数器、阈值判定、`nudge_interval=0` 合规硬开关、工具集隔离、频率旋钮逐条证伪；§2.10 再用 1 个完整链路测试证明 `_spawn_review → create_agent → tool call → write_skill → SKILL.md` 真的跑通。两条路最终都会回到第 3 章——用**四维评价框架**横向看 `Hermes` 总体选了什么、放弃了什么。**这才是今天这节课最有迁移价值的结论:不是"`Hermes` 多厉害",而是"任何 harness 都能用同一把尺评价它的取舍"**。

### 2.9 自动化验证（一）：先测每个小部件是否可靠

&emsp;&emsp;上一节我们用 `langchain AgentMiddleware` + `@tool` 把 `Hermes` 的 nudge 机制完整实现了一遍——但**「能跑」不等于「契约成立」**。`main_agent.invoke(...)` 跑出来一句 `[落盘] CREATE skill: ...` 只能证明今天这一次运行没报错，证明不了"`nudge_interval=0` 永远不会 fork"，也证明不了"计数器在第 3 次累加时阈值判定**真的**触发"。真正带走 langchain `middleware` / `tool` 工程能力的标志，不是"我跑通了 demo"，而是"我能写出一组测试，把契约证给自己看"。

&emsp;&emsp;你可以把 nudge 机制想成一条自动化流水线。流水线里有很多小部件：计数器、阈值判断、禁用开关、工具白名单、落盘工具。**这一节先不测完整 agent，只把这些小部件一个个拿出来测**。这样一旦测试挂了，我们能马上知道是哪一个部件坏了。工程测试里有时会把这类测试叫 `Tier 1`，但这里你不用先记这个英文标签，只需要记住一句话：**2.9 测的是单个小部件，不测整条流水线**。

&emsp;&emsp;**第一性原理的测试设计核心是两问**：先问"这段代码定义承诺了什么"，再问"什么样的输出能证明承诺真的成立、而不是被绕开"。比如 `write_skill` 承诺"把 markdown 真正写到磁盘"——那测试就必须真的去磁盘检查 `SKILL.md`，而不是 `assert result is not None` 看函数有返回值就放过；`nudge_interval=0` 承诺"任何输入下永不 fork"——那测试必须高频跑几轮都看不到一次 fork，而不是单个 happy path 跑完就收工。**hello world 断言（`assert is not None`、跑通即过）能证明的事跟"今天没下雨"差不多——它绕开所有负向场景（合规禁用、阈值边界、工具集隔离），把最致命的契约漏洞留在生产环境炸**——这就是大量项目里"测试都过了但生产还是炸"的根本原因。

&emsp;&emsp;接下来这一节用 6 个部件级测试覆盖 §2.8 `NudgeMiddleware` 的关键小部件。每个测试都先用 1-2 段叙述讲清"这个测试在守护什么承诺、什么样的输出能证伪它"，再上代码。下面这张表先列出这一节要测的 6 个部件，跑完之后回过头再看一次会更清楚。完整链路测试会放到 §2.10 单独讲。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 2-3：§2.9 六个部件级测试覆盖的契约一览</font></p>
<div class="center">

| 测试 | 验证的契约 | 关键负向 / 边界 |
|---|---|---|
| 测试 1 | `write_skill` 真落盘 + 内容字节对字节相等 | 路径结构（`name/SKILL.md` 而不是 `name.md`）|
| 测试 2 | `after_model` 计数器累加正确 + 不破坏 state | 初始值是 `0` 不是 `None` / `-1` |
| 测试 3 | `after_agent` 阈值判定 + fork 1 次 + counter 重置归 0 | 重置后能再次累加触发，下一轮再 fork |
| 测试 4 | `nudge_interval=0` 永不 fork（合规硬开关）| 高频调用下不能有任何一次 fork |
| 测试 5 | 工具集隔离：`MAIN_AGENT_TOOLS` 常量含业务工具、不含 `write_skill` | 正向 2 + 负向 1 + 数量精确 |
| 测试 6 | 频率控制契约：高 `interval` 触发更少 | `nudge=3` 跑 15 次 → 5 次 fork；`nudge=5` → 3 次 fork |

</div>

&emsp;&emsp;在写第一个测试之前，我们先准备两个共用的 helper——`mock LLM` 和测试目录管理。这两段代码会在后面这些测试里反复出现，先一次写好。

&emsp;&emsp;为什么需要 mock LLM？真实 `ChatDeepSeek` 调一次要联网、要消耗 token、返回还有不确定性——这三件事都跟测试要的"快、确定、零成本"矛盾。`langchain_core` 自带 `FakeListChatModel`，传一个回复列表进去，它就按顺序吐回——离线、即时、跨次完全一致。**适用边界要先讲清楚**：`FakeListChatModel` 只实现 `_generate`、**未实现 `bind_tools`**——这意味着它只能用在**不走 tool calling 的 middleware 单元测试**（测试 2-4 / 测试 6 这类只验证 `after_model` / `after_agent` 钩子语义的场景）；**任何涉及 sub-agent 真调 tool 的完整链路测试不能用它**——`create_agent` 内部 `_get_bound_model` 会抛 `NotImplementedError`。涉及 tool calling 的完整链路测试必须用 §2.10 自定义的 `ScriptedChatModel`，那里我们补出 `bind_tools` 接口并按脚本吐 `tool_calls`。

&emsp;&emsp;测试目录用 `./hermes_mvp_skills_test/` 固定路径而非 `tempfile.mkdtemp()`——`tempfile` 测完即删，学员失去 `ls` 看产物的视觉锚点，落盘成果就只剩"绿勾"一个抽象信号；固定目录每次测试前清空，学员可以真切看到 `SKILL.md` 在磁盘上长出来。

In [28]:
# §2.9 测试 fixture：mock LLM + 测试目录管理 + fork 计数 spy
import shutil
from pathlib import Path
from langchain_core.language_models.fake_chat_models import FakeListChatModel

# 测试落盘目录（固定路径，方便学员 ls 验证）
TEST_SKILLS_DIR = Path("./hermes_mvp_skills_test")


def setup_test_dir():
    """每次测试前清空目录，保证从干净状态开始。

    没有这个 helper，测试间会有残留文件互相污染——
    比如测试 1 写的 SKILL.md 让完整链路测试误以为"端到端跑通"。
    """
    # 每个测试都从空目录开始，避免历史文件影响判断。
    if TEST_SKILLS_DIR.exists():
        shutil.rmtree(TEST_SKILLS_DIR)
    TEST_SKILLS_DIR.mkdir(parents=True, exist_ok=True)


def make_fake_llm(scripted_responses):
    """装配按脚本顺序回复的 mock LLM。

    Args:
        scripted_responses: list[str]，按顺序消费——第 N 次 invoke 返回第 N 条

    用途：让 NudgeMiddleware 链路在不联网、不消耗 token 下运行；
         测试可精确控制"LLM 调了几次"，这是计数器契约的前提。
    """
    # FakeListChatModel 只适合不走 tool calling 的部件级测试。
    return FakeListChatModel(responses=scripted_responses)


class SpyNudgeMiddleware(NudgeMiddleware):
    """只记录 fork 次数，不真创建 review agent。

    后面的阈值、禁用、频率测试都会复用它，避免每个测试重复写一遍 spy 类。
    """
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # 记录每次 fork 发生时的 counter 值。
        self.spawn_calls = []

    def _spawn_review(self, state):
        # 不创建 review agent，只记录"本来会 fork"。
        self.spawn_calls.append(self.counter)


def count_forks(nudge_interval: int, steps: int = 15) -> int:
    """跑 steps 次 after_model + after_agent，返回触发 fork 的次数。"""
    mw = SpyNudgeMiddleware(
        review_model=make_fake_llm(["Nothing to save."]),
        nudge_interval=nudge_interval,
    )
    fake_state = {"messages": []}
    for _ in range(steps):
        # 模拟一次模型调用结束 + 一次 turn 末判定。
        mw.after_model(fake_state, runtime=None)
        mw.after_agent(fake_state, runtime=None)
    return len(mw.spawn_calls)

&emsp;&emsp;这段 fixture 里真正需要学员记住的只有三个东西：`setup_test_dir()` 保证每次从干净目录开始，`make_fake_llm()` 避免真实联网，`SpyNudgeMiddleware` 用来数"本来会不会 fork"。后面的测试只围绕"构造场景 → 触发动作 → 断言契约"展开，不再重复测试夹具代码。

#### 测试 1：`write_skill` 落盘契约——真磁盘真有文件、真字节相等

&emsp;&emsp;`write_skill` 是 nudge 链路的"输出端"——文件没建、内容不等、路径不合规，下游所有依赖（`Curator` 整理、用户 `cat`、`Hermes` 启动扫描加载到 system prompt）全部失效。课堂里不需要把所有边界都展开成复杂测试，只抓两个最关键证据：**文件路径对不对、文件内容对不对**。

In [29]:
# 测试 1：write_skill 落盘契约
def test_write_skill_落盘契约():
    """验证 write_skill 真把 markdown 写到 <name>/SKILL.md。"""
    setup_test_dir()
    # write_skill 读取全局 SKILLS_DIR，这里临时改到测试目录。
    global SKILLS_DIR  # 临时把全局 SKILLS_DIR 指向测试目录（write_skill 在 §2.8 闭包读此变量）
    original_skills_dir = SKILLS_DIR
    SKILLS_DIR = TEST_SKILLS_DIR

    try:
        expected_content = (
            "---\n"
            "name: kafka-debugging\n"
            "description: kafka rebalance 频繁触发时的排查清单(渐进式披露 L1 入口)\n"
            "---\n"
            "\n"
            "# kafka-debugging\n\n## 适用场景\nkafka rebalance 频繁触发\n"
        )
        # 通过 LangChain tool 标准入口调用,而不是直接调函数。
        write_skill.invoke({
            "name": "kafka-debugging",
            "action": "create",
            "content": expected_content,
        })

        # 核心证据:① 路径结构正确 ② 含 frontmatter 的 content 字节级保留 ③ frontmatter 含 name+description
        skill_file = TEST_SKILLS_DIR / "kafka-debugging" / "SKILL.md"
        assert skill_file.exists(), f"SKILL.md 未落盘: {skill_file}"
        actual = skill_file.read_text(encoding="utf-8")
        assert actual == expected_content, "路径 A 应字节级保留输入(_validate_frontmatter 已通过)"
        assert actual.startswith("---\n"), "渐进式披露 L1: 必须以 frontmatter 开头"
        import yaml as _yaml, re as _re
        _end = _re.search(r'\n---\s*\n', actual[3:])
        _fm = _yaml.safe_load(actual[3:_end.start() + 3])
        assert _fm.get("name") == "kafka-debugging", "frontmatter name 字段必须是 class-level slug"
        assert "description" in _fm and len(str(_fm["description"])) <= 1024, "description ≤1024 chars"

        # 打印正向证据,避免"assert 静默通过"。
        print(f"测试 1 证据: file={skill_file}, bytes={skill_file.stat().st_size}, "
              f"L1 metadata: name={_fm['name']}, desc 长度={len(_fm['description'])}")
        return "PASS"
    finally:
        SKILLS_DIR = original_skills_dir  # 恢复全局，避免污染后续


print(test_write_skill_落盘契约())

测试 1 证据: file=hermes_mvp_skills_test/kafka-debugging/SKILL.md, bytes=182, L1 metadata: name=kafka-debugging, desc 长度=39
PASS


&emsp;&emsp;这个测试只看两个可见结果：`kafka-debugging/SKILL.md` 是否存在，文件字节数是否大于 0 且内容完全等于输入。路径和内容都对，才能说明落盘工具这一步可信。

#### 测试 2：`after_model` 计数器累加——0 起始、连跑 N 次后等于 N

&emsp;&emsp;整个 fork 决策都依赖一个数——`self.counter`。算少永远不到阈值（机制失效），算多过早触发（资源浪费）。课堂里先只验证最核心的一点：**每次 `after_model` 被调用，counter 就加 1**。生产测试可以再补 `state` 不变、并发安全等更细的断言。

In [53]:
# 测试 2：after_model 计数器累加契约
def test_counter_累加契约():
    """验证 counter 初始为 0，连跑 5 次 after_model 后等于 5。"""
    # interval 设很大，避免测试过程中触发 fork。
    mw = NudgeMiddleware(review_model=make_fake_llm(["Nothing to save."]), nudge_interval=999)
    assert mw.counter == 0, f"counter 初始应为 0，实际 {mw.counter}"

    fake_state = {"messages": []}
    for _ in range(5):
        # after_model 是 counter 累加点。
        mw.after_model(fake_state, runtime=None)

    assert mw.counter == 5, f"5 次 after_model 后 counter 应为 5，实际 {mw.counter}"
    print(f"测试 2 证据: counter={mw.counter}")

    return "PASS"


print(test_counter_累加契约())

[counter] nudge_counter = 1 / 999
[counter] nudge_counter = 2 / 999
[counter] nudge_counter = 3 / 999
[counter] nudge_counter = 4 / 999
[counter] nudge_counter = 5 / 999
测试 2 证据: counter=5
PASS


&emsp;&emsp;测试 2 守护的是整个 nudge 机制的最底层不变量——它挂了，后面的触发、禁用、频率测试全都不能信，所以必须独立测、不能混进测试 3 顺便验证。

#### 测试 3：`after_agent` 阈值 + fork + 重置——nudge 的输出动作三件套

&emsp;&emsp;`after_agent` 是 nudge 输出端，做三件事：① 阈值判定 ② 达标 fork sub-agent ③ fork 后 counter 重置归 0。这里复用前面的 `SpyNudgeMiddleware`，不真跑 sub-agent，只记录"如果是真环境，会 fork 几次"。

In [54]:
# 测试 3：after_agent 阈值 + fork + 重置契约
def test_after_agent_阈值与重置():
    """验证未到阈值不 fork，到阈值 fork，fork 后 counter 归 0。"""
    # 用 spy 避免真创建 review agent，专心测试判定逻辑。
    mw = SpyNudgeMiddleware(
        review_model=make_fake_llm(["Nothing to save."]),
        nudge_interval=3,
    )
    fake_state = {"messages": []}

    # 未达阈值：不能 fork。
    mw.counter = 2
    mw.after_agent(fake_state, runtime=None)
    assert len(mw.spawn_calls) == 0, "counter=2 < 3 时不应 fork"

    # 刚达阈值：必须 fork，随后归零。
    mw.counter = 3
    mw.after_agent(fake_state, runtime=None)
    assert len(mw.spawn_calls) == 1, "counter=3 应 fork 1 次"
    assert mw.counter == 0, "fork 后 counter 应重置归 0"

    print(f"测试 3 证据: spawn_calls={mw.spawn_calls}, counter_after={mw.counter}")

    return "PASS"


print(test_after_agent_阈值与重置())

[trigger] 计数器达阈值（3 ≥ 3），启动 background review
测试 3 证据: spawn_calls=[3], counter_after=0
PASS


&emsp;&emsp;这段代码只保留两个关键 case：未到阈值不能触发，刚到阈值必须触发并归零。学员先把这两个分支看懂，就能理解 nudge 的 turn-end 判定。

#### 测试 4：`nudge_interval=0` 永不 fork——合规场景的硬开关

&emsp;&emsp;`nudge_interval=0` 是 §2.8 demo 提供的**合规硬开关**——医疗、金融、法律、政府这些场景里，agent 自动写文件可能踩 GDPR / HIPAA / 等保红线。一旦学员在生产环境配了 `nudge_interval=0` 期望"完全禁用 nudge"，**任何一次意外 fork 都意味着合规风险事故**。这不是"少触发一两次"的优化问题，而是"任何一次都不能发生"的硬契约。

&emsp;&emsp;所以测试 4 不需要复杂场景，只要高频跑几轮后证明 fork 次数仍然是 0。课堂里用 20 次足够建立直觉：**禁用开关不是少触发，而是绝对不触发**。

In [55]:
# 测试 4：nudge_interval=0 合规硬开关契约
def test_nudge_interval_zero_永不_fork():
    """验证 interval=0 时严格 0 次 fork。"""
    # interval=0 是合规禁用开关，任何输入下都不能 fork。
    mw = SpyNudgeMiddleware(
        review_model=make_fake_llm(["Nothing to save."]),
        nudge_interval=0,  # 关键：硬开关禁用
    )
    fake_state = {"messages": []}

    for _ in range(20):
        # 高频模拟：counter 可以涨，但 after_agent 必须短路。
        mw.after_model(fake_state, runtime=None)
        mw.after_agent(fake_state, runtime=None)

    assert len(mw.spawn_calls) == 0, (
        f"interval=0 时应严格 0 次 fork，实际 {len(mw.spawn_calls)} 次——合规风险"
    )
    print(f"测试 4 证据: after_model_runs=20, fork_count={len(mw.spawn_calls)}")

    return "PASS"


print(test_nudge_interval_zero_永不_fork())

[counter] nudge_counter = 1 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 2 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 3 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 4 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 5 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 6 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 7 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 8 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 9 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 10 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 11 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 12 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 13 / 0
[disabled] nudge_interval=0，完全禁用 skill n

&emsp;&emsp;测试 4 跟测试 3 同样调 `after_agent`，但契约严苛度不同：测试 3 守护"功能正确"（达阈值 fork、未达不 fork），测试 4 守护"绝对禁用"（任何输入下严格 0 次 fork）——合规事故是不可逆的，弱契约 `<=1` 不够。

#### 测试 5：工具集隔离——`write_skill` 不在主 agent 业务工具列表

&emsp;&emsp;工具集隔离是 §2.4 表 2-2 的 **4 重约束 ③**——主 agent 在 user turn 里只能调业务工具，**不能直接调 `write_skill`**。这里不用钻 LangChain graph 内部，只检查我们自己定义的工具白名单：业务工具在里面，`write_skill` 不在里面。

In [56]:
# 测试 5：工具集隔离契约——直接断言模块级常量（设计意图源头）
def test_main_agent_工具集隔离():
    """验证 MAIN_AGENT_TOOLS 含 search_kafka_logs+read_yaml_config，不含 write_skill。"""
    # 直接检查白名单，避免依赖 LangChain 内部 graph 结构。
    tool_names = [t.name for t in MAIN_AGENT_TOOLS]

    # 主 agent 只能有业务工具，不能有落盘工具。
    assert "search_kafka_logs" in tool_names, f"业务工具缺失：{tool_names}"
    assert "read_yaml_config" in tool_names, f"业务工具缺失：{tool_names}"
    assert "write_skill" not in tool_names, (
        f"write_skill 不应在主 agent 工具列表（违反 4 重约束 ③）：{tool_names}"
    )
    print(f"测试 5 证据: main_agent_tools={tool_names}")

    return "PASS"


print(test_main_agent_工具集隔离())

测试 5 证据: main_agent_tools=['search_kafka_logs', 'read_yaml_config']
PASS


&emsp;&emsp;这里最关键的是负向断言：`write_skill` 必须不在 `MAIN_AGENT_TOOLS`。这比检查 agent 内部 graph 更稳定，也更贴近教学目标。

&emsp;&emsp;测试 5 是最后一个负向断言测试——下面进入测试 6：频率控制，验证 `nudge_interval` 这个调节旋钮真有效。完整链路测试移到了 §2.10 单独展开：本节守护"每个小部件能用"，§2.10 守护"组装起来真能跑"。这就是两层验证方法论的分工：**只测小部件不等于完整链路能跑，必须两层都做才算契约真证明**。

#### 测试 6：频率控制契约——`nudge_interval` 旋钮真有效

&emsp;&emsp;`nudge_interval` 是 §2.8 demo 的核心调节旋钮——学员调高它应该看到 fork 触发更少。这里复用 `count_forks`，让代码只表达一个结论：**同样跑 15 步，interval 越大，fork 越少**。

In [57]:
# 测试 6：频率控制契约——高 interval 触发更少次
def test_higher_interval_触发更少():
    """验证 interval=5 跑 15 次 fork 3 次；interval=3 跑 15 次 fork 5 次。"""
    # 同样 steps 下，对比两个 interval 的 fork 次数。
    fork_count_3 = count_forks(nudge_interval=3, steps=15)
    fork_count_5 = count_forks(nudge_interval=5, steps=15)

    # 精确次数能抓 off-by-one 和重置错误。
    assert fork_count_3 == 5, f"interval=3 跑 15 次应触发 5 次，实际 {fork_count_3}"
    assert fork_count_5 == 3, f"interval=5 跑 15 次应触发 3 次，实际 {fork_count_5}"
    print(f"测试 6 证据: interval=3 forks={fork_count_3}, interval=5 forks={fork_count_5}")

    return "PASS"


print(test_higher_interval_触发更少())

[counter] nudge_counter = 1 / 3
[counter] nudge_counter = 2 / 3
[counter] nudge_counter = 3 / 3
[trigger] 计数器达阈值（3 ≥ 3），启动 background review
[counter] nudge_counter = 1 / 3
[counter] nudge_counter = 2 / 3
[counter] nudge_counter = 3 / 3
[trigger] 计数器达阈值（3 ≥ 3），启动 background review
[counter] nudge_counter = 1 / 3
[counter] nudge_counter = 2 / 3
[counter] nudge_counter = 3 / 3
[trigger] 计数器达阈值（3 ≥ 3），启动 background review
[counter] nudge_counter = 1 / 3
[counter] nudge_counter = 2 / 3
[counter] nudge_counter = 3 / 3
[trigger] 计数器达阈值（3 ≥ 3），启动 background review
[counter] nudge_counter = 1 / 3
[counter] nudge_counter = 2 / 3
[counter] nudge_counter = 3 / 3
[trigger] 计数器达阈值（3 ≥ 3），启动 background review
[counter] nudge_counter = 1 / 5
[counter] nudge_counter = 2 / 5
[counter] nudge_counter = 3 / 5
[counter] nudge_counter = 4 / 5
[counter] nudge_counter = 5 / 5
[trigger] 计数器达阈值（5 ≥ 5），启动 background review
[counter] nudge_counter = 1 / 5
[counter] nudge_counter = 2 / 5
[counter] nudge_counter = 

&emsp;&emsp;强契约（精确次数）挂了反而是好事——它精准告诉你哪类 bug，比"弱契约 `<=` 跑通就过"提供的诊断信息多一个数量级。这就是为什么测试不该只敢写 `<=`。

#### 一键运行全部部件测试：`run_all_tests` + 视觉验证

&emsp;&emsp;6 个部件测试逐个跑也能通，但教学场景下我们希望**一个 cell 就能跑全套**——这样学员实验改 §2.8 代码后能立刻看到每个小部件的契约是否还成立。下面这个 runner 把 6 个测试串起来，统计 PASS/FAIL，最后 `ls TEST_SKILLS_DIR` 把测试落盘的产物目录树打印出来——既验证测试结果、又给出视觉锚点（学员能看到自己的测试在磁盘上写了什么）。

In [58]:
# §2.9 测试 runner：一站式跑 6 个部件级测试
# 注：完整链路测试在 §2.10 单独跑（不在本 runner 内）
def run_all_tests():
    """跑 6 个部件级测试，统计 PASS/FAIL，最后 ls 测试目录给视觉验证。"""
    tests = [
        ("测试 1 write_skill 落盘", test_write_skill_落盘契约),
        ("测试 2 counter 累加", test_counter_累加契约),
        ("测试 3 after_agent 阈值+重置", test_after_agent_阈值与重置),
        ("测试 4 interval=0 永不 fork", test_nudge_interval_zero_永不_fork),
        ("测试 5 工具集隔离", test_main_agent_工具集隔离),
        ("测试 6 频率控制", test_higher_interval_触发更少),
    ]

    # 逐个跑+收集结果（捕获异常显示 FAIL，不中断后续测试）
    results = []
    for name, fn in tests:
        try:
            # 单个测试失败不影响后续测试继续执行。
            result = fn()
            results.append((name, result))
            print(f"[{result}] {name}")
        except AssertionError as e:
            results.append((name, f"FAIL: {e}"))
            print(f"[FAIL] {name}: {e}")
        except Exception as e:
            results.append((name, f"ERROR: {type(e).__name__}: {e}"))
            print(f"[ERROR] {name}: {type(e).__name__}: {e}")

    # 统计
    pass_count = sum(1 for _, r in results if r == "PASS")
    print(f"\n=== 测试结果：{pass_count}/{len(tests)} PASS ===")

    # 视觉验证：列出测试目录所有产物（学员能看到落盘了什么）
    print(f"\n=== 测试落盘目录 {TEST_SKILLS_DIR} 产物 ===")
    if TEST_SKILLS_DIR.exists():
        for p in sorted(TEST_SKILLS_DIR.rglob("*")):
            indent = "  " * (len(p.relative_to(TEST_SKILLS_DIR).parts) - 1)
            print(f"{indent}{p.name}")
    else:
        print("  (目录不存在——测试 1 或完整链路测试未跑过，或目录已被清理)")

    return pass_count == len(tests)


# 一站式跑全套
run_all_tests()

测试 1 证据: file=hermes_mvp_skills_test/kafka-debugging/SKILL.md, bytes=64
[PASS] 测试 1 write_skill 落盘
[counter] nudge_counter = 1 / 999
[counter] nudge_counter = 2 / 999
[counter] nudge_counter = 3 / 999
[counter] nudge_counter = 4 / 999
[counter] nudge_counter = 5 / 999
测试 2 证据: counter=5
[PASS] 测试 2 counter 累加
[trigger] 计数器达阈值（3 ≥ 3），启动 background review
测试 3 证据: spawn_calls=[3], counter_after=0
[PASS] 测试 3 after_agent 阈值+重置
[counter] nudge_counter = 1 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 2 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 3 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 4 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 5 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 6 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 7 / 0
[disabled] nudge_interval=0，完全禁用 skill nudge
[counter] nudge_counter = 8 / 0
[

True

&emsp;&emsp;读完 runner 你应该看到 6 行 `[PASS]` + `6/6 PASS` 总结 + 测试目录产物树（含测试 1 落盘的 `kafka-debugging/SKILL.md`）——这棵树就是教学价值的最直接证据，把抽象绿勾变成磁盘上能 `cat` 的真文件。**但 6 个部件级测试全 PASS 也只证明小部件能用——`_spawn_review → create_agent → tool call → write_skill` 这条完整链路任意一环断了，6 个部件测试都不会挂，但 user turn 里 fork agent 一句话不说就退出，"自动学 skill"从未真发生**。这就是 §2.10 完整链路测试要守护的契约边界——下一节用 1 个代表性 case + `ScriptedChatModel`，强制全链路真跑通磁盘真出文件，才宣告"组装起来能跑"。

---

### 2.10 自动化验证（二）：再测完整链路是否真的跑通

&emsp;&emsp;§2.9 的 6 个部件测试都通过，只能说明每个小部件单独看没问题。但真实 nudge 机制不是单个函数，而是一条流水线：计数器达阈值 → `_spawn_review` 创建 review agent → review agent 读取 prompt → LLM 返回 `tool_calls` → LangChain dispatch 到 `write_skill` → 磁盘出现 `SKILL.md`。只要这条流水线中间任何一环接错，前面的部件测试仍然可能全部通过。

&emsp;&emsp;所以 2.10 只做一个更重的测试：让 review agent 真走 tool calling 链路，最后检查磁盘上是否真的出现预期的 `SKILL.md`。如果用工程测试分层的说法，这类测试可以叫 `Tier 2` 或端到端测试；在这节课里你只需要记住：**2.10 测的是小部件组装起来以后，完整链路是不是真的能跑通**。

#### 测试 7：完整链路测试——经过 `_spawn_review` 真创建 review agent + 真触发 tool call

&emsp;&emsp;前 6 个测试验证了 nudge 机制的**单个小部件**，但小部件组装起来真能跑通 `_spawn_review → create_agent → tool call → write_skill` 这条完整链路吗？e2e 的成功标志不是"没报错"，而是"磁盘文件路径 + 内容字节级匹配 `tool_calls.args`"——这两件事同时成立才能证明 4 重约束 ③ 工具集隔离、④ 防递归、对话上下文注入三条契约都没断。

&emsp;&emsp;关键技术挑战：怎么让 review agent **真走 langchain tool calling 链路**而不调真 LLM？`FakeListChatModel` 没有 `bind_tools`，不能驱动 `create_agent` 的 tool calling。下面先写一个小夹具 `make_tool_calling_llm(...)`，把复杂的 `BaseChatModel` 细节藏起来；主测试只看三件事：构造 fake tool call、调用 `_spawn_review`、检查 `SKILL.md`。

In [59]:
# 测试 7 夹具：构造一个会返回 tool_calls 的假 LLM
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.language_models import BaseChatModel
from langchain_core.outputs import ChatGeneration, ChatResult


class ScriptedChatModel(BaseChatModel):
    """最小脚本化 chat model：按顺序吐 AIMessage，支持 bind_tools。"""
    responses: list
    i: int = 0

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        # 每次模型调用返回脚本中的下一条消息。
        idx = self.i
        if idx >= len(self.responses):
            idx = len(self.responses) - 1
        else:
            object.__setattr__(self, "i", idx + 1)
        return ChatResult(generations=[ChatGeneration(message=self.responses[idx])])

    def bind_tools(self, tools, **kwargs):
        # create_agent 需要模型支持 bind_tools；测试里直接返回 self。
        return self

    @property
    def _llm_type(self):
        return "scripted-chat-model"


def make_tool_calling_llm(tool_name: str, args: dict):
    """返回一个假 LLM：第一轮要求调用 tool，第二轮正常结束。"""
    return ScriptedChatModel(responses=[
        # 第一轮：要求 LangChain 调指定 tool。
        AIMessage(content="", tool_calls=[{
            "name": tool_name,
            "args": args,
            "id": "call_1",
        }]),
        # 第二轮：tool 执行后，让 agent 正常收束。
        AIMessage(content="Skill 已保存。"),
    ])

&emsp;&emsp;有了这个夹具，完整链路测试就变得很短：我们不手动调用 `write_skill`，而是让 `_spawn_review` 创建 review agent，再由 fake LLM 返回 `tool_calls`，最后让 LangChain 自己 dispatch 到 `write_skill`。这才是在测试"组装起来能不能跑"。

In [60]:
# 测试 7：完整链路契约——真走 _spawn_review → create_agent → tool call → write_skill
def test_e2e_经过_spawn_review_完整链路():
    """验证完整链路真能通过 tool_calls 写出 SKILL.md。"""
    setup_test_dir()
    # 让完整链路产生的 SKILL.md 写入测试目录。
    global SKILLS_DIR
    original_skills_dir = SKILLS_DIR
    SKILLS_DIR = TEST_SKILLS_DIR

    try:
        # 脚本：第 1 条 tool_calls 让 langchain dispatch 到 write_skill；第 2 条纯文本让 agent 正常结束
        skill_content = (
            "# Kafka Rebalance Debug\n\n"
            "consumer group order-svc 频繁 rebalance，通过调高 session.timeout.ms 缓解。\n"
        )

        fake_review_llm = make_tool_calling_llm(
            tool_name="write_skill",
            args={
                "name": "kafka-rebalance-debug",
                "action": "create",
                "content": skill_content,
            },
        )
        # 使用真实 NudgeMiddleware._spawn_review，不手动调用 write_skill。
        mw = NudgeMiddleware(review_model=fake_review_llm, nudge_interval=3, verbose=False)
        fake_state = {"messages": [
            HumanMessage(content="我的 kafka consumer 频繁 rebalance 怎么办"),
            AIMessage(content="先看 session.timeout.ms 配置"),
        ]}

        # 核心动作：fork review agent → fake LLM 返回 tool_calls → LangChain dispatch。
        mw._spawn_review(fake_state)

        # 核心证据：tool_calls.args.name 对应的 SKILL.md 真落盘。
        skill_file = TEST_SKILLS_DIR / "kafka-rebalance-debug" / "SKILL.md"
        assert skill_file.exists(), f"完整链路未产出 SKILL.md: {skill_file}"
        content = skill_file.read_text(encoding="utf-8")
        assert "Kafka Rebalance" in content, "args.content 标题传递断了"
        assert "session.timeout.ms" in content, "args.content 配置关键词缺失"
        print(f"测试 7 证据: file={skill_file}, bytes={skill_file.stat().st_size}")

        return "PASS"
    finally:
        SKILLS_DIR = original_skills_dir


print(test_e2e_经过_spawn_review_完整链路())

测试 7 证据: file=hermes_mvp_skills_test/kafka-rebalance-debug/SKILL.md, bytes=111
PASS


&emsp;&emsp;**这些测试分别守护了哪些设计约束**——读完 §2.9 + §2.10 的 7 个测试后，回过头看每个测试在守护 §2.4 表 2-2 的哪条 4 重约束（或 nudge 机制其他部件）：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 2-4：7 个测试 → 4 重约束 + nudge 机制部件映射</font></p>
<div class="center">

| 测试 | 守护的 4 重约束 / nudge 机制部件 | §2.8 实现锚点 | 测试层级 |
|---|---|---|---|
| 测试 1 | `write_skill` 真落盘契约（落盘动作本身）| `write_skill` @tool 定义（`skill_dir.mkdir + write_text`）| 部件级测试 |
| 测试 2 | nudge 计数器累加正确性（计数基础）| `after_model` `self.counter += 1` | 部件级测试 |
| 测试 3 | nudge 触发判定 + counter 重置（判定+重置逻辑）| `after_agent` 阈值判定 + `_spawn_review` 调用 + counter 归零 | 部件级测试 |
| 测试 4 | 合规硬开关（`nudge_interval=0` 永不 fork） | `after_agent` 中 `if nudge_interval == 0` 短路判定 | 部件级测试 |
| **测试 5** | **4 重约束 ③ 工具集隔离**（`write_skill` 不在主 agent）| `MAIN_AGENT_TOOLS` 模块级常量 | 部件级测试 |
| 测试 6 | nudge 频率调节器有效性（旋钮真有用） | `nudge_interval` 参数 + `after_agent` 阈值判定 | 部件级测试 |
| **测试 7** | **4 重约束 ① ② ③ ④ 全链路**（max_iterations + verbose + tools + middleware）| `_spawn_review` 内 `.with_config({"recursion_limit": 8})` + `tools=[write_skill, read_memory]` + `middleware=[]` + `verbose` 控制 | 完整链路测试 |

</div>

&emsp;&emsp;**7 个测试覆盖 4 重约束 + 计数 + 触发 + 频率 + 合规开关 共 8 个机制部件**，每个部件至少 1 个测试——把 §2.4 表 2-2 的 8 条文字契约全部转成了机器可验证的断言。**两层验证方法论的核心**：测试 7 是唯一覆盖 4 重约束 ①②③④ 全部 4 项的测试，PASS 的完整链路信心最足；但测试 7 一挂，你不知道是哪个小部件坏了——这就是为什么前面 6 个部件级测试不能省，它们给的是精确诊断。**带回自己 langchain 项目时**，最高复用价值的两个技法是：`SpyMiddleware`（实例 spy）收集 fork 调用 + 模块级工具白名单常量做工具集隔离断言——这两招几乎覆盖所有 `AgentMiddleware` 部件级测试场景。**第一性原理本身的价值不只在测试**——它是工程师看任何契约（API 文档、SLA、合同）时都该问的两问：承诺什么 / 怎么证伪。下一章我们把 §2.4-§2.10 这条完整链路当作证据，用**四维评价框架**(GC / AC / CE / 入口治理)横向看 `Hermes` 在 harness 赛道上选了什么、放弃了什么。

---

## <center>第3章：取舍分析——3 问评价法 + 四维评价框架</center>

&emsp;&emsp;前两章我们已经看清 `Hermes` 的两个核心能力：第 1 章讲它怎么把记忆存下来、搜出来、摘要出来；第 2 章讲它怎么在后台把一次任务经验变成新的 `SKILL.md`。现在问题来了——**这些能力放到生产里，到底算优势还是风险？**

&emsp;&emsp;这一章不再加新机制,而是给你一把判断尺。假设你是团队里的工程负责人，准备把 `Hermes` 接到飞书、`Telegram`、`Slack`，还允许它在本地或远程环境里执行任务。你真正该问的不是"`Hermes` 功能多不多"，而是下面三件事：

1. **能力怎么增长？** 是人提前把配置和 skill 写死，还是系统运行中会自己积累记忆、自己产出 skill？

2. **风险怎么控制？** 是主要相信 agent 自觉别乱来，还是用沙盒、权限、独立工作区、回滚机制把它限制住？

3. **入口怎么组织？** 是所有入口和权限走同一套中枢，还是飞书、`Telegram`、IDE、API 各自一套接入和认证？

&emsp;&emsp;这就是你下课要带走的 **3 问评价法**——前 2 问直接对应第一节《认知地图》里学过的**三支柱**:能力增长 = `Garbage Collection`(熵减/动态化)、风险控制 = `Architectural Constraints`(架构约束);第 3 问的入口组织三支柱**不直接覆盖**,所以单独补一个**入口治理**维度。再加上第 1 章已经讲透的 `Context Engineering`(三支柱第 3 项,上下文工程),3.5 节我们会把这 4 维合成一张完整的**Hermes 取舍位置图**。

> &emsp;&emsp;**关于这把尺的来历**:能力/风险/入口这 3 问对应 L1 三支柱里的 GC + AC,加上一个 L1 没单独列的**入口治理**(ACP 服务端 vs gateway 平台入口的认证统一性,这是 Hermes 双入口设计在合规视角下的真实代价)。所以这章不是引入新词,是把 L1 你已经学过的三支柱**换个评价坐标系**——而且补一个三支柱不直接覆盖的入口治理维度,让评价完整。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260508155348931.png" width=50%></div>

### 3.1 三支柱回顾 + 入口治理为何要补一维

&emsp;&emsp;先把 3 问翻译成你已经学过的术语。第一节《认知地图》给过你**三支柱**——`Context Engineering`(上下文工程，输入端)、`Architectural Constraints`(架构约束，工程边界)、`Garbage Collection`(熵减/动态化，自维护)。这章的 3 问跟三支柱一一对应,只是切到"评价坐标系"视角:

&emsp;&emsp;**问题一·能力怎么增长 → `Garbage Collection` 维度**——系统能力是人提前写死的，还是运行中会自己长出来？手写一个 `SKILL.md`、手写一份配置，这是 GC 弱端(静态);Agent 跑完任务后自己补记忆、自己产 skill、后台 curator 再清理旧 skill，这是 GC 强端(动态化)。

&emsp;&emsp;**问题二·风险怎么控制 → `Architectural Constraints` 维度**——你是相信 agent 别乱来，还是用工程边界把它锁住？AC 强端的具体形式是权限确认、沙盒、**独立工作区(worktree subagent)**、checkpoint 回滚、工具集白名单/黑名单。AC 越强，agent 越不容易破坏环境；但系统也越复杂，很多动作会变慢或需要用户确认。AC 越弱(偏信任端)，体验更顺滑，自动化更强；但出事时更依赖 agent 自律和团队兜底。

&emsp;&emsp;**问题三·入口怎么组织 → 入口治理维度(三支柱外的第 4 维)**——所有入口和权限是不是统一管理？这一维三支柱不直接覆盖。集中端不是说所有代码写在一个文件里，而是说**权限、身份、入口边界有统一中枢**；分治端也不是坏事，它让系统能接很多平台、很多客户端，但入口越多，边界越多，越容易出现"这边认证了，那边没同步"的缝隙。这一维是合规/审计视角的核心评价点,跟三支柱的工程视角正交,所以单独补一维。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 3-1 3 问评价法 ↔ 三支柱+入口治理 对照表</font></p>
<div class="center">

| 工程问题 | 评价维度 | 来源 | 白话判断 |
|---|---|---|---|
| 能力怎么增长 | `Garbage Collection`(动态化) | L1 三支柱 | 人写死配置 / skill，还是系统运行中自动记忆、自动产 skill、自动清理 |
| 风险怎么控制 | `Architectural Constraints`(架构约束) | L1 三支柱 | 相信 agent 自觉，还是用沙盒、权限、独立工作区(worktree)、回滚机制限制它 |
| 入口怎么组织 | **入口治理**(集中↔分治) | 三支柱外补充第 4 维 | 统一入口和统一权限，还是多平台、多协议、多认证各自管理 |
| (上下文管理) | `Context Engineering`(上下文工程) | L1 三支柱 | 第 1 章已详讲(FTS5 双索引 + LLM 摘要)，3.5 节四维表统一收口 |

</div>

&emsp;&emsp;所以这把尺不是新词，而是把你已经看到的 `Hermes` 源码现象放到三支柱坐标系下评价 + 补一个入口治理维度。下面三节就按这三个问题走：先看它为什么 GC 强、再看它为什么 AC 偏信任、最后看它为什么入口偏分治；3.5 节合成 4 维位置图(把 CE 加进来),就是 Hermes 的完整取舍画像。

### 3.2 能力怎么增长(Garbage Collection 视角)：Hermes 动态化强

&emsp;&emsp;先看 `Hermes` 为什么动态强。源码里能数到 `19` 个 gateway 平台入口和 `8` 个执行 backend。这里的数字不是让你背平台清单，而是证明一件事：`Hermes` 不是只在一个 CLI 里跑，它试图把同一个 agent 放到很多入口和很多执行环境里复用。

> **【数字口径说明】** `19` 是**用户感知的 platform 类别数**(把 `wecom_callback` 折进 `wecom` 主入口算一个 WeCom)。`gateway/config.py:90-110` 的 Platform enum 严格计数是 20 个非-LOCAL 内置 platform(WECOM_CALLBACK 单列一个)。本课用 19 计数,跟用户视角的"接消息平台数"一致;关心 enum 层精确计数的同学请直接看源码。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 3-2 Hermes 跨平台入口与执行环境</font></p>
<div class="center">

| 类别 | 数量 | 代表性 adapter | 关键设计 |
|---|---|---|---|
| 消息平台 | 6 | telegram / discord / slack / whatsapp / signal / matrix | 用户可以像给机器人发消息一样跟 Hermes 对话 |
| 企业/国内 | 6 | mattermost / dingtalk / wecom / weixin / feishu / qqbot | 团队 IM、群聊、企业机器人入口 |
| 其他入口 | 7 | bluebubbles / webhook / api_server / yuanbao / homeassistant / email / sms | API、邮件、短信、智能家居等非聊天入口 |
| 5 月新增 plugins | 2 | teams、irc（均位于 `plugins/platforms/`，非 gateway）| 课件主线以 19 个 gateway 平台为锚，plugins 层另加 2 入口仅作补充说明 |
| 终端 backend | 8 | local / docker / ssh / daytona / singularity / modal / managed_modal / vercel_sandbox | agent 真正执行命令或任务的环境 |

</div>

&emsp;&emsp;这里先解释一下 `Telegram / Slack / 飞书` 这些名字。它们不是 `Hermes` 的"智能"本身，而是**用户从哪里把消息发给 agent**。比如你在 `Telegram` 给机器人发一句"帮我查昨天部署失败原因"，或者在飞书群里 @ 一个机器人问"这个 Kafka rebalance 怎么处理"，这些消息都会从各自平台进入 `Hermes gateway`。平台 adapter 的工作就是把不同平台的消息格式、认证方式、群聊路由统一翻译成 `Hermes` 能处理的内部消息。

&emsp;&emsp;动态强的关键不只是"入口多"，而是**同一个 agent 的记忆和 skill 能跨入口复用**。你在 `Slack` 里让 Hermes 学到"这个项目用 pytest"，下次从飞书问它，它仍然能带着同一套 `MEMORY.md`、`USER.md`、`skills/` 工作。你不需要给每个平台各部署一套 agent，也不需要给每个平台各写一份 skill。这个设计让能力可以在运行中积累，然后跨入口复用。

&emsp;&emsp;再加上第 2 章讲过的 `Nudge`：它会在任务结束后后台 review，把可复用经验写成新的 `SKILL.md`。这就是 `Hermes` 在 `Garbage Collection` 维度偏动态化端的根本原因：**它不是只加载人类写好的能力，而是在运行中把对话经验变成后续能力。**

> **【常见误区】**：看到 19 个平台，很容易得出"覆盖越广越好"。这不对。入口越多，维护成本越高：每个平台都有自己的 bot token、OAuth、webhook、消息格式、rate limit。`Hermes` 的选择是把很多工程预算投到"更多入口复用同一个 agent"，这让动态能力很强，但也把认证和平台维护压力推高了。

&emsp;&emsp;一句话收束：**Hermes 动态强，不是因为平台名字多，而是因为同一个 agent 的记忆、skill、工具能力能在多入口里持续复用和增长。**

### 3.3 风险怎么控制(Architectural Constraints 视角)：Hermes 偏信任端

&emsp;&emsp;动态化强不等于风险小。现在问第二个问题：`Hermes` 让 agent 执行任务时，工程边界够不够硬？AC 维度的具体形式有 4 种,Hermes 在前 3 种上**显著弱**、第 4 种上**反而强**——这是这一维评价的关键区分点。先看它**弱在哪 3 处**:`native sandbox`、`worktree subagent`、`step checkpoint`。

&emsp;&emsp;**第一处弱·没有 native sandbox**，意思是没有操作系统级别的"硬围栏"。你可以把 `Docker` 理解成一间隔开的房间，能挡住很多普通误操作；但 `native sandbox` 更像楼里的门禁系统，哪些文件能读、哪些进程能启动、哪些系统能力能用，是系统级白名单在管。`Hermes` 的执行 backend 里有 `Docker`、`ssh`、`modal` 等环境，但没有像 macOS `sandbox-exec` 这种内核级强限制。所以它不是完全没隔离，而是**隔离强度没有到最硬那档**。

&emsp;&emsp;**第二处弱·没有 worktree subagent**(注意:这里**严格限定**指"per-delegate-subagent worktree"——给每个 fork 出的子 agent 自动隔离工作区。Hermes 在系统层**有** worktree:`cli.py:751-752` 的 `hermes -w` 启动 CLI session worktree、`tools/kanban_tools.py:644-650` 的 dispatcher `workspace_kind: worktree` 选项,但 delegate fork 出的 subagent 不自动隔离),意思是子 agent 没有天然独立的项目副本。可以想象两个人同时改同一份 Word 文档,如果没有"各自一份副本再合并",就可能互相覆盖。代码里也是一样:如果父 agent 和子 agent 共享同一个 `cwd`(当前工作目录),多个 agent 同时改文件时,冲突就要靠业务流程自己处理。`Claude Code` 那类 per-subagent worktree 的做法,则更像给每个子 agent 一份独立工作区,最后再合并结果。**这一处是 AC 弱端的头号证据**——`tools/delegate_tool.py:609-623` 子 workspace 探测只走父 agent cwd / TERMINAL_CWD,**无 per-delegate worktree 创建**。

&emsp;&emsp;**第三处弱·没有完整 step checkpoint**(同样要严格区分:Hermes **有** filesystem checkpoint——`tools/checkpoint_manager.py` 通过 shadow git repo 实现,触发逻辑是**在 agent tool iteration 循环内、文件变更工具或破坏性 terminal 命令执行前**按工作目录 snapshot(`run_agent.py:9481-9499` / `9919-9939` 是触发点,`run_agent.py:10811-10813` 在每次 iteration 重置去重)。所以一次 user conversation turn 内可能出现多次 snapshot,但**不是每个 LLM 推理步都无条件 checkpoint**;opt-in 通过 `checkpoints` 配置或 `--checkpoints` 启用,默认 `max_snapshots=50`,可通过 `cli.py:4051-4059` 的 `/rollback` 命令列出/差分/恢复 + 同时撤销 last chat turn。**真正没有的**是每个 LLM 推理步无条件细粒度 step checkpoint),意思是 agent 走到第 50 步发现第 30 步错了,不能精确回到第 30 步继续(只能从最近一次"文件变更前"的 snapshot 回滚)。生产里这意味着:长任务跑偏后,更多时候要从最近 mutating-op 边界重来,而不是像调试程序一样精确回到中间某个推理步。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 3-3 Hermes 6 主 Backend 隔离强度对比（不含 2 个变体）</font></p>
<div class="center">

| Backend | 隔离层 | 强度评级 | 适用场景 |
|---|---|---|---|
| `local` | 无 | 最弱 | 信任环境本地调试 |
| `docker` | 容器 user-space | 中 | 大多数生产场景 |
| `ssh` | 远程主机 + 网络层 | 中 | 跨机执行 |
| `daytona` | 远程开发环境 | 中 | dev workspace |
| `singularity` | 容器（HPC 友好）| 中 | 学术/HPC 集群 |
| `modal` | Serverless 容器 | 中 | 弹性云执行 |

</div>

&emsp;&emsp;上表只列 6 个主 backend；`managed_modal` 和 `vercel_sandbox` 是变体，隔离强度按各自基础 backend 同档处理。表里最重要的不是名字，而是结论：**没有任何一个 backend 是 native sandbox 那种系统级硬围栏。**

&emsp;&emsp;**第四种 AC 形式·工具集隔离·Hermes 反而强**:`tools/delegate_tool.py:39-48` 里有 5 项黑名单,子 agent 不能递归 `delegate_task`、不能自己调用 `clarify` 卡住用户、不能直接写共享 `memory`、不能跨平台发消息、不能走 `execute_code` 绕开推理。这说明 `Hermes` 不是全维度 AC 弱，而是 **AC 三维(沙盒/worktree/checkpoint)弱、AC 一维(工具集隔离)强**。

&emsp;&emsp;所以这一维的结论要说准确：`Hermes` 不是"没 AC"，而是 **AC 维度整体偏信任端**。它信任 agent 和团队流程能处理共享目录、任务跑偏、执行环境边界这些问题；它把更多工程预算投给了 GC 动态化能力，而不是把每个动作都放进强沙盒、独立 worktree、细粒度 checkpoint 里。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260508155348911.png" width=50%></div>

### 3.4 入口怎么组织(入口治理维度·三支柱外的第 4 维补充)：Hermes 偏分治端

&emsp;&emsp;第三个问题是入口怎么组织。这一维 L1 三支柱**不直接覆盖**——三支柱讲的是单 agent 实例内部的工程姿态(GC / AC / CE 都是),而入口治理讲的是**多入口共享同一 agent 实例时的认证边界统一性**,本质上是合规/审计视角的评价点。所以本课把它单列为第 4 维补充。

&emsp;&emsp;`Hermes` 能从很多地方接消息：IDE 可以通过 `ACP` 接入，飞书、`Telegram`、`Slack` 这些聊天平台通过 `gateway/platforms/` 接入，另有 `Cron` 和 `Batch Runner` 这种内部入口。入口多本身不是坏事，但要看权限和身份是不是有统一中枢。

&emsp;&emsp;这里先把 `ACP` 和 `MCP` 讲清。`MCP` 是工具协议，解决的是 agent 怎么连数据库、搜索、文件系统等外部工具；`ACP` 是 agent 通信协议，解决的是 IDE 或另一个 agent 怎么跟当前 agent 对话。你可以简单记：**MCP 连工具，ACP 连 agent。**

&emsp;&emsp;源码核查里 Hermes 实际有**三套独立认证**:

1. **ACP 入口认证** (`acp_adapter/server.py:462-507` + `acp_adapter/auth.py`): IDE 客户端通过 `auth_methods` advertise + `authenticate(method_id)` 用 ACP runtime credential 认证

2. **Platform 入口认证** (`gateway/platforms/<各>.py`): 飞书/Telegram/Slack 等各自的 bot token / OAuth / webhook 签名,每个 platform adapter 独立实现

3. **用户级 DM 审批** (`gateway/pairing.py`): 8-char 一次性 pairing code 让 bot owner 通过 CLI 审批未知 messaging platform 用户(注意:**这是用户白名单管理层,不是 platform 认证**)

&emsp;&emsp;问题不是"没有认证",而是**这三套认证没有统一抽象**——它们各自管理身份、权限、审计,互相不交叉验证。

&emsp;&emsp;用人话说：同一个 `Hermes` 实例，VS Code 入口走 ACP runtime credential 认证、飞书入口走 platform OAuth、Telegram 入口走 bot token、未知 DM 用户走 pairing code 审批——这四条认证路径都能把消息送进同一 agent,但"谁有权限、能做什么、身份怎么映射"不是同一个中枢在管。入口越多，这类边界越多；边界越多，缝隙越多。这就是入口治理维度上 Hermes 偏分治端的代价。

> **【入口治理 ≠ 内部架构分治】**:这一维**不讨论 agent 内部架构**(子代理、工作区隔离这些已在 3.3 `Architectural Constraints` 维度评价过)。这一维专问:**`acp_adapter/server.py` (IDE) + `gateway/platforms/*.py` (messaging) + `gateway/pairing.py` (用户级 DM 审批)三套独立认证体系共享同一 agent 实例时,身份/权限/审计是不是同一中枢统一管理**。

> **【常见误区·"分治不是坏词"】**：分治本身不是坏词。很多成熟系统都靠分治扩展复杂度。问题在于分治后有没有统一身份、统一权限、统一审计。如果有，它是健康分治；如果没有，它就会变成"每个入口各管各的"，长期维护时最容易出安全和合规问题。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260508155348989.png" width=50%></div>

### 3.5 四维位置图：Hermes 取舍全貌

&emsp;&emsp;现在把三节证据 + 第 1 章已展开的 `Context Engineering` 实现合起来，就能得到 `Hermes` 的**四维位置图**。注意这不是打分表，也不是说哪个系统更好；它只回答一个问题：**`Hermes` 把工程预算押在了哪里，又牺牲了哪里。**

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 3-4 Hermes 在四维评价框架上的位置(三支柱 + 入口治理)</font></p>
<div class="center">

| 评价维度 | 来源 | Hermes 位置 | 关键证据 | 代价 |
|---|---|---|---|---|
| `Garbage Collection`(动态化) | L1 三支柱 | **强选(动态端)** | 19 平台一致 + `Nudge` 自学 skill + `Curator` 后台清理 | 部署复杂 + 自学习虚假产出风险 |
| `Architectural Constraints`(架构约束) | L1 三支柱 | **部分放弃(偏信任端)** | 无 native sandbox、无 per-delegate-subagent worktree(系统层有 `hermes -w` CLI session worktree + kanban dispatcher worktree)、无每步 LLM 推理无条件级 step checkpoint(有 filesystem checkpoint: agent iteration 内、文件变更/破坏性命令前按目录 snapshot + `/rollback` 命令撤销 + 默认 max 50, opt-in);但工具集黑名单(`tools/delegate_tool.py:39-48` 5 项)反而强 | 信任端押注用户自律 |
| `Context Engineering`(上下文工程) | L1 三支柱 | **增强但有代价** | 热/冷记忆双层 + `FTS5` 双索引(unicode61 + trigram) + `LLM` 摘要(并发默认 3、上限 5 / 重试 3) | 热记忆越长 token 越贵 + 14,000 tokens schema 固定开销 |
| **入口治理**(集中↔分治) | 三支柱外补充第 4 维 | **偏分治端** | 三套独立认证: `acp_adapter/server.py` (ACP runtime credential) + `gateway/platforms/*.py` (per-platform bot token/OAuth) + `gateway/pairing.py` (用户级 DM 审批),未统一成同一权限中枢 | 多入口边界缝隙(合规/审计风险) |

</div>

&emsp;&emsp;这张表可以翻译成一句更容易记的话：**`Hermes` 选择了强 GC 动态化和增强的 CE,让 agent 能跨入口复用记忆/skill 并自动增长;代价是 AC 偏信任端 + 入口治理偏分治端,执行环境、工作区、入口认证都更依赖信任与后续治理。**

&emsp;&emsp;到这里，第三章真正要你带走的不是这把尺的术语,而是一个评价动作：看到任何 harness，都先问 3 件事——能力怎么增长、风险怎么控制、入口怎么组织;再加上第 1 章的上下文工程一并看,就是 4 维完整画像。你能回答这 3 问 + 1 维,就不会被"平台很多""有 sandbox""有 skill"这种单点能力带偏。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 3-5 可选对照:OpenClaw 在四维评价框架上的位置(vs Hermes)</font></p>
<div class="center">

| 评价维度 | OpenClaw 位置 | 关键证据 | 跟 Hermes 的差异 |
|---|---|---|---|
| `Garbage Collection`(动态化) | **静态偏强(动态弱)** | skill 由工程师手写 + `Git` 版本化 + `ClawHub` 市场分享，无 nudge 自学习，无后台 curator 清理 | Hermes GC 维度显著更深——这是 Hermes 押注的核心方向 |
| `Architectural Constraints`(架构约束) | **约束端中等** | 显式 sandbox 模型 + 多 channel 隔离 + Mobile/Desktop 三端独立认证 | OpenClaw 比 Hermes 强一档(Hermes 仅 Docker user-space) |
| `Context Engineering`(上下文工程) | **轻量端** | 单端 chat history + 无双索引召回 + 无 LLM 摘要层 | Hermes CE 增强幅度显著更深(FTS5 + LLM 摘要) |
| **入口治理**(集中↔分治) | **分治端深(健康分治)** | macOS/iOS/Android 三端原生 + Live Canvas + 24 渠道,每个渠道是独立分治模块 | 跟 Hermes 持平偏深,但 OpenClaw 是"模块多"的健康分治,Hermes 是"双入口认证不一致"的缝隙分治 |

</div>

&emsp;&emsp;这张 `OpenClaw` 表是可选对照，不是本章主线。它的作用只是提醒你：都有 skill，不代表四维位置一样。`OpenClaw` 的 skill 更偏人写、社区分发、Git 管理；`Hermes` 的 skill 更偏 agent 运行中自动产出、后台 curator 再清理。所以前者 GC 维度弱一些，后者 GC 维度强很多。

> **【跨讲数字一致性提示】** 本节涉及的前三节关键词都跟前三节课一字不差——第一节八大故障是"循环失控/context 溢出/cache miss/tool 错误吞/状态丢失/缺权限闸/缺自动化评审/成本失控"；第一节八大机制是"Agent Loop/Tool Use/Progress Tracking/Context Management/Feature List/Verification Loop/Subagents/Generator-Evaluator"；第一节三支柱是 `Context Engineering / Architectural Constraints / Garbage Collection`(本章 3 问 + 入口治理共 4 维评价直接复用前三个);第二节 11 机制 = 8 核心 + 3 扩展(`Permission Gate / Hooks / Token Budget`);第二节 demo 数据 Baseline 1104 vs Full 15053 tokens(13.6×);第三节用 `deepagents==0.5.3`、4 大组件 `Middleware / Backend / 子代理 / HITL`。如果这一节让你联想起前三节的某个具体场景，那正是这把四维尺工作的方式——它是把已学概念换坐标系做评价的回顾工具。

---

## <center>第4章：收束总结</center>

### 4.1 学员出讲带走（能力锚点）

&emsp;&emsp;今天这节课结束你应该能从教室里带走四件具体产物——它们对应节 0.2 三问题破题（问题一/问题二/问题三）和本讲核心动作（读源码 + 跑 demo + 用**四维评价**框架做取舍判断）的全部产出。

&emsp;&emsp;**第一件**是一张 `Hermes` 默认记忆系统架构图——**默认链路**画出热层 `MEMORY.md`（Agent 自维护事实）+ `USER.md`（用户画像，默认空、启用 `Honcho` 后自动填充）+ 冷层 `SQLite 持久化` + `FTS5 双索引（unicode61 + trigram）` + `LLM 摘要层（异步并发 3、最大 5、失败 3 次重试、可降级 raw preview）`；并能说清**扩展层**——8 种外部 `Memory Provider`（含 `Holographic` / `Mem0` / `Supermemory` 等）全部 opt-in，不在默认链路。

&emsp;&emsp;**第二件**是 `Nudge` 机制源码 walk 笔记——你能完整复述 `计数器累加（memory user-turn / skill tool-iteration） → turn 末判定 → fork 子 agent 4 重约束（max_iter=16 / quiet_mode=True / toolsets=memory+skills / 防递归 interval=0） → _SKILL_REVIEW_PROMPT 驱动反思（PATCH > CREATE，ACTIVE 倾向） → _create_skill 五步校验 → 落盘到 ~/.hermes/skills/<name>/SKILL.md`这条完整链路，并知道这不是 AI 自主学习，是 prompt 驱动的机械闭环。

&emsp;&emsp;**第三件**是一份本地可跑的 `MVP demo` 代码包，把 Hermes 的 nudge 机制实现成 `langchain` 三件套——`NudgeMiddleware`（计数器 + turn 末判定 + fork sub-agent，对应 Hermes hook 链）+ `write_skill` / `read_memory` 双 `@tool`（写侧落盘 + 读侧反思素材，对应 `skill_manager_tool.py` + memory store）+ `create_agent(tools=[write_skill, read_memory], middleware=[]).with_config({"recursion_limit": 8})` 受限 sub-agent（4 重约束 ① ② ③ ④ 全部显式表达），配 5 轮真实 kafka 调试场景 + 三组 `nudge_interval` 改参实验，你能在自己机器上跑出 `agent-created` skill 文件并对比触发频率。

&emsp;&emsp;**第四件**也是最重要的一件——一套 **3 问评价法 + 四维评价框架**:3 问(能力怎么增长 / 风险怎么控制 / 入口怎么组织)直接对应 L1 三支柱里的 `Garbage Collection` / `Architectural Constraints` 和三支柱外补充的**入口治理**维度,再加上第 1 章已展开的 `Context Engineering`,3.5 节合成完整的**四维位置图**。但你不用先背术语;以后看任何 harness,先用这 3 问判断它选了什么、放弃了什么。

&emsp;&emsp;对比学习前的状态——课前你知道怎么写 `SKILL.md`，但不知道它能不能自动维护；课后你能判断一个 harness 的**能力是人写死的，还是运行中会自己增长**；也能判断它靠不靠**强沙盒、独立工作区、统一权限**来控制风险；还能在合规场景一行配置禁用 `nudge`，并预见到 5 月新增的 `skill_provenance.py` 是为下节课 `Curator` 做的工程铺垫。这是从"会用"到"会评价"的跃迁。